# Three-parameter PINN inversion

Run the cells in order. The notebook contains the physical equations, inverse-model implementation, training configuration, and execution cell. It starts from **Keos=0.5, RH=35, pH50=7** and does not restore an inversion checkpoint.

Adaptive stopping is active. The iteration counts are safety limits, not mandatory stopping steps.


In [ ]:
from pathlib import Path
import os


def locate_repository():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "model_and_data" / "data").is_dir() and (candidate / "uncertainty_analysis").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Run this notebook from inside the repository folder.")


REPOSITORY_ROOT = locate_repository()
DATA_DIR = REPOSITORY_ROOT / "model_and_data" / "data"
OUTPUT_ROOT = REPOSITORY_ROOT / "model_and_data" / "results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("DDE_BACKEND", "tensorflow.compat.v1")
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_ROOT / ".mplconfig"))

print("Data:", DATA_DIR)
print("Removable results:", OUTPUT_ROOT)


## Physical-model definitions

These definitions are executed in the notebook kernel and do not start training.


In [ ]:
# -*- coding: utf-8 -*-
"""
PINN (TF v1 + DeepXDE): coupled transport with electrokinetics and surface/precipitation equilibrium
Units: time=day, length=m, potential=V; concentrations: mol/m^3 (water); bulk stores: mol/m^3 (bulk)

- Fast chemistry: algebraic equilibrium plus total-Pb invariant transport
- Acid/base network transports a = cH - cOH and enforces cH*cOH = Kw algebraically
- Stable numerics: safe water-content evaluation, stable quadratic solution, clipping, finite guards, and gradient clipping
"""

import tensorflow as tf
import deepxde as dde
import numpy as np
import random
import os

# ===== Global time scaling (PDEs are in day units) =====
SEC_PER_DAY = 86400.0  # s/day

# ===== TF v1 setup =====
tf.compat.v1.disable_eager_execution()
tf.compat.v1.reset_default_graph()
tf.compat.v1.set_random_seed(0)
random.seed(0)
np.random.seed(0)

# ---------------------------
# 0) CONFIG (day-based transport; m, V)
# ---------------------------
CFG = {
    "GLOBAL": {
        "CONSTANTS": {"F": 96485.3329, "R": 8.314462618, "T": 298.15},
        # Kw at 25 C in (mol/m^3)^2; pH 7 corresponds to cH=1e-4 mol/m^3 water
        "CHEM": {"Kw": 1.0e-8},
        "EO": {
            "zeta_mV": -30.0,                     # mV; clay effective value, pH/ionic-strength sensitive
            "eps_w": 80.0 * 8.8541878128e-12,     # F/m
            "mu_water_Pa_s": 1.0e-3,              # Pa*s
        },
        "POROUS": {"eo_tau_model": "bruggeman", "diff_model": "bruggeman", "eo_power": 2.0},
    },
    "DOMAIN": {"xmin": 0.0, "xmax": 0.2, "tmin": 0.0, "tmax": 5.0},
    "SCENARIO": {"cathode_drainage": "fixed_zero"},
    "NUMERICS": {
        "DATASET": {"n_res": 12000, "n_ic": 1200, "n_left": 1200, "n_right": 1200, "pred_n": 121},
        "NET": {"hidden_layers": 5, "width": 20, "act_scale": 10.0, "trainable_layer_gain": True},
        "TRAIN": {
            "water_iters": 1200,
            "elec_A_iters": 500,
            "hydro_elec_outer_iters": 1,
            "hydro_elec_water_iters": 1200,
            "hydro_elec_elec_iters": 500,
            "hplus_iters": 5600,
            "pb_inv_iters": 2600,
            "hplus_refine_iters": 0,
            "pb_refine_iters": 0,
            "batch_size": 512,
            "hplus_batch_size": 1536,
            "hplus_use_lbfgs": False,
            "adam_lr": 1e-3,
            "hplus_adam_lr": 2e-4,
            "hplus_grad_clip": 1.0,
            "pb_adam_lr": 8e-4,
            "pb_grad_clip": 5.0,
            "lbfgs": {"maxiter": 5000, "maxfun": 5000, "maxcor": 50, "maxls": 50,
                      "ftol": np.finfo(float).eps, "gtol": np.finfo(float).eps},
            "freeze_k_phi": 1,
            "freeze_k_c": 1,
        },
        # Amini, Haghighat & Juanes (2022), Algorithm 1 adapted to the
        # acid/base-Pb blocks. Water/electric also get a small sequential
        # closure so electroosmosis and Archie conductivity share theta/phi.
        "SEQUENTIAL": {
            "outer_max_iters": 0,
            "outer_min_iters": 1,
            "acid_iters": 1800,
            "pb_iters": 2600,
            "parameter_rel_tol": 5.0e-2,
            "pH_rms_tol": 5.0e-2,
            "pH_max_tol": 2.5e-1,
            "psi_rel_tol": 1.0e-2,
            "fixed_rel_tol": 1.0e-2,
            "diagnostic_nt": 81,
            "diagnostic_nx": 121,
        },
        "FULL_COUPLED": {
            "enabled": True,
            "iters": 1600,
            "batch_size": 1024,
            "adam_lr": 2.0e-4,
            "acid_loss_weight": 1.0,
            "pb_loss_weight": 1.0,
            "print_every": 100,
        },
    },
    "FIELDS": {
        # === Water (Richards, day units) ===
        "WATER": {
            "PARAM": {
                "SOIL": {"n": 1.843, "alpha": 0.02343789, "Ks": 4.32e-5, "theta_r": 0.068, "theta_s": 0.50, "Ss_theta": 1.0e-3, "K_sat_head_slope": 0.0},  # clay/kaolinite WRC; Ks=5e-10 m/s = 4.32e-5 m/day
                "head_transition_tau": 0.02,
                "bandai_beta_m": 0.01,
                "ic_theta_weight": 500.0,
                "boundary_theta_weight": 0.0,
                "boundary_head_weight": 80.0,
                "mass_integral_weight": 500.0,
                "mass_integral_nx": 81,
                "mass_integral_nt": 41,
                "include_eo": True,
                # For theta050/saturated runs: keep theta fixed at theta_s=0.50 and skip WaterNet training.
                "skip_water_training_at_saturation": True,
                "saturation_psi_tol": 1e-12,
            },
            "INITIAL": {"psi_ic": 0.0},  # initial pressure head, m
            "BOUNDARY": {
                "left": {
                    "face": "LEFT",
                    "type": "dirichlet",
                    "psi": 0.0,
                },
                "right": {
                    "face": "RIGHT",
                    "type": "dirichlet",
                    "psi": 0.0,
                },
            },
        },
        # === Electric (constant sigma Laplace) ===
        "ELECTRIC": {
            "PARAM": {
                "sigma_model": "archie_rhoades",
                "sigma_s": 1.8e-2,         # S/m surface conductivity
                "sigma_b": 2.5e-1,         # S/m pore-water/bulk-brine conductivity
                "rhoades_a": 1.3,
                "rhoades_b": -0.2,
                "sigma_sat": 7.425e-2,     # sigma_s + sigma_b*theta_s*(a*theta_s+b)
                "sigma_const": 7.425e-2,   # fallback / legacy name
            },
            "BOUNDARY": {"electrodes": {"anode_face": "LEFT", "cathode_face": "RIGHT", "phi_anode": 5.0, "phi_cathode": 0.0}},
        },
        # === H+ (NP; Faradaic fluxes; NO kinetic water source) ===
        "HPLUS": {
            "PARAM": {
                "DL": 0.0,                             # Kim2005: no mechanical/hydrodynamic dispersion term
                "Dw": 9.312e-9 * 86400.0,              # m^2/day (m^2/s * s/day)
                "z": +1.0, "Dw_OH": 5.273e-9 * 86400.0, "z_OH": -1.0,
                "use_softplus": True, "include_eo": True,
                "pH_min": 0.0, "pH_max": 14.0,
                "pH_obs_weight": 0.0,
                "a_ic_gate_tau": 0.01,
                "bc_tmin_eps": 0.0,
                "faraday_ramp_tau": 0.01,
                "pH_init_left": 1.45,
                "pH_init_right": 12.85,
                "pH_init_raw_scale": 0.25,
                "a_raw_scale": 20.0,
                "pb_source_stop_gradient": True,
                "H_retardation": 20.0,             # soil acid buffering; was 4.6
                "aqueous_tortuosity_model": "millington_quirk_modified",  # D_i* = Dw_i * theta^beta / theta_s^2
                "mq_theta_power": 4.736965594166206,  # beta chosen so D*_sat/Dw = 0.15 at theta_s=0.50
                "tortuosity_factor": 0.4,
                "faraday_bc_mode": "species_nonadv",
                "faraday_flux_weight": 250.0,
                "coion_flux_weight": 300.0,
                "coion_flux_weight_min": 300.0,
                "pH_range_weight": 0.0,
                "pH_range_ramped": True,
                "pH_left_max_phys": 4.0,
                "pH_right_min_phys": 10.0,
                "residual_weight": 1.0,
                "faraday_saturation_power": 0.0,
                # Residual-based adaptive sampling / adaptive loss / time-slab continuation for stiff pH fronts.
                "adaptive_residual_sampling": False,
                # True RAR/RBAS: most collocation points are selected from a candidate pool
                # according to the current PDE residual magnitude; a small uniform background
                # is kept to avoid losing global coverage.
                "residual_adaptive_frac": 0.70,    # high-|residual| collocation fraction
                "residual_uniform_frac": 0.30,     # uniform background fraction
                "residual_cache_refresh": 50,      # refresh high-residual cache every N Adam steps
                "residual_cache_candidates": 8192, # candidate pool size for residual ranking
                "residual_cache_keep": 4096,       # number of top residual candidates retained
                "residual_score_power": 1.0,       # score = |residual|**power

                "adaptive_loss_weights": False,
                "adaptive_loss_update_every": 100,
                "adaptive_loss_alpha": 0.35,
                "adaptive_loss_min_scale": 0.20,
                "adaptive_loss_max_scale": 5.00,

                "time_slab_training": False,
                "time_slabs": [0.10, 0.25, 0.50, 1.0, 2.0, 5.0, 10.0],
                "time_slab_mode": "expanding",    # train on [tmin, slab_end], not disjoint intervals
                "training_schedule": None
            },
            "INITIAL": {"pH_ic": 7.0, "c_ic": None},   # if None, computed from pH
            "BOUNDARY": {
                # Convention: positive flux = inject (into domain), negative = extract (out of domain)
                "left_BC":  "dirichlet",
                "right_BC": "dirichlet",
                "left_flux_sign":  "inject",   # inject / extract
                "right_flux_sign": "extract",  # inject / extract
                # Voltage-controlled Faradaic source: I = sigma_eff(Se) * |dphi/dx|.
                "I_app_Aperm2": None,
                "faraday_current_mode": "archie_voltage",
                "current_efficiency": 0.19,
                "I_APP_MODE": {"type": "fixed_current", "tH": 0.19},
                # Dirichlet reservoir-pH boundary, Kim-style and easier than species flux BC.
                "dirichlet_mode": "reservoir_pH",
                "reservoir_depth_m": 0.20,      # V_reservoir / electrode_area; 2 L / 100 cm^2
                "reservoir_pH_ramp_tau": 0.05,  # day
                "pH_loss_scale": 10.0,
                "pH_dirichlet_weight": 30.0,
                "reservoir_clip_to_config": True,
                "pH_left":  0.0, "pH_right": 14.0,
            },
        },
        # === Total-Pb invariant transport ===
        "PB": {
            "PARAM": {
                "DL": 0.0,
                "psi_raw_scale": 20.0,
                "dae_res_norm": 10.0,
                "dae_res_weight": 5.0,
                "dae_alg_weight": 5.0,
                "constitutive_weight": 20.0,
                "constitutive_norm": 1.0,
                "left_constitutive_weight": 1000.0,
                "right_constitutive_weight": 300.0,
                "mass_balance_weight": 0.0,      # local dM/dt check; integral loss carries conservation
                "mass_balance_nx": 31,
                "mass_balance_batch_size": 128,
                "mass_integral_weight": 200.0,
                "mass_integral_nx": 21,
                "mass_integral_nt": 21,
                "solver": "pinn_component_equilibrium",
                "fv_nx": 161,
                "fv_nt": 1201,
                "Dw": 9.25e-10 * 86400.0,   # m^2/day
                "z": +2.0, "use_softplus": True, "include_eo": True,
                "acid_release_pH": 5.0,
                "acid_release_width": 0.35,
            },
            "INITIAL": {"c_ic": 0.0},       # initial dissolved Pb concentration in water
            # Pb has no Faradaic source: anode is strict zero total flux; cathode is natural outflow.
            "BOUNDARY": {"left": {"type": "flux", "J": 0.0}, "right": {"type": "open_outflow"}},
            "CHEM": {
                # Equilibrium constants built from kinetic ratios; no seconds in residuals
                "k_pr_f": 1.0e5,    # m^3/(mol*s)   SOH + H+ -> SOH2+
                "k_pr_b": 2.5e3,    # 1/s
                "k_dpr_f": 1.0e-5,  # 1/s          SOH -> SO- + H+
                "k_dpr_b": 4.8e-2,  # m^3/(mol*s)  SO- + H+ -> SOH
                "k_ad_f": 1.0e5,    # m^3/(mol*s)  SOH + Pb2+ -> SOPb+ + H+
                "k_ad_b": 7.1e5,    # m^3/(mol*s)  reverse
                "m": 2.0,                           # free-Pb2+ + 2OH- saturation exponent
                "Ksp_PbOH2": 1.43e-11,             # (mol/m^3)^3; effective Pb hydroxide/oxyhydroxide Ksp
                "HYDROLYSIS": {
                    # Pb2+ + jOH- <-> Pb(OH)j^(2-j); log beta values in mol/L units.
                    # Converted internally to mol/m^3 units by subtracting 3*j from log10(beta).
                    # Representative IUPAC/PHREEQC-style mononuclear hydrolysis set; sensitivity required.
                    "include": False,
                    "species": ["PbOH+", "Pb(OH)2(aq)", "Pb(OH)3-", "Pb(OH)4--"],
                    "log_beta_OH_molL": [6.54, 11.06, 13.97, 15.20],
                    "include_polynuclear": False
                },
                "PRECIP": {
                    "mode": "solubility_product_complementarity",
                    "smooth_min_eps": 1.0e-10
                },
            },
        },
        # === Initial surface/solid stores in bulk basis (mol/m^3 bulk) ===
        "SURF_SOLID": {"INITIAL": {"SOH0": 0.0, "SOPb0": 5.0, "SOH2_0": 0.0, "SOm0": 40.0, "Pp0": 0.0}},
    },
}

def _env_float(name, default=None):
    raw = os.environ.get(name)
    if raw is None or str(raw).strip() == "":
        return default
    return float(raw)


def _psi_from_theta0(theta0, soil):
    theta0 = float(theta0)
    theta_r = float(soil["theta_r"])
    theta_s = float(soil["theta_s"])
    if not (theta_r < theta0 <= theta_s):
        raise ValueError(f"PINN_THETA0={theta0:g} must be in (theta_r={theta_r:g}, theta_s={theta_s:g}]")
    Se = (theta0 - theta_r) / max(theta_s - theta_r, 1e-12)
    Se = float(np.clip(Se, 1e-8, 1.0))
    if Se >= 1.0 - 1e-12:
        return 0.0
    n = float(soil["n"])
    m = 1.0 - 1.0 / n
    alpha = float(soil["alpha"])
    return -((Se ** (-1.0 / m) - 1.0) ** (1.0 / n)) / alpha


def apply_water_initial_case(cfg):
    water = cfg["FIELDS"]["WATER"]
    soil = water["PARAM"]["SOIL"]
    theta0_env = _env_float("PINN_THETA0")
    psi_env = _env_float("PINN_PSI_IC")
    if theta0_env is None and psi_env is None:
        return cfg
    psi_ic = float(psi_env) if psi_env is not None else _psi_from_theta0(theta0_env, soil)
    water["INITIAL"]["psi_ic"] = psi_ic
    water_case = os.environ.get("PINN_WATER_CASE")
    if not water_case:
        water_case = f"theta{int(round(float(theta0_env) * 100)):03d}" if theta0_env is not None else "psi_ic"
    cfg.setdefault("SCENARIO", {})["water_case"] = water_case
    cfg["SCENARIO"]["theta0"] = theta0_env
    cfg["SCENARIO"]["psi_ic_m"] = psi_ic
    if theta0_env is not None:
        theta_s = float(soil["theta_s"])
        ab_bc = os.environ.get("PINN_UNSAT_AB_BC", "flux").strip().lower()
        if float(theta0_env) < theta_s - 1e-12 and ab_bc in ("flux", "archie_flux", "faraday_flux"):
            Hbd = cfg["FIELDS"]["HPLUS"]["BOUNDARY"]
            Hbd["left_BC"] = "flux"
            Hbd["right_BC"] = "flux"
            cfg["SCENARIO"]["acidbase_boundary_case"] = "archie_flux"
            print("[A/B_BC] unsaturated: H/OH use Archie-voltage flux boundary")
        elif ab_bc in ("dirichlet", "reservoir", "reservoir_ph"):
            cfg["SCENARIO"]["acidbase_boundary_case"] = "reservoir_pH"
    if theta0_env is None:
        print(f"[WATER_CASE] {water_case} | psi_ic={psi_ic:g} m")
    else:
        print(f"[WATER_CASE] {water_case} | theta0={float(theta0_env):g} -> psi_ic={psi_ic:g} m")
    return cfg

def apply_cathode_drainage_case(cfg):
    case = os.environ.get("OH_DRAINAGE_CASE", cfg.get("SCENARIO", {}).get("cathode_drainage", "head_open")).strip().lower()
    psi_ic = float(cfg["FIELDS"]["WATER"]["INITIAL"].get("psi_ic", 0.0))
    water_left = cfg["FIELDS"]["WATER"]["BOUNDARY"]["left"]
    water_right = cfg["FIELDS"]["WATER"]["BOUNDARY"]["right"]
    pb_left = cfg["FIELDS"]["PB"]["BOUNDARY"]["left"]
    pb_right = cfg["FIELDS"]["PB"]["BOUNDARY"]["right"]
    if case == "head_equal":
        cathode_head = psi_ic
        scenario = "head_equal"
    elif case == "head_open":
        cathode_head = 0.0
        scenario = "head_open"
    elif case == "fixed_zero":
        anode_head = 0.0
        cathode_head = 0.0
        scenario = "fixed_zero"
    else:
        raise ValueError(f"Unknown OH_DRAINAGE_CASE={case!r}; use head_open, head_equal, or fixed_zero")
    anode_head = locals().get("anode_head", psi_ic)
    water_left.clear()
    water_left.update({"type": "dirichlet", "face": "LEFT", "psi": anode_head})
    water_right.clear()
    water_right.update({"type": "dirichlet", "face": "RIGHT", "psi": cathode_head})
    pb_left.clear()
    pb_left.update({"type": "flux", "J": 0.0})
    pb_right.clear()
    pb_right.update({"type": "open_outflow"})
    cfg["SCENARIO"]["cathode_drainage"] = scenario
    cfg["SCENARIO"]["anode_head_m"] = anode_head
    cfg["SCENARIO"]["cathode_head_m"] = cathode_head
    print(f"[SCENARIO] {scenario} | psi_ic={psi_ic:g} m | psi_left={anode_head:g} m | psi_right={cathode_head:g} m | Pb left=flux0 right=open_outflow")
    return cfg

CFG = apply_water_initial_case(CFG)
CFG = apply_cathode_drainage_case(CFG)

# -------- helpers & constants --------
def pH_to_c_m3(pH):  return 10.0 ** (-float(pH)) * 1000.0
if CFG["FIELDS"]["HPLUS"]["INITIAL"]["c_ic"] is None:
    CFG["FIELDS"]["HPLUS"]["INITIAL"]["c_ic"] = pH_to_c_m3(CFG["FIELDS"]["HPLUS"]["INITIAL"]["pH_ic"])

DOMAIN = CFG["DOMAIN"]
def x_of_face(face_name):
    if face_name == "LEFT":  return float(DOMAIN["xmin"])
    if face_name == "RIGHT": return float(DOMAIN["xmax"])
    raise ValueError("face must be LEFT/RIGHT")

def signed_flux_along_pos_x(mag, face, direction):
    # positive x is to the right.
    if direction not in ("into_domain","out_of_domain"):
        raise ValueError("direction must be 'into_domain'/'out_of_domain'")
    if face == "LEFT":  return +mag if direction=="into_domain" else -mag
    return -mag if direction=="into_domain" else +mag

# ---- dataset grids ----
DATASET = CFG["NUMERICS"]["DATASET"]
pred_n = DATASET["pred_n"]
x_lin = np.linspace(DOMAIN["xmin"], DOMAIN["xmax"], pred_n).astype(np.float32)
t_lin = np.linspace(DOMAIN["tmin"], DOMAIN["tmax"], pred_n).astype(np.float32)
x_pred, t_pred = np.meshgrid(x_lin, t_lin)
t_star = t_pred.flatten().reshape(-1,1).astype(np.float32)
x_star = x_pred.flatten().reshape(-1,1).astype(np.float32)

def get_collocations(box, n):
    x = np.random.uniform(box["xmin"], box["xmax"], n).reshape(-1, 1)
    t = np.random.uniform(box["tmin"], box["tmax"], n).reshape(-1, 1)
    return t.astype(np.float32), x.astype(np.float32)

t_res, x_res = get_collocations(DOMAIN, DATASET["n_res"])
t_ic,  x_ic  = get_collocations({"xmin":DOMAIN["xmin"],"xmax":DOMAIN["xmax"],
                                 "tmin":DOMAIN["tmin"],"tmax":DOMAIN["tmin"]}, DATASET["n_ic"])

def face_samples(face, n, tmin_override=None):
    x0 = x_of_face(face)
    t0 = DOMAIN["tmin"] if tmin_override is None else max(float(tmin_override), DOMAIN["tmin"])
    t = np.random.uniform(t0, DOMAIN["tmax"], n).reshape(-1,1).astype(np.float32)
    x = np.full_like(t, x0, dtype=np.float32)
    return t, x

TF_CONFIG = tf.compat.v1.ConfigProto(allow_soft_placement=True, log_device_placement=False)

# ---- unpack ----
SOIL   = CFG["FIELDS"]["WATER"]["PARAM"]["SOIL"]
EO_PAR = CFG["GLOBAL"]["EO"]
ELEC_PAR = CFG["FIELDS"]["ELECTRIC"]["PARAM"]
ELEC_BD = CFG["FIELDS"]["ELECTRIC"]["BOUNDARY"]["electrodes"]
H_PAR, H_INIT, H_BC = CFG["FIELDS"]["HPLUS"]["PARAM"], CFG["FIELDS"]["HPLUS"]["INITIAL"], CFG["FIELDS"]["HPLUS"]["BOUNDARY"]
PB_PAR, PB_INIT, PB_BC = CFG["FIELDS"]["PB"]["PARAM"], CFG["FIELDS"]["PB"]["INITIAL"], CFG["FIELDS"]["PB"]["BOUNDARY"]
SURF_INIT = CFG["FIELDS"]["SURF_SOLID"]["INITIAL"]
W_INIT, W_BC = CFG["FIELDS"]["WATER"]["INITIAL"], CFG["FIELDS"]["WATER"]["BOUNDARY"]

ANODE_FACE   = ELEC_BD["anode_face"]; CATHODE_FACE = ELEC_BD["cathode_face"]
phi_anode, phi_cathode = ELEC_BD["phi_anode"], ELEC_BD["phi_cathode"]

# Fixed-current auto-fill is intentionally disabled here.
# Faradaic H/OH flux is computed later from prescribed voltage and Archie-type sigma_eff(Se).

# ---- constants ----
F_c = tf.constant(CFG["GLOBAL"]["CONSTANTS"]["F"], tf.float32)
R_c = tf.constant(CFG["GLOBAL"]["CONSTANTS"]["R"], tf.float32)
T_c = tf.constant(CFG["GLOBAL"]["CONSTANTS"]["T"], tf.float32)
Kw_const = tf.constant(CFG["GLOBAL"]["CHEM"]["Kw"], tf.float32)
sigma_sat_const = tf.constant(float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2))), tf.float32)
sigma_const = tf.constant(float(ELEC_PAR.get("sigma_const", ELEC_PAR.get("sigma_sat", 7.425e-2))), tf.float32)
sigma_s_const = tf.constant(float(ELEC_PAR.get("sigma_s", 1.8e-2)), tf.float32)
sigma_b_const = tf.constant(float(ELEC_PAR.get("sigma_b", 2.5e-1)), tf.float32)
rhoades_a_const = tf.constant(float(ELEC_PAR.get("rhoades_a", 1.3)), tf.float32)
rhoades_b_const = tf.constant(float(ELEC_PAR.get("rhoades_b", -0.2)), tf.float32)

DL_H = tf.constant(H_PAR["DL"], tf.float32); Dw_H = tf.constant(H_PAR["Dw"], tf.float32); z_H  = tf.constant(H_PAR["z"], tf.float32)
DL_OH = tf.constant(H_PAR.get("DL", H_PAR["DL"]), tf.float32)
Dw_OH = tf.constant(H_PAR.get("Dw_OH", 5.273e-9 * 86400.0), tf.float32)
z_OH  = tf.constant(float(H_PAR.get("z_OH", -1.0)), tf.float32)
A_CLIP = tf.constant(float(H_PAR.get("a_clip", 1.0e3)), tf.float32)
DL_Pb= tf.constant(PB_PAR["DL"], tf.float32); Dw_Pb= tf.constant(PB_PAR["Dw"], tf.float32); z_Pb= tf.constant(PB_PAR["z"], tf.float32)

# ---------------------------
# 1) Unsaturated functions in day-based units: robust variants
# ---------------------------
nvg = tf.constant([SOIL["n"]], tf.float32); mvg = 1. - 1./nvg
ksvg = tf.constant([SOIL["Ks"]], tf.float32)  # m/day
alphavg = tf.constant([SOIL["alpha"]], tf.float32)
thetaRvg = tf.constant([SOIL["theta_r"]], tf.float32)
thetaSvg = tf.constant([SOIL["theta_s"]], tf.float32)

zeta_mV_const = EO_PAR["zeta_mV"]; eps_w = EO_PAR["eps_w"]; mu_water_Pa_s = EO_PAR["mu_water_Pa_s"]

def grad0(y, x):
    g = tf.gradients(y, x, unconnected_gradients='zero')[0]
    if g is None:
        return tf.zeros_like(x)
    return g

# PDF-style nondimensional coordinates for all neural nets: t_bar in [0,1], x_bar in [0,1].
X_MIN_TF = tf.constant(float(DOMAIN["xmin"]), tf.float32)
T_MIN_TF = tf.constant(float(DOMAIN["tmin"]), tf.float32)
X_SCALE_TF = tf.constant(max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12), tf.float32)
T_SCALE_TF = tf.constant(max(float(DOMAIN["tmax"] - DOMAIN["tmin"]), 1e-12), tf.float32)

def nondim_tx(t, x):
    return tf.concat([(tf.cast(t, tf.float32) - T_MIN_TF) / T_SCALE_TF,
                      (tf.cast(x, tf.float32) - X_MIN_TF) / X_SCALE_TF], 1)

def nondim_X(X):
    X = tf.cast(X, tf.float32)
    return tf.concat([(X[:, 0:1] - T_MIN_TF) / T_SCALE_TF,
                      (X[:, 1:2] - X_MIN_TF) / X_SCALE_TF], 1)

def theta_function(h):
    # Unsaturated VG below water table; small saturated storage above it.
    s = tf.maximum(-alphavg * h, 0.0)                 # base >= 0
    term3 = tf.pow(1.0 + tf.pow(s, nvg), -mvg)
    theta_unsat = thetaRvg + (thetaSvg - thetaRvg) * term3
    Ss_theta = tf.constant(float(SOIL.get("Ss_theta", 0.0)), tf.float32)
    theta_sat = thetaSvg + Ss_theta * tf.maximum(h, 0.0)
    return tf.where(h >= 0.0, theta_sat, theta_unsat)

def K_function(h):
    theta_h = theta_function(h)
    Se = (theta_h - thetaRvg)/(thetaSvg-thetaRvg+1e-12)
    # Clay n is close to 1, so the Mualem derivative is singular at exact saturation.
    Se_clip = tf.clip_by_value(Se, 1e-8, 1.0 - 1e-6)
    inner = tf.clip_by_value(1.0 - tf.pow(Se_clip, 1.0/mvg), 1e-12, 1.0)
    term2 = 1.0 - tf.pow(inner, mvg)
    K_unsat = ksvg * tf.pow(Se_clip, 0.5) * tf.pow(term2, 2.0)
    sat_slope = tf.constant(float(SOIL.get("K_sat_head_slope", 0.0)), tf.float32)
    K_sat = ksvg * tf.exp(sat_slope * tf.maximum(h, 0.0))
    return tf.where(h >= 0.0, K_sat, K_unsat)  # m/day

def sat_vars(theta):
    theta_s = thetaSvg; theta_r = thetaRvg
    Se = (theta - theta_r) / (theta_s - theta_r + 1e-12)
    Se = tf.clip_by_value(Se, 1e-8, 1.0)
    return theta_s, theta_r, Se

def tau_from_Se(Se, model):
    if model == "bruggeman":
        return tf.pow(Se + 1e-30, -0.5)
    return tf.ones_like(Se)

# k_eo(theta): m^2/(V day)
def keo_of_theta(theta):
    _, _, Se = sat_vars(theta)
    tau_eo = tau_from_Se(Se, CFG["GLOBAL"]["POROUS"]["eo_tau_model"])
    chi_eo = tf.pow(Se, CFG["GLOBAL"]["POROUS"]["eo_power"])
    zeta = tf.constant(zeta_mV_const*1e-3, tf.float32)
    return (tf.constant(eps_w, tf.float32) * zeta * chi_eo * tf.constant(SEC_PER_DAY, tf.float32)) / (tf.constant(mu_water_Pa_s, tf.float32) * tau_eo)

def sigma_eff_of_theta(theta):
    """Rhoades Archie-type conductivity: sigma_s + sigma_b*theta*(a*theta+b)."""
    model = str(ELEC_PAR.get("sigma_model", "const")).lower()
    if model.startswith("archie") or model.startswith("rhoades"):
        theta_c = tf.maximum(theta, 0.0)
        sigma = sigma_s_const + sigma_b_const * theta_c * (rhoades_a_const * theta_c + rhoades_b_const)
        return tf.maximum(sigma, 1e-12)
    return tf.ones_like(theta) * sigma_const

def Dstar_of_theta(theta, Dw):
    # Modified Millington-Quirk power law: D*_sat/Dw = theta_s^(beta-2).
    theta_c = tf.clip_by_value(theta, 0.0, thetaSvg)
    beta = tf.constant(float(H_PAR.get("mq_theta_power", 10.0/3.0)), tf.float32)
    return Dw * tf.pow(theta_c, beta) / (tf.pow(thetaSvg, 2.0) + 1e-30)  # m^2/day

def diffusion_pieces(theta, q_adv, DL, Dw, z_val):
    Dstar = Dstar_of_theta(theta, Dw)
    Deff  = Dstar + DL * tf.abs(q_adv)            # m^2/day
    ustar = (z_val*F_c/(R_c*T_c)) * Dstar         # m^2/(V day)
    return Deff, ustar

# ---------------------------
# 2) Generic losses  (with per-term weights)
# ---------------------------
_EPS = 1e-12
def _loss_from_type(loss_type, err, lhs=None, rhs=None, delta=1.0):
    lt=loss_type.lower()
    if lt=="mse": return tf.reduce_mean(tf.square(err))
    if lt=="mae": return tf.reduce_mean(tf.abs(err))
    if lt=="huber":
        d=tf.constant(float(delta),tf.float32); ae=tf.abs(err)
        return tf.reduce_mean(tf.where(ae<=d,0.5*tf.square(err), d*(ae-0.5*d)))
    if lt=="relative_mse":
        denom = tf.reduce_mean(tf.square(rhs)) if rhs is not None else tf.reduce_mean(tf.square(lhs)) if lhs is not None else tf.constant(1.0,tf.float32)
        return tf.reduce_mean(tf.square(err))/(denom+_EPS)
    return tf.reduce_mean(tf.square(err))

def build_loss(terms_cfg, tensors):
    parts = {}; total = 0.0
    for tcfg in terms_cfg:
        name  = tcfg.get("name","term")
        ltype = tcfg.get("type","mse")
        delta = tcfg.get("delta",1.0)
        w_val = float(tcfg.get("w", 1.0))
        if "target" in tcfg:
            x = tensors[tcfg["target"]]
            li = _loss_from_type(ltype, x, delta=delta)
        else:
            lhs = tensors[tcfg["lhs"]]; rhs_key=tcfg.get("rhs","zero")
            rhs = tf.zeros_like(lhs) if rhs_key=="zero" else tensors[rhs_key]
            err = lhs - rhs
            li  = _loss_from_type(ltype, err, lhs=lhs, rhs=rhs, delta=delta)
        li_w = tf.constant(w_val, tf.float32) * li
        parts[name] = li_w
        total = total + parts[name]
    return total, parts

LOS = {
# Boundary-field weighting for stable enforcement
    "elec": {
        "terms": [
            {"name":"res","type":"mse","target":"residual_res", "w":1.0},
            {"name":"bc_left","type":"huber","lhs":"phi_left_pred","rhs":"phi_left_true","delta":0.01, "w":100.0},
            {"name":"bc_right","type":"huber","lhs":"phi_right_pred","rhs":"phi_right_true","delta":0.01, "w":100.0}
        ]
    },
    "hplus_flux": [
        {"name":"res","type":"relative_mse","target":"residual_res", "w":1.0},
        {"name":"left_flux","type":"mse","lhs":"J_left_pred","rhs":"J_left_true", "w":80.0},
        {"name":"right_flux","type":"mse","lhs":"J_right_pred","rhs":"J_right_true", "w":80.0},
        {"name":"ic","type":"mse","lhs":"c_ic_pred","rhs":"c_ic_true", "w":25.0}
    ],
}

# ---------------------------
# 3) Dense MLP
# ---------------------------
NETC = CFG["NUMERICS"]["NET"]; TRNC = CFG["NUMERICS"]["TRAIN"]; SEQC = CFG["NUMERICS"]["SEQUENTIAL"]
LOSS_HISTORY_BY_STAGE = []

def record_loss_history(stage, model, step_interval=100):
    hist = [float(v) for v in np.asarray(getattr(model, "loss_hist", []), dtype=float).reshape(-1) if np.isfinite(v)]
    record_loss_values(stage, hist, step_interval)

def record_loss_values(stage, hist, step_interval=100):
    hist = [float(v) for v in np.asarray(hist, dtype=float).reshape(-1) if np.isfinite(v)]
    LOSS_HISTORY_BY_STAGE.append({
        "stage": stage,
        "step_interval": int(step_interval),
        "loss": hist,
    })
    if hist:
        print(f"[LossHistory] {stage}: points={len(hist)}, first={hist[0]:.3e}, last={hist[-1]:.3e}")
    else:
        print(f"[LossHistory] {stage}: no recorded Adam loss points")
class DenseMLP:
    def __init__(self, layers, act_scale=10.0, trainable_gain=True):
        self.layers=layers; self.act_scale=act_scale; self.trainable_gain=trainable_gain
        self.weights=[]; self.biases=[]; self.A=[]
        for l in range(len(layers)-1):
            in_dim,out_dim=layers[l],layers[l+1]
            std=np.sqrt(2.0/(in_dim+out_dim))
            W=tf.Variable(tf.random.truncated_normal([in_dim,out_dim], stddev=std), dtype=tf.float32, trainable=True)
            b=tf.Variable(np.zeros([1,out_dim]), dtype=tf.float32, trainable=True)
            a=tf.Variable(0.05, dtype=tf.float32, trainable=trainable_gain)
            self.weights.append(W); self.biases.append(b); self.A.append(a)
    def forward(self, X):
        H=nondim_X(X)
        for l in range(len(self.weights)):
            H=tf.add(tf.matmul(H,self.weights[l]), self.biases[l])
            if l < len(self.weights)-1:
                H=tf.tanh(self.act_scale*self.A[l]*H)
        return H

def dense_eval_from_weights(weights, biases, gains, t, x):
    H = nondim_tx(t, x)
    for l in range(len(weights)):
        H = tf.add(tf.matmul(H, weights[l]), biases[l])
        if l < len(weights) - 1:
            H = tf.tanh(NETC["act_scale"] * gains[l] * H)
    return H

def water_head_from_raw(raw, t, x):
    # Bandai-Ghezzehei style pressure-head transform: h = beta - exp(N).
    # It keeps the water network in the unsaturated/saturated-pressure range
    # without hard-wiring the whole interior to a boundary value.
    beta = tf.constant(float(CFG["FIELDS"]["WATER"]["PARAM"].get("bandai_beta_m", 0.01)), tf.float32)
    psi = beta - tf.exp(tf.clip_by_value(raw, -30.0, 30.0))
    return tf.where(tf.math.is_finite(psi), psi, tf.zeros_like(psi))

def water_head_from_weights(weights, biases, gains, t, x):
    return water_head_from_raw(dense_eval_from_weights(weights, biases, gains, t, x), t, x)


def make_constant_water_weights(layers, psi_value=0.0):
    """Return non-trainable-looking DenseMLP weights that give a constant water head.

    The water network transform is psi = beta - exp(raw).  Setting all weights to
    zero and the final bias to log(beta - psi_value) makes
        water_head_from_weights(...) == psi_value
    everywhere.  For the saturated case we use psi_value=0, so theta=theta_s.
    """
    beta = float(CFG["FIELDS"]["WATER"]["PARAM"].get("bandai_beta_m", 0.01))
    raw_const = np.log(max(beta - float(psi_value), 1e-12))
    W = []
    B = []
    A = []
    for i in range(len(layers) - 1):
        W.append(np.zeros((int(layers[i]), int(layers[i + 1])), dtype=np.float32))
        b = np.zeros((1, int(layers[i + 1])), dtype=np.float32)
        if i == len(layers) - 2:
            b[:] = np.float32(raw_const)
        B.append(b)
        A.append(np.array(0.05, dtype=np.float32))
    return W, B, A


def use_fixed_saturated_water():
    """Auto-detect the saturated water-content case and skip WaterNet training.

    The run scripts set theta050 through psi_ic=0.  With the VG function used here,
    any non-negative pressure head gives the saturated branch; psi=0 gives exactly
    theta_s (0.50, unless changed in SOIL).  In this case training WaterNet is
    unnecessary and can introduce instability, so we freeze water at psi=0.
    """
    wp = CFG["FIELDS"]["WATER"].get("PARAM", {})
    if not bool(wp.get("skip_water_training_at_saturation", True)):
        return False
    psi_ic = float(CFG["FIELDS"]["WATER"].get("INITIAL", {}).get("psi_ic", 0.0))
    left_psi = float(CFG["FIELDS"]["WATER"].get("BOUNDARY", {}).get("left", {}).get("psi", psi_ic))
    right_psi = float(CFG["FIELDS"]["WATER"].get("BOUNDARY", {}).get("right", {}).get("psi", psi_ic))
    tol = float(wp.get("saturation_psi_tol", 1e-12))
    return (psi_ic >= -tol) and (left_psi >= -tol) and (right_psi >= -tol)

# ---------------------------
# 4) Water / Electric / H+ 
# ---------------------------
class WaterNet:
    def __init__(self, layers, include_eo_in_water=False, elec_weights=None, wbc_cfg=None, init_from=None, freeze_k=0):
        self.mlp=DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
        if init_from is not None:
            w0,b0,a0=init_from
            for i in range(len(self.mlp.weights)):
                self.mlp.weights[i]=tf.Variable(np.array(w0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.biases[i] =tf.Variable(np.array(b0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.A[i]      =tf.Variable(a0[i], dtype=tf.float32, trainable=(i>=freeze_k))
        (self.t_res,self.x_res,self.t_ic,self.x_ic,self.t_left,self.x_left,self.t_right,self.x_right)= \
            [tf.compat.v1.placeholder(tf.float32,[None,1]) for _ in range(8)]
        self.sess=tf.compat.v1.Session(config=TF_CONFIG)
        self.include_eo=include_eo_in_water; self.elec_weights=elec_weights
        self.wbc = wbc_cfg if wbc_cfg is not None else {"left": {"type":"dirichlet", "psi":W_INIT["psi_ic"]}, "right":{"type":"dirichlet", "psi":0.0}}

        # Core residuals
        self.psi_res,self.residual_res=self.net_res(self.t_res,self.x_res)
        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        Tspan = max(float(DOMAIN["tmax"] - DOMAIN["tmin"]), 1e-12)
        theta_scale_val = max(abs(float(SOIL["theta_s"]) - float(SOIL["theta_r"])), 1e-6)
        q_scale_val = max(abs(float(SOIL.get("Ks", 0.0))), 1e-12)
        water_res_norm = tf.constant(float(CFG["FIELDS"]["WATER"]["PARAM"].get("res_norm", max(theta_scale_val / Tspan, q_scale_val / Ldom, 1e-8))), tf.float32)
        self.residual_res_nd = self.residual_res / water_res_norm
        self.psi_ic_pred=self.net_psi(tf.concat([self.t_ic,self.x_ic],1))
        self.psi_ic_true=tf.fill(tf.shape(self.psi_ic_pred), tf.constant(W_INIT["psi_ic"],tf.float32))
        self.theta_ic_pred=theta_function(self.psi_ic_pred)
        self.theta_ic_true=theta_function(self.psi_ic_true)

        # ---- Build boundary conditions dynamically from config ----
        terms = [
            {"name":"res","type":"mse","target":"residual_res_nd", "w":1.0},
            {"name":"ic_theta", "type":"mse","lhs":"theta_ic_pred","rhs":"theta_ic_true", "w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("ic_theta_weight", 500.0))},
        ]
        tensors={"residual_res":self.residual_res, "residual_res_nd":self.residual_res_nd, "psi_ic_pred":self.psi_ic_pred, "psi_ic_true":self.psi_ic_true,
                 "theta_ic_pred":self.theta_ic_pred, "theta_ic_true":self.theta_ic_true}

        lcfg = self.wbc.get("left", {"type":"dirichlet", "psi":W_INIT["psi_ic"]})
        rcfg = self.wbc.get("right", {"type":"dirichlet", "psi":0.0})
        head_scale_val = max(abs(float(W_INIT["psi_ic"])), abs(float(lcfg.get("psi", W_INIT["psi_ic"]))), abs(float(rcfg.get("psi", 0.0))), 1.0)
        head_scale = tf.constant(head_scale_val, tf.float32)
        if str(lcfg.get("type", "dirichlet")).lower() != "dirichlet" or str(rcfg.get("type", "dirichlet")).lower() != "dirichlet":
            raise ValueError("Water boundary only supports dirichlet head in this model copy.")
        self.psi_left_pred = self.net_psi(tf.concat([self.t_left,self.x_left],1))
        _t0_left = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        _tau_left = tf.constant(float(CFG["FIELDS"]["WATER"]["PARAM"].get("head_transition_tau", 0.02)), tf.float32)
        _gate_left = tf.where(self.t_left <= _t0_left, tf.zeros_like(self.t_left), 1.0 - tf.exp(-(self.t_left - _t0_left) / tf.maximum(_tau_left, 1e-6)))
        self.psi_left_true = tf.constant(float(W_INIT["psi_ic"]), tf.float32) + _gate_left * (tf.constant(float(lcfg.get("psi", W_INIT["psi_ic"])), tf.float32) - tf.constant(float(W_INIT["psi_ic"]), tf.float32))
        self.theta_left_pred = theta_function(self.psi_left_pred)
        self.theta_left_true = theta_function(self.psi_left_true)
        self.psi_left_err_scaled = (self.psi_left_pred - self.psi_left_true) / head_scale
        terms.append({"name":"left_theta","type":"mse","lhs":"theta_left_pred","rhs":"theta_left_true","w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("boundary_theta_weight", 0.0))})
        terms.append({"name":"left_head_scaled","type":"mse","target":"psi_left_err_scaled","w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("boundary_head_weight", 80.0))})
        tensors.update({"psi_left_pred": self.psi_left_pred, "psi_left_true": self.psi_left_true, "psi_left_err_scaled": self.psi_left_err_scaled,
                        "theta_left_pred": self.theta_left_pred, "theta_left_true": self.theta_left_true})
        self.psi_right_pred = self.net_psi(tf.concat([self.t_right,self.x_right],1))
        _t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        _tau = tf.constant(float(CFG["FIELDS"]["WATER"]["PARAM"].get("head_transition_tau", 0.02)), tf.float32)
        _gate = tf.where(self.t_right <= _t0, tf.zeros_like(self.t_right), 1.0 - tf.exp(-(self.t_right - _t0) / tf.maximum(_tau, 1e-6)))
        self.psi_right_true = tf.constant(float(W_INIT["psi_ic"]), tf.float32) + _gate * (tf.constant(float(rcfg.get("psi", W_INIT["psi_ic"])), tf.float32) - tf.constant(float(W_INIT["psi_ic"]), tf.float32))
        self.theta_right_pred = theta_function(self.psi_right_pred)
        self.theta_right_true = theta_function(self.psi_right_true)
        self.psi_right_err_scaled = (self.psi_right_pred - self.psi_right_true) / head_scale
        terms.append({"name":"right_theta","type":"mse","lhs":"theta_right_pred","rhs":"theta_right_true","w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("boundary_theta_weight", 0.0))})
        terms.append({"name":"right_head_scaled","type":"mse","target":"psi_right_err_scaled","w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("boundary_head_weight", 80.0))})
        tensors.update({"psi_right_pred": self.psi_right_pred, "psi_right_true": self.psi_right_true, "psi_right_err_scaled": self.psi_right_err_scaled,
                        "theta_right_pred": self.theta_right_pred, "theta_right_true": self.theta_right_true})

        self.water_mass_integral_res = self.net_mass_integral_res()
        terms.append({"name":"water_mass_integral","type":"mse","target":"water_mass_integral_res",
                      "w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("mass_integral_weight", 0.0))})
        tensors["water_mass_integral_res"] = self.water_mass_integral_res

        # Build loss (with weights)
        self.loss,self.loss_parts=build_loss(terms,tensors)

        # Optimizer with global grad clipping
        self.global_step=tf.Variable(0,trainable=False)
        lr=tf.compat.v1.train.exponential_decay(TRNC["adam_lr"], self.global_step, 1000,0.9)
        opt = tf.compat.v1.train.AdamOptimizer(lr)
        vars_all = [v for v in (self.mlp.weights + self.mlp.biases + self.mlp.A) if getattr(v, "trainable", True)]
        grads_vars = [(g, v) for g, v in opt.compute_gradients(self.loss, var_list=vars_all) if g is not None]
        grads, vars_ = zip(*grads_vars)
        grads, _ = tf.clip_by_global_norm(grads, 5.0)
        self.train_op = opt.apply_gradients(list(zip(grads, vars_)), global_step=self.global_step)

        self.lbfgs=dde.optimizers.tensorflow_compat_v1.scipy_optimizer.ScipyOptimizerInterface(
            self.loss,method='L-BFGS-B',options=TRNC["lbfgs"])
        self.sess.run(tf.compat.v1.global_variables_initializer()); self.loss_hist=[]
        self._best_loss=np.inf; self._best_weights=None; self._best_it=None

    def net_psi(self, X): 
        return water_head_from_raw(self.mlp.forward(X), X[:, 0:1], X[:, 1:2])

    def _phi_x_if_needed(self,t,x):
        if (not self.include_eo) or (self.elec_weights is None):
            return None
        w_e,b_e,a_e=self.elec_weights
        H=nondim_tx(t,x)
        for l in range(len(w_e)):
            H=tf.add(tf.matmul(H,w_e[l]), b_e[l])
            if l < len(w_e)-1: H=tf.tanh(NETC["act_scale"]*a_e[l]*H)
        phi=H; return tf.gradients(phi,x)[0]

    def net_q_parts(self,t,x):
        X=tf.concat([t,x],1); psi=self.net_psi(X); K=K_function(psi); theta=theta_function(psi)
        psi_x=tf.gradients(psi,x)[0]; q_hyd=-K*(psi_x)
        q_eo=tf.zeros_like(q_hyd)
        if self.include_eo:
            phi_x=self._phi_x_if_needed(t,x)
            if phi_x is not None: q_eo=keo_of_theta(theta)*phi_x
        return q_hyd, q_eo, q_hyd + q_eo

    def net_q_total(self,t,x):
        # total water flux (hydraulic + electroosmotic if enabled)
        return self.net_q_parts(t,x)[2]

    def net_psix(self,t,x):
        X=tf.concat([t,x],1); psi=self.net_psi(X); return tf.gradients(psi,x)[0]

    def net_res(self,t,x):
        X=tf.concat([t,x],1); psi=self.net_psi(X); theta=theta_function(psi)
        q_tot=self.net_q_total(t,x)
        res=tf.gradients(q_tot,x)[0] + tf.gradients(theta,t)[0]
        return psi,res

    def net_mass_integral_res(self):
        wp = CFG["FIELDS"]["WATER"]["PARAM"]
        nx = max(int(wp.get("mass_integral_nx", 81)), 3)
        nt = max(int(wp.get("mass_integral_nt", 41)), 3)
        xmin = float(CFG["DOMAIN"]["xmin"]); xmax = float(CFG["DOMAIN"]["xmax"])
        tmin = float(CFG["DOMAIN"]["tmin"]); tmax = float(CFG["DOMAIN"]["tmax"])
        early = np.array([0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35], dtype=np.float32)
        uniform = np.linspace(tmin, tmax, nt, dtype=np.float32)
        t_vals = np.unique(np.clip(np.concatenate([uniform, early]), tmin, tmax)).astype(np.float32)
        t_vals.sort()
        nt_eff = int(t_vals.size)
        t_col = tf.constant(t_vals.reshape(-1, 1), tf.float32)
        x_row = tf.constant(np.linspace(xmin, xmax, nx, dtype=np.float32).reshape(1, -1), tf.float32)
        tt = tf.reshape(tf.tile(t_col, [1, nx]), [-1, 1])
        xx = tf.reshape(tf.tile(x_row, [nt_eff, 1]), [-1, 1])
        theta = tf.reshape(theta_function(self.net_psi(tf.concat([tt, xx], 1))), [nt_eff, nx])
        dx = tf.constant((xmax - xmin) / float(nx - 1), tf.float32)
        xw = tf.concat([tf.ones([1], tf.float32) * 0.5, tf.ones([nx - 2], tf.float32), tf.ones([1], tf.float32) * 0.5], 0)[None, :]
        S = dx * tf.reduce_sum(theta * xw, axis=1, keepdims=True)
        x_left = tf.ones_like(t_col) * tf.constant(xmin, tf.float32)
        x_right = tf.ones_like(t_col) * tf.constant(xmax, tf.float32)
        flux_out = self.net_q_total(t_col, x_right) - self.net_q_total(t_col, x_left)
        dt = t_col[1:] - t_col[:-1]
        step = 0.5 * (flux_out[1:] + flux_out[:-1]) * dt
        cum_flux = tf.concat([tf.zeros([1, 1], tf.float32), tf.cumsum(step, axis=0)], axis=0)
        L = tf.constant(max(xmax - xmin, 1e-6), tf.float32)
        return (S - S[0:1] + cum_flux) / L

    def _remember_best(self, loss, label):
        if np.isfinite(loss) and loss < self._best_loss:
            self._best_loss=float(loss); self._best_it=label; self._best_weights=self.export_weights()
            return True
        return False

    def _restore_best_checkpoint(self):
        if self._best_weights is None:
            return False
        w0,b0,a0=self._best_weights
        assigns=[]
        for var,val in zip(self.mlp.weights,w0): assigns.append(tf.compat.v1.assign(var,np.asarray(val,dtype=np.float32)))
        for var,val in zip(self.mlp.biases,b0): assigns.append(tf.compat.v1.assign(var,np.asarray(val,dtype=np.float32)))
        for var,val in zip(self.mlp.A,a0): assigns.append(tf.compat.v1.assign(var,np.asarray(val,dtype=np.float32)))
        self.sess.run(assigns)
        print(f"[Water] restored best checkpoint: {self._best_it}, L={self._best_loss:.3e}")
        return True

    def train(self,N_iter,batch=True,batch_size=512, face_feed=None):
        t_left,x_left = face_feed["left"]; t_right,x_right = face_feed["right"]
        names=sorted(self.loss_parts.keys())
        feed=None
        for it in range(N_iter):
            idx=np.random.choice(t_res.shape[0],batch_size,replace=False); tr,xr=t_res[idx,:],x_res[idx,:]
            feed={self.t_res:tr,self.x_res:xr,self.t_ic:t_ic,self.x_ic:x_ic,
                  self.t_left:t_left,self.x_left:x_left,self.t_right:t_right,self.x_right:x_right}
            self.sess.run(self.train_op,feed)
            if it%100==0:
                vals=self.sess.run([self.loss]+[self.loss_parts[n] for n in names],feed)
                loss_now=float(vals[0])
                best_mark=" *best" if self._remember_best(loss_now, f"adam:{it}") else ""
                print(f"[Water] It {it:5d} L={loss_now:.3e}{best_mark} | "+", ".join([f"{n}={v:.2e}" for n,v in zip(names,vals[1:])]))
                self.loss_hist.append(loss_now)
        if feed is not None:
            self.lbfgs.minimize(self.sess, feed_dict=feed, fetches=[self.loss])
            loss_after=float(self.sess.run(self.loss, feed))
            self._remember_best(loss_after, "lbfgs")
            print("[Water] LBFGS done.")
            self._restore_best_checkpoint()

    def export_weights(self):
        return self.sess.run(self.mlp.weights), self.sess.run(self.mlp.biases), self.sess.run(self.mlp.A)

class ElectricNet:
    def __init__(self,layers,init_from=None,freeze_k=0, phi_left=10.0, phi_right=0.0, water_weights=None):
        self.mlp=DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
        if init_from is not None:
            w0,b0,a0=init_from
            for i in range(len(self.mlp.weights)):
                self.mlp.weights[i]=tf.Variable(np.array(w0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.biases[i] =tf.Variable(np.array(b0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.A[i]      =tf.Variable(a0[i], dtype=tf.float32, trainable=(i>=freeze_k))
        (self.t_res,self.x_res,self.t_left,self.x_left,self.t_right,self.x_right)= \
            [tf.compat.v1.placeholder(tf.float32,[None,1]) for _ in range(6)]
        self.water_weights = water_weights
        if water_weights is not None:
            ww, wb, wa = water_weights
            self.water_w = [tf.constant(np.array(W), tf.float32) for W in ww]
            self.water_b = [tf.constant(np.array(B), tf.float32) for B in wb]
            self.water_a = [tf.constant(np.array(A), tf.float32) for A in wa]
        self.sess=tf.compat.v1.Session(config=TF_CONFIG)
        self.phi_res,self.residual_res=self.net_res(self.t_res,self.x_res)
        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        dphi = max(abs(float(phi_left) - float(phi_right)), 1.0)
        sigma_ref = max(abs(float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2)))), 1e-12)
        elec_res_norm = tf.constant(float(ELEC_PAR.get("res_norm", max(sigma_ref * dphi / (Ldom * Ldom), 1e-8))), tf.float32)
        self.residual_res_nd = self.residual_res / elec_res_norm
        self.phi_left_pred=self.net_phi(tf.concat([self.t_left,self.x_left],1))
        self.phi_right_pred=self.net_phi(tf.concat([self.t_right,self.x_right],1))
        self.phi_left_true=tf.fill(tf.shape(self.phi_left_pred), tf.constant(phi_left,tf.float32))
        self.phi_right_true=tf.fill(tf.shape(self.phi_right_pred), tf.constant(phi_right,tf.float32))
        phi_scale = tf.constant(dphi, tf.float32)
        self.phi_left_pred_nd = self.phi_left_pred / phi_scale
        self.phi_left_true_nd = self.phi_left_true / phi_scale
        self.phi_right_pred_nd = self.phi_right_pred / phi_scale
        self.phi_right_true_nd = self.phi_right_true / phi_scale
        tensors={"residual_res":self.residual_res_nd,
                 "phi_left_pred":self.phi_left_pred_nd,"phi_left_true":self.phi_left_true_nd,
                 "phi_right_pred":self.phi_right_pred_nd,"phi_right_true":self.phi_right_true_nd}
        self.loss,self.loss_parts=build_loss(LOS["elec"]["terms"],tensors)

        # Optimizer with global grad clipping
        self.global_step=tf.Variable(0,trainable=False)
        lr=tf.compat.v1.train.exponential_decay(TRNC["adam_lr"], self.global_step, 1000,0.9)
        opt = tf.compat.v1.train.AdamOptimizer(lr)
        grads_vars = opt.compute_gradients(self.loss)
        grads, vars_ = zip(*grads_vars)
        grads, _ = tf.clip_by_global_norm(grads, 5.0)
        self.train_op = opt.apply_gradients(list(zip(grads, vars_)), global_step=self.global_step)

        self.lbfgs=dde.optimizers.tensorflow_compat_v1.scipy_optimizer.ScipyOptimizerInterface(
            self.loss,method='L-BFGS-B',options=TRNC["lbfgs"])
        self.sess.run(tf.compat.v1.global_variables_initializer()); self.loss_hist=[]
    def net_phi(self,X): return self.mlp.forward(X)
    def net_res(self,t,x):
        X=tf.concat([t,x],1); phi=self.net_phi(X); phi_x=tf.gradients(phi,x)[0]
        if self.water_weights is None:
            sigma=tf.fill(tf.shape(t), sigma_sat_const)
        else:
            psi = water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
            sigma = sigma_eff_of_theta(theta_function(psi))
        return phi, tf.gradients(sigma*phi_x,x)[0]
    def train(self,N_iter,batch=True,batch_size=512, face_feed=None):
        t_left,x_left = face_feed["left"]; t_right,x_right = face_feed["right"]
        names=sorted(self.loss_parts.keys())
        for it in range(N_iter):
            idx=np.random.choice(t_res.shape[0],batch_size,replace=False); tr,xr=t_res[idx,:],x_res[idx,:]
            feed={self.t_res:tr,self.x_res:xr,self.t_left:t_left,self.x_left:x_left,self.t_right:t_right,self.x_right:x_right}
            self.sess.run(self.train_op,feed)
            if it%100==0:
                vals=self.sess.run([self.loss]+[self.loss_parts[n] for n in names],feed)
                print(f"[Elec ] It {it:5d} L={vals[0]:.3e} | "+", ".join([f"{n}={v:.2e}" for n,v in zip(names,vals[1:])]))
                self.loss_hist.append(vals[0])
        self.lbfgs.minimize(self.sess, feed_dict=feed, fetches=[self.loss]); print("[Elec ] LBFGS done.")
    def export_weights(self): return self.sess.run(self.mlp.weights), self.sess.run(self.mlp.biases), self.sess.run(self.mlp.A)

class AcidBaseNet:
    """
    Transported state: a = c_H - c_OH  [mol/m^3_water]
    Closure:
        c_H  = (a + sqrt(a^2 + 4Kw))/2
        c_OH = (-a + sqrt(a^2 + 4Kw))/2

    Boundary strategy:
    - anode (left): prescribe species-resolved non-advective Faradaic flux
        JH_F = +JF, JOH_F = 0
    - cathode (right): prescribe species-resolved non-advective Faradaic flux
        JH_F = 0, JOH_F = -JF
      This removes the acid-equivalent wrong-branch solution in which OH flux
      is used at the anode, or H flux at the cathode, to satisfy Ja = JH - JOH.
    """
    def __init__(self, layers, water_weights, elec_weights, init_from=None, freeze_k=0,
                 include_eo=True, leftBC="flux", rightBC="flux",
                 JH_left_true_val=0.0, JOH_left_true_val=0.0,
                 JH_right_true_val=0.0, JOH_right_true_val=0.0,
                 a_ic_val=0.0,
                 cH_left_dir=None, cH_right_dir=None,
                 chem_kernel=None, pb_weights=None, include_pb_source=False):
        self.mlp=DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
        if init_from is not None:
            w0,b0,a0=init_from
            for i in range(len(self.mlp.weights)):
                self.mlp.weights[i]=tf.Variable(np.array(w0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.biases[i] =tf.Variable(np.array(b0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.A[i]      =tf.Variable(a0[i], dtype=tf.float32, trainable=(i>=freeze_k))

        self.water_w,self.water_b,self.water_a=water_weights
        self.elec_w,self.elec_b,self.elec_a=elec_weights
        self.water_w = [tf.constant(np.array(W), tf.float32) for W in self.water_w]
        self.water_b = [tf.constant(np.array(B), tf.float32) for B in self.water_b]
        self.water_a = [tf.constant(np.array(A), tf.float32) for A in self.water_a]
        self.elec_w  = [tf.constant(np.array(W), tf.float32) for W in self.elec_w]
        self.elec_b  = [tf.constant(np.array(B), tf.float32) for B in self.elec_b]
        self.elec_a  = [tf.constant(np.array(A), tf.float32) for A in self.elec_a]

        self.chem = chem_kernel
        self.include_pb_source = bool(include_pb_source)
        self.pb_w = self.pb_b = self.pb_a = None
        if pb_weights is not None:
            w_pb, b_pb, a_pb = pb_weights
            self.pb_w = [tf.constant(np.array(W), tf.float32) for W in w_pb]
            self.pb_b = [tf.constant(np.array(B), tf.float32) for B in b_pb]
            self.pb_a = [tf.constant(np.array(A), tf.float32) for A in a_pb]

        self.include_eo=bool(include_eo)
        self.leftBC=str(leftBC).lower()
        self.rightBC=str(rightBC).lower()

        (self.t_res,self.x_res,self.t_ic,self.x_ic,self.t_left,self.x_left,self.t_right,self.x_right)= \
            [tf.compat.v1.placeholder(tf.float32,[None,1]) for _ in range(8)]
        self.sess=tf.compat.v1.Session(config=TF_CONFIG)

        self.a_res, self.cH_res, self.cOH_res, self.residual_a_res = self.net_res(self.t_res,self.x_res)
        self.a_ic_pred = self.net_a(tf.concat([self.t_ic,self.x_ic],1))
        self.a_ic_true = tf.fill(tf.shape(self.a_ic_pred), tf.constant(a_ic_val, tf.float32))

        self.JH_left_pred  = self.net_JH(self.t_left,self.x_left)
        self.JOH_left_pred = self.net_JOH(self.t_left,self.x_left)
        self.JH_right_pred = self.net_JH(self.t_right,self.x_right)
        self.JOH_right_pred= self.net_JOH(self.t_right,self.x_right)
        self.Ja_left_pred  = self.JH_left_pred - self.JOH_left_pred
        self.Ja_right_pred = self.JH_right_pred - self.JOH_right_pred
        self.JaF_left_pred  = self.net_Ja_nonadv(self.t_left,self.x_left)
        self.JaF_right_pred = self.net_Ja_nonadv(self.t_right,self.x_right)
        self.JH_F_left_pred, self.JOH_F_left_pred = self.net_species_nonadv(self.t_left,self.x_left)
        self.JH_F_right_pred, self.JOH_F_right_pred = self.net_species_nonadv(self.t_right,self.x_right)

        self.c_left_pred  = self.net_c(tf.concat([self.t_left,self.x_left],1))
        self.c_right_pred = self.net_c(tf.concat([self.t_right,self.x_right],1))
        self.cOH_left_pred  = self.net_cOH(tf.concat([self.t_left,self.x_left],1))
        self.cOH_right_pred = self.net_cOH(tf.concat([self.t_right,self.x_right],1))
        self.pH_left_pred  = self.net_pH_unclipped(tf.concat([self.t_left,self.x_left],1))
        self.pH_right_pred = self.net_pH_unclipped(tf.concat([self.t_right,self.x_right],1))

        ramp_left = self._faraday_ramp(self.t_left)
        ramp_right = self._faraday_ramp(self.t_right)
        dynamic_faraday = str(H_BC.get("faraday_current_mode", "fixed_current")).lower() == "archie_voltage"
        if dynamic_faraday:
            Jmag_left = self._faraday_flux_from_archie_current(self.t_left, self.x_left)
            Jmag_right = self._faraday_flux_from_archie_current(self.t_right, self.x_right)
            self.JH_left_true = Jmag_left * ramp_left
            self.JOH_left_true = tf.zeros_like(self.JH_left_true)
            self.JH_right_true = tf.zeros_like(Jmag_right)
            self.JOH_right_true = -Jmag_right * ramp_right
        else:
            sat_left = self._faraday_saturation_scale(self.t_left, self.x_left)
            sat_right = self._faraday_saturation_scale(self.t_right, self.x_right)
            self.JH_left_true   = tf.fill(tf.shape(self.JH_left_pred), tf.constant(JH_left_true_val, tf.float32)) * ramp_left * sat_left
            self.JOH_left_true  = tf.fill(tf.shape(self.JOH_left_pred), tf.constant(JOH_left_true_val, tf.float32)) * ramp_left * sat_left
            self.JH_right_true  = tf.fill(tf.shape(self.JH_right_pred), tf.constant(JH_right_true_val, tf.float32)) * ramp_right * sat_right
            self.JOH_right_true = tf.fill(tf.shape(self.JOH_right_pred), tf.constant(JOH_right_true_val, tf.float32)) * ramp_right * sat_right
        self.Ja_left_true   = self.JH_left_true - self.JOH_left_true
        self.Ja_right_true  = self.JH_right_true - self.JOH_right_true
        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        eta_i = float(H_BC.get("current_efficiency", 1.0))
        dphi = abs(float(ELEC_BD["phi_anode"]) - float(ELEC_BD["phi_cathode"]))
        Ja_ref_val = max(eta_i * float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2))) * dphi / Ldom / float(CFG["GLOBAL"]["CONSTANTS"]["F"]) * SEC_PER_DAY, 1e-8)
        self.res_a_ref = tf.fill(tf.shape(self.residual_a_res), tf.constant(Ja_ref_val / Ldom, tf.float32))

        if cH_left_dir is None:
            cH_left_dir = pH_to_c_m3(H_BC.get("pH_left", H_INIT["pH_ic"]))
        if cH_right_dir is None:
            cH_right_dir = pH_to_c_m3(H_BC.get("pH_right", H_INIT["pH_ic"]))
        self.pH_left_true = self._boundary_pH_target(self.t_left, "left")
        self.pH_right_true = self._boundary_pH_target(self.t_right, "right")
        self.c_left_true = self._pH_to_cH_tensor(self.pH_left_true)
        self.c_right_true = self._pH_to_cH_tensor(self.pH_right_true)
        self.a_left_pred = self.net_a(tf.concat([self.t_left, self.x_left], 1))
        self.a_right_pred = self.net_a(tf.concat([self.t_right, self.x_right], 1))
        self.a_left_true = self._a_from_pH_tensor(self.pH_left_true)
        self.a_right_true = self._a_from_pH_tensor(self.pH_right_true)
        self.a_bc_ref = tf.constant(float(H_PAR.get("a_dirichlet_loss_scale", 10.0)), tf.float32)

        def _finite(z):
            return tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))
        def _mse(z):
            return tf.reduce_mean(tf.square(z))
        def _rel_mse(err, ref):
            return tf.reduce_mean(tf.square(err)) / (tf.reduce_mean(tf.square(ref)) + 1e-12)

        self.residual_a_nd = _finite(self.residual_a_res / tf.maximum(self.res_a_ref, 1e-12))
        self.residual_weight_ph = tf.compat.v1.placeholder_with_default(
            tf.constant(float(H_PAR.get("residual_weight", 1.0)), tf.float32), shape=())
        self.faraday_flux_weight_ph = tf.compat.v1.placeholder_with_default(
            tf.constant(float(H_PAR.get("faraday_flux_weight", 140.0)), tf.float32), shape=())
        self.coion_flux_weight_ph = tf.compat.v1.placeholder_with_default(
            tf.constant(float(H_PAR.get("coion_flux_weight", 0.0)), tf.float32), shape=())
        loss_parts = {
            "res_a": self.residual_weight_ph * _mse(self.residual_a_nd),
        }

        # Species-resolved electrode boundary conditions.
        # The transported state is still a = cH - cOH, but the Faradaic boundary
        # is no longer imposed only through Ja_F = JH_F - JOH_F.  We constrain
        # the two non-advective species fluxes separately so the network cannot
        # use the wrong co-ion branch to satisfy the acid-equivalent flux:
        #   anode:   JH_F = +JF,  JOH_F = 0
        #   cathode: JH_F = 0,    JOH_F = -JF
        # Advective transport is supplied by the separately trained water field.
        faraday_mode = str(H_PAR.get("faraday_bc_mode", "acid_equivalent")).lower()
        pH_loss_scale = tf.constant(float(H_BC.get("pH_loss_scale", 10.0)), tf.float32)
        pH_dirichlet_weight = float(H_BC.get("pH_dirichlet_weight", 30.0))
        if self.leftBC == "dirichlet":
            loss_parts["left_dirichlet_a"] = pH_dirichlet_weight * _mse(_finite((self.a_left_pred - self.a_left_true) / self.a_bc_ref))
        elif faraday_mode in ("acid_equivalent", "species", "species_flux", "species_nonadv"):
            loss_parts["left_HF_flux"] = self.faraday_flux_weight_ph * _rel_mse(
                _finite(self.JH_F_left_pred - self.JH_left_true), self.JH_left_true
            )
            if float(H_PAR.get("coion_flux_weight", 0.0)) > 0.0:
                loss_parts["left_OHF_zero"] = self.coion_flux_weight_ph * _rel_mse(
                    _finite(self.JOH_F_left_pred - self.JOH_left_true), self.JH_left_true
                )
        else:
            loss_parts["left_H_flux"]  = self.faraday_flux_weight_ph * _rel_mse(_finite(self.JH_left_pred  - self.JH_left_true),  self.JH_left_true)
            loss_parts["left_OH_flux"] = 40.0  * _mse(_finite(self.JOH_left_pred - self.JOH_left_true))

        if self.rightBC == "dirichlet":
            loss_parts["right_dirichlet_a"] = pH_dirichlet_weight * _mse(_finite((self.a_right_pred - self.a_right_true) / self.a_bc_ref))
        elif faraday_mode in ("acid_equivalent", "species", "species_flux", "species_nonadv"):
            if float(H_PAR.get("coion_flux_weight", 0.0)) > 0.0:
                loss_parts["right_HF_zero"] = self.coion_flux_weight_ph * _rel_mse(
                    _finite(self.JH_F_right_pred - self.JH_right_true), self.JOH_right_true
                )
            loss_parts["right_OHF_flux"] = self.faraday_flux_weight_ph * _rel_mse(
                _finite(self.JOH_F_right_pred - self.JOH_right_true), self.JOH_right_true
            )
        else:
            loss_parts["right_H_flux"]  = 40.0  * _mse(_finite(self.JH_right_pred  - self.JH_right_true))
            loss_parts["right_OH_flux"] = self.faraday_flux_weight_ph * _rel_mse(_finite(self.JOH_right_pred - self.JOH_right_true), self.JOH_right_true)

        pH_w = float(H_PAR.get("pH_obs_weight", 0.0))
        if pH_w > 0.0:
            loss_parts["weak_pH_left"]  = pH_w * _mse(_finite(self.pH_left_pred  - self.pH_left_true))
            loss_parts["weak_pH_right"] = pH_w * _mse(_finite(self.pH_right_pred - self.pH_right_true))

        # Physics-only admissibility hinge: electrolysis should make the anode acidic
        # and cathode alkaline. Ramp the hinge with the Faradaic ramp so it does not
        # conflict with the neutral initial condition at t=0.
        pH_range_w = float(H_PAR.get("pH_range_weight", 0.0))
        if pH_range_w > 0.0:
            pH_left_max = tf.constant(float(H_PAR.get("pH_left_max_phys", 4.0)), tf.float32)
            pH_right_min = tf.constant(float(H_PAR.get("pH_right_min_phys", 10.0)), tf.float32)
            if bool(H_PAR.get("pH_range_ramped", True)):
                pH0 = tf.constant(float(H_INIT.get("pH_ic", 7.0)), tf.float32)
                left_limit = pH0 - (pH0 - pH_left_max) * ramp_left
                right_limit = pH0 + (pH_right_min - pH0) * ramp_right
            else:
                left_limit = pH_left_max
                right_limit = pH_right_min
            loss_parts["phys_pH_left_acid"] = pH_range_w * _mse(tf.nn.relu(_finite(self.pH_left_pred) - left_limit))
            loss_parts["phys_pH_right_alk"] = pH_range_w * _mse(tf.nn.relu(right_limit - _finite(self.pH_right_pred)))

        self.loss_parts = loss_parts
        self.loss = tf.add_n(list(self.loss_parts.values()))

        self.c_res = self.cH_res
        self.residual_res = self.residual_a_res
        self.J_left_pred = self.Ja_left_pred
        self.J_right_pred = self.Ja_right_pred
        self.c_ic_pred = self.net_c(tf.concat([self.t_ic,self.x_ic],1))
        self.c_ic_true = tf.fill(tf.shape(self.c_ic_pred), tf.constant(float(H_INIT["c_ic"]), tf.float32))

        self.global_step=tf.Variable(0,trainable=False)
        base_lr = float(TRNC.get("hplus_adam_lr", TRNC["adam_lr"]))
        lr=tf.compat.v1.train.exponential_decay(base_lr, self.global_step, 1000,0.9)
        opt = tf.compat.v1.train.AdamOptimizer(lr)
        vars_all = [v for v in (self.mlp.weights + self.mlp.biases + self.mlp.A) if getattr(v, "trainable", True)]
        grads_vars = [(g, v) for g, v in opt.compute_gradients(self.loss, var_list=vars_all) if g is not None]
        if not grads_vars:
            raise ValueError("AcidBaseNet has no trainable gradients; check freeze_k_c and net_a.")
        grads, vars_ = zip(*grads_vars)
        safe_grads, finite_counts, total_counts = [], [], []
        for g in grads:
            g = tf.convert_to_tensor(g)
            finite_mask = tf.math.is_finite(g)
            finite_counts.append(tf.reduce_sum(tf.cast(finite_mask, tf.float32)))
            total_counts.append(tf.cast(tf.size(g), tf.float32))
            safe_grads.append(tf.where(finite_mask, g, tf.zeros_like(g)))
        self.grad_finite_frac = tf.add_n(finite_counts) / tf.maximum(tf.add_n(total_counts), 1.0)
        self.grad_norm = tf.linalg.global_norm(safe_grads)
        safe_grads, _ = tf.clip_by_global_norm(safe_grads, float(TRNC.get("hplus_grad_clip", 5.0)))
        self.train_op = opt.apply_gradients(list(zip(safe_grads, vars_)), global_step=self.global_step)
        self.lbfgs=dde.optimizers.tensorflow_compat_v1.scipy_optimizer.ScipyOptimizerInterface(
            self.loss,method='L-BFGS-B',options=TRNC["lbfgs"])
        self.sess.run(tf.compat.v1.global_variables_initializer())
        self.loss_hist=[]
        # Python-side adaptive loss scales used only as feed_dict multipliers.
        # They do not change the graph structure and are reset for every AcidBaseNet instance.
        self._adapt_scale_res = 1.0
        self._adapt_scale_flux = 1.0
        self._adapt_scale_coion = 1.0
        self._res_cache = None
        self._res_cache_meta = None
        self._best_loss = np.inf
        self._best_weights = None
        self._best_it = None
        self._zero_grad_warned = False

    def _faraday_ramp(self, t):
        tau = tf.constant(float(H_PAR.get("faraday_ramp_tau", 0.0)), tf.float32)
        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        if float(H_PAR.get("faraday_ramp_tau", 0.0)) <= 0.0:
            return tf.ones_like(t)
        return 1.0 - tf.exp(-tf.maximum(t - t0, 0.0) / tf.maximum(tau, 1e-12))

    def _faraday_saturation_scale(self, t, x):
        return tf.ones_like(t)

    def _faraday_flux_from_archie_current(self, t, x):
        psi = water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
        theta = theta_function(psi)
        _, phi_x = self._phi_grad(t, x)
        sigma_eff = sigma_eff_of_theta(theta)
        eta_i = tf.constant(float(H_BC.get("current_efficiency", 1.0)), tf.float32)
        i_mag = tf.abs(-sigma_eff * phi_x)  # A/m^2 under prescribed voltage
        J_day = eta_i * i_mag / F_c * tf.constant(SEC_PER_DAY, tf.float32)
        return tf.where(tf.math.is_finite(J_day), J_day, tf.zeros_like(J_day))

    def _faraday_flux_from_initial_theta_current(self, t):
        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        dphi = abs(float(ELEC_BD["phi_anode"]) - float(ELEC_BD["phi_cathode"]))
        if "_initial_sigma_eff_num" in globals():
            sigma = float(_initial_sigma_eff_num())
        else:
            sigma = float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2)))
        eta_i = float(H_BC.get("current_efficiency", 1.0))
        J_const = eta_i * sigma * dphi / Ldom / float(CFG["GLOBAL"]["CONSTANTS"]["F"]) * SEC_PER_DAY
        return tf.fill(tf.shape(t), tf.constant(J_const, tf.float32))

    def _a_from_pH_tensor(self, pH):
        ln10 = tf.constant(np.log(10.0), tf.float32)
        cH = 1000.0 * tf.exp(-ln10 * pH)
        cOH = Kw_const / tf.maximum(cH, 1e-30)
        return cH - cOH

    def _pH_to_cH_tensor(self, pH):
        ln10 = tf.constant(np.log(10.0), tf.float32)
        return 1000.0 * tf.exp(-ln10 * pH)

    def _boundary_pH_target(self, t, side):
        mode = str(H_BC.get("dirichlet_mode", "static")).lower()
        key = "pH_left" if side == "left" else "pH_right"
        pH0 = tf.constant(float(H_INIT.get("pH_ic", 7.0)), tf.float32)
        if mode not in ("reservoir", "reservoir_ph"):
            return tf.fill(tf.shape(t), tf.constant(float(H_BC.get(key, H_INIT.get("pH_ic", 7.0))), tf.float32))

        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        if str(H_BC.get("faraday_current_mode", "fixed_current")).lower() == "archie_voltage" and H_BC.get("I_app_Aperm2") is None:
            J_day = self._faraday_flux_from_initial_theta_current(t)
        else:
            if H_BC.get("I_app_Aperm2") is None:
                dphi = abs(float(ELEC_BD["phi_anode"]) - float(ELEC_BD["phi_cathode"]))
                Iapp = float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2))) * dphi / Ldom
            else:
                Iapp = float(H_BC["I_app_Aperm2"])
            J_const = float(H_BC.get("current_efficiency", 1.0)) * Iapp / float(CFG["GLOBAL"]["CONSTANTS"]["F"]) * SEC_PER_DAY
            J_day = tf.fill(tf.shape(t), tf.constant(J_const, tf.float32))
        depth = max(float(H_BC.get("reservoir_depth_m", Ldom)), 1e-12)
        tday = tf.maximum(t - tf.constant(float(DOMAIN["tmin"]), tf.float32), 0.0)
        c0_H = float(H_INIT.get("c_ic", pH_to_c_m3(H_INIT.get("pH_ic", 7.0))))
        c0_OH = float(CFG["GLOBAL"]["CHEM"]["Kw"]) / max(c0_H, 1e-30)
        c = tf.constant(c0_H if side == "left" else c0_OH, tf.float32) + (J_day / tf.constant(depth, tf.float32)) * tday
        ln10 = tf.constant(np.log(10.0), tf.float32)
        if side == "left":
            raw = -tf.math.log(tf.maximum(c / 1000.0, 1e-14)) / ln10
        else:
            raw = 14.0 + tf.math.log(tf.maximum(c / 1000.0, 1e-14)) / ln10
        tau = tf.constant(float(H_BC.get("reservoir_pH_ramp_tau", H_PAR.get("faraday_ramp_tau", 0.01))), tf.float32)
        ramp = 1.0 - tf.exp(-tday / tf.maximum(tau, 1e-12))
        target = pH0 + ramp * (raw - pH0)
        if bool(H_BC.get("reservoir_clip_to_config", True)):
            limit = tf.constant(float(H_BC.get(key, H_INIT.get("pH_ic", 7.0))), tf.float32)
            target = tf.maximum(target, limit) if side == "left" else tf.minimum(target, limit)
        return tf.clip_by_value(target, float(H_PAR.get("pH_min", 0.0)), float(H_PAR.get("pH_max", 14.5)))

    def net_a(self, X):
        # Do not impose a left-acid/right-alkaline profile in the interior.
        # The acid/base front should be learned from PDE + Faraday boundary fluxes.
        raw0 = self.mlp.forward(X)
        raw = 8.0 * tf.tanh(raw0 / 8.0)
        a_scale = tf.constant(float(H_PAR.get("a_raw_scale", 20.0)), tf.float32)

        t = X[:, 0:1]
        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        tau_gate = tf.constant(float(H_PAR.get("a_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(
            t <= t0,
            tf.zeros_like(t),
            1.0 - tf.exp(-(t - t0) / tf.maximum(tau_gate, 1e-12)),
        )
        a_ic = tf.constant(float(H_INIT.get("a_ic", 0.0)), tf.float32)
        a_free = a_ic + a_scale * raw
        a = (1.0 - gate) * a_ic + gate * a_free
        return tf.where(tf.math.is_finite(a), a, tf.zeros_like(a))

    def acid_base_from_a(self, a):
        # Stable algebraic Kw closure for a = cH - cOH and cH*cOH = Kw.
        s = tf.sqrt(tf.square(a) + 4.0 * Kw_const)
        cH_pos = 0.5 * (a + s)
        cH_neg = (2.0 * Kw_const) / tf.maximum(s - a, 1e-30)
        cOH_neg = 0.5 * (-a + s)
        cOH_pos = (2.0 * Kw_const) / tf.maximum(s + a, 1e-30)
        cH = tf.where(a >= 0.0, cH_pos, cH_neg)
        cOH = tf.where(a <= 0.0, cOH_neg, cOH_pos)
        cH = tf.where(tf.math.is_finite(cH), cH, tf.zeros_like(cH))
        cOH = tf.where(tf.math.is_finite(cOH), cOH, tf.zeros_like(cOH))
        return cH, cOH

    def net_c(self, X):
        cH, _ = self.acid_base_from_a(self.net_a(X))
        return cH

    def net_cOH(self, X):
        _, cOH = self.acid_base_from_a(self.net_a(X))
        return cOH

    def net_pH_unclipped(self, X):
        cH = tf.maximum(self.net_c(X), 1e-30)
        ln10 = tf.constant(np.log(10.0), tf.float32)
        pH = -tf.math.log(cH / 1000.0) / ln10
        return tf.where(tf.math.is_finite(pH), pH, tf.zeros_like(pH))

    def net_pH(self, X):
        pH = self.net_pH_unclipped(X)
        pH_min = tf.constant(float(H_PAR.get("pH_min", 0.0)), tf.float32)
        pH_max = tf.constant(float(H_PAR.get("pH_max", 14.5)), tf.float32)
        return tf.clip_by_value(pH, pH_min, pH_max)

    def _water_fields(self,t,x):
        psi=water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
        theta=theta_function(psi)
        K=K_function(psi)
        psi_x=grad0(psi,x)
        q_hyd=-K*psi_x
        return theta,q_hyd

    def _phi_grad(self,t,x):
        H=nondim_tx(t,x)
        for l in range(len(self.elec_w)):
            H=tf.add(tf.matmul(H,self.elec_w[l]), self.elec_b[l])
            if l<len(self.elec_w)-1:
                H=tf.tanh(NETC["act_scale"]*self.elec_a[l]*H)
        phi=H
        return phi, grad0(phi,x)

    def _species_fields(self,t,x):
        X=tf.concat([t,x],1)
        a = self.net_a(X)
        cH, cOH = self.acid_base_from_a(a)
        cHx = grad0(cH, x)
        cOHx = grad0(cOH, x)
        theta,q_hyd=self._water_fields(t,x)
        _,phi_x=self._phi_grad(t,x)
        q_adv=q_hyd + (keo_of_theta(theta)*phi_x if self.include_eo else 0.0)
        return a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x

    def net_JH(self,t,x):
        a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x = self._species_fields(t,x)
        Deff_H, u_H = diffusion_pieces(theta, q_adv, DL_H, Dw_H, z_H)
        JH = q_adv*cH - Deff_H*cHx - u_H*cH*phi_x
        return tf.where(tf.math.is_finite(JH), JH, tf.zeros_like(JH))

    def net_JOH(self,t,x):
        a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x = self._species_fields(t,x)
        Deff_OH, u_OH = diffusion_pieces(theta, q_adv, DL_OH, Dw_OH, z_OH)
        JOH = q_adv*cOH - Deff_OH*cOHx - u_OH*cOH*phi_x
        return tf.where(tf.math.is_finite(JOH), JOH, tf.zeros_like(JOH))

    def net_Ja(self,t,x):
        return self.net_JH(t,x) - self.net_JOH(t,x)

    def net_species_nonadv(self,t,x):
        a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x = self._species_fields(t,x)
        Deff_H, u_H = diffusion_pieces(theta, q_adv, DL_H, Dw_H, z_H)
        Deff_OH, u_OH = diffusion_pieces(theta, q_adv, DL_OH, Dw_OH, z_OH)
        JH_F = -Deff_H*cHx - u_H*cH*phi_x
        JOH_F = -Deff_OH*cOHx - u_OH*cOH*phi_x
        finite = lambda z: tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))
        return finite(JH_F), finite(JOH_F)

    def net_Ja_nonadv(self,t,x):
        # Faraday boundary constrains only non-advective electrochemical flux:
        # Ja_F = Ja - q*a = (JH-JOH) - q*(cH-cOH).
        JH_F, JOH_F = self.net_species_nonadv(t,x)
        JaF = JH_F - JOH_F
        return tf.where(tf.math.is_finite(JaF), JaF, tf.zeros_like(JaF))

    def net_J(self,t,x):
        # compatibility: return H+ flux
        return self.net_JH(t,x)

    def _pb_psi_from_weights(self, t, x):
        if self.pb_w is None:
            return None
        H=nondim_tx(t,x)
        for l in range(len(self.pb_w)):
            H = tf.add(tf.matmul(H, self.pb_w[l]), self.pb_b[l])
            if l < len(self.pb_w) - 1:
                H = tf.tanh(NETC["act_scale"] * self.pb_a[l] * H)

        # Use the same Psi_Pb ansatz as PbInvNet.net_Psi. This makes the
        # acid/base coupling source see the same transported Pb inventory as
        # the Pb transport model, including the hard initial condition.
        raw_scale = tf.constant(float(PB_PAR.get("psi_raw_scale", 20.0)), tf.float32)
        raw = raw_scale * tf.tanh(H[:, 0:1] / tf.maximum(raw_scale, 1e-6))

        Hw = water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
        theta0 = theta_function(Hw)
        psi0_store = tf.constant(float(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"]), tf.float32)
        c_ic_water = tf.constant(float(CFG["FIELDS"]["PB"]["INITIAL"]["c_ic"]), tf.float32)
        Psi0 = tf.maximum(theta0 * c_ic_water + psi0_store, 1e-6)
        eta0 = tf.math.log(tf.math.expm1(Psi0) + 1e-12)

        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        tau = tf.constant(float(PB_PAR.get("psi_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(t <= t0, tf.zeros_like(t), 1.0 - tf.exp(-(t - t0) / tf.maximum(tau, 1e-6)))
        Psi = tf.nn.softplus(eta0 + gate * raw)
        if self.chem is not None:
            Psi = tf.clip_by_value(Psi, 0.0, self.chem.S_tot + 1.0e4)
        return tf.where(tf.math.is_finite(Psi), Psi, tf.zeros_like(Psi))

    def _pb_acid_source(self, t, x, theta_loc, cH_w, cOH_w):
        """Frozen previous-iterate Pb chemistry source for one acid/base block."""
        if (not self.include_pb_source) or (self.chem is None) or (self.pb_w is None):
            return tf.zeros_like(t)
        Psi_b = self._pb_psi_from_weights(t, x)
        if Psi_b is None:
            return tf.zeros_like(t)
        # Kim-style coupled source: previous Pb fixes CT and the active Ksp
        # branch; precipitated Pb still responds to the current pH field.
        *_, active_gate = self.chem.reconstruct_pb_equil(t, x, Psi_b, return_gate=True)
        _, _, _, _, _, Pp, _ = self.chem.reconstruct_pb_equil_from_fields(
            theta_loc, cH_w, cOH_w, Psi_b, t,
            precip_gate_override=active_gate > 0.5,
            stop_acid=bool(H_PAR.get("pb_source_stop_gradient", True)),
        )
        S_ab = 2.0 * grad0(Pp, t)
        S_ab = tf.where(tf.math.is_finite(S_ab), S_ab, tf.zeros_like(S_ab))
        if bool(H_PAR.get("pb_source_stop_gradient", True)):
            S_ab = tf.stop_gradient(S_ab)
        return S_ab

    def net_res(self,t,x):
        a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x = self._species_fields(t,x)
        Deff_H,  u_H  = diffusion_pieces(theta, q_adv, DL_H,  Dw_H,  z_H)
        Deff_OH, u_OH = diffusion_pieces(theta, q_adv, DL_OH, Dw_OH, z_OH)
        JH  = q_adv*cH  - Deff_H*cHx   - u_H*cH*phi_x
        JOH = q_adv*cOH - Deff_OH*cOHx - u_OH*cOH*phi_x
        Ja = JH - JOH
        theta_t = grad0(theta, t)
        cH_t = grad0(cH, t)
        cOH_t = grad0(cOH, t)
        S_pb = self._pb_acid_source(t, x, theta, cH, cOH)
        RH = tf.constant(float(H_PAR.get("H_retardation", 1.0)), tf.float32)
        # Kim Eq. (11)/(13), extended to variable water content.
        storage_H = RH * (theta * cH_t + theta_t * cH)
        storage_OH = theta * cOH_t + theta_t * cOH
        storage = storage_H - storage_OH
        res = storage + grad0(Ja,x) - S_pb
        finite = lambda z: tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))
        return finite(a), finite(cH), finite(cOH), finite(res)

    def _weight_feed(self, it, N_iter):
        """Return feed_dict weights. Base weights follow the existing schedule;
        optional adaptive scales equalize weighted loss-group contributions."""
        frac = float(it + 1) / max(float(N_iter), 1.0)
        schedule = H_PAR.get("training_schedule", None)
        res_w = float(H_PAR.get("residual_weight", 1.0))
        flux_w = float(H_PAR.get("faraday_flux_weight", 140.0))
        if schedule:
            for stage in schedule:
                if frac <= float(stage.get("until", 1.0)):
                    res_w = float(stage.get("residual_weight", res_w))
                    flux_w = float(stage.get("faraday_flux_weight", flux_w))
                    break
        coion_w = float(H_PAR.get("coion_flux_weight", 0.0))

        if bool(H_PAR.get("adaptive_loss_weights", True)):
            res_w *= float(self._adapt_scale_res)
            flux_w *= float(self._adapt_scale_flux)
            coion_w *= float(self._adapt_scale_coion)

        # The co-ion zero-flux constraints are branch-selection constraints,
        # not ordinary residual terms.  Do not let adaptive weighting weaken
        # them below the configured floor.
        if coion_w > 0.0:
            coion_floor = float(H_PAR.get("coion_flux_weight_min", H_PAR.get("coion_flux_weight", 0.0)))
            coion_w = max(coion_w, coion_floor)

        self._last_weight_values = (res_w, flux_w, coion_w)
        return {
            self.residual_weight_ph: res_w,
            self.faraday_flux_weight_ph: flux_w,
            self.coion_flux_weight_ph: coion_w,
        }, res_w, flux_w, coion_w

    def _sample_uniform_tx(self, n, t_low=None, t_high=None):
        xmin, xmax = float(DOMAIN["xmin"]), float(DOMAIN["xmax"])
        tmin = float(DOMAIN["tmin"]) if t_low is None else float(t_low)
        tmax = float(DOMAIN["tmax"]) if t_high is None else float(t_high)
        tmax = max(tmax, tmin + 1e-9)
        tr = np.random.uniform(tmin, tmax, (int(n), 1)).astype(np.float32)
        xr = np.random.uniform(xmin, xmax, (int(n), 1)).astype(np.float32)
        return tr, xr

    def _sample_face_window(self, template_t, template_x, t_low=None, t_high=None):
        """Sample boundary points in the current time slab while preserving face x."""
        n = int(template_t.shape[0])
        tmin = float(DOMAIN["tmin"]) if t_low is None else float(t_low)
        tmax = float(DOMAIN["tmax"]) if t_high is None else float(t_high)
        tmax = max(tmax, tmin + 1e-9)
        tr = np.random.uniform(tmin, tmax, (n, 1)).astype(np.float32)
        xval = float(np.asarray(template_x[:1]).reshape(-1)[0])
        xr = np.full((n, 1), xval, dtype=np.float32)
        return tr, xr

    def _refresh_residual_cache(self, t_low=None, t_high=None):
        """Residual-based adaptive refinement (RAR/RBAS).

        A large candidate pool is sampled uniformly inside the active time window.
        The current AcidBase PDE residual is evaluated on the candidates, and the
        highest-|residual| points are cached.  The training batch then draws most
        collocation points from this cache, plus a small uniform background.
        No governing equation, boundary condition, or chemistry expression is changed.
        """
        n_cand = int(H_PAR.get("residual_cache_candidates", 8192))
        n_keep = int(H_PAR.get("residual_cache_keep", max(512, n_cand // 2)))
        n_cand = max(1, n_cand)
        n_keep = max(1, min(n_keep, n_cand))
        tc, xc = self._sample_uniform_tx(n_cand, t_low, t_high)
        try:
            res_val = self.sess.run(
                self.residual_a_res,
                {self.t_res: tc, self.x_res: xc}
            )
            res_abs = np.abs(np.asarray(res_val).reshape(-1))
            res_abs = np.where(np.isfinite(res_abs), res_abs, 0.0)
            power = float(H_PAR.get("residual_score_power", 1.0))
            power = max(power, 1e-6)
            score = np.power(res_abs + 1e-30, power)
            if np.all(score <= 0.0):
                keep = np.random.choice(n_cand, n_keep, replace=False)
            else:
                keep = np.argpartition(score, -n_keep)[-n_keep:]
            self._res_cache = (tc[keep, :].astype(np.float32), xc[keep, :].astype(np.float32))
            self._res_cache_score = score[keep].astype(np.float64)
            self._res_cache_meta = (float(t_low) if t_low is not None else None,
                                    float(t_high) if t_high is not None else None)
        except Exception as exc:
            print(f"[A/B RAR] residual cache refresh skipped: {exc}")
            self._res_cache = None
            self._res_cache_score = None
            self._res_cache_meta = None

    def _residual_batch(self, batch_size, t_low=None, t_high=None, it=0):
        """Build a residual-collocation batch for the active time slab.

        If adaptive sampling is enabled, the batch is:
            residual-adaptive points from the high-|PDE residual| cache
            + uniform background points.
        This is true residual-based adaptive sampling; electrode/early-time
        heuristics are intentionally not used here.
        """
        tmin = float(DOMAIN["tmin"]) if t_low is None else float(t_low)
        tmax = float(DOMAIN["tmax"]) if t_high is None else float(t_high)
        tmax = max(tmax, tmin + 1e-9)
        batch_size = int(batch_size)

        if (not bool(H_PAR.get("adaptive_residual_sampling", True))) or batch_size <= 0:
            return self._sample_uniform_tx(batch_size, tmin, tmax)

        adapt_frac = float(H_PAR.get("residual_adaptive_frac", 0.70))
        uniform_frac = float(H_PAR.get("residual_uniform_frac", max(0.0, 1.0 - adapt_frac)))
        adapt_frac = min(max(adapt_frac, 0.0), 1.0)
        uniform_frac = min(max(uniform_frac, 0.0), 1.0)
        # Normalize if the two fractions were edited inconsistently.
        sfrac = adapt_frac + uniform_frac
        if sfrac <= 0.0:
            return self._sample_uniform_tx(batch_size, tmin, tmax)
        adapt_frac /= sfrac
        n_adapt = int(round(batch_size * adapt_frac))
        n_adapt = max(0, min(batch_size, n_adapt))
        n_uniform = batch_size - n_adapt

        ts, xs = [], []

        # Uniform background coverage.
        if n_uniform > 0:
            tr_u, xr_u = self._sample_uniform_tx(n_uniform, tmin, tmax)
            ts.append(tr_u); xs.append(xr_u)

        # High-residual adaptive points.
        if n_adapt > 0:
            refresh_every = int(H_PAR.get("residual_cache_refresh", 50))
            meta = (float(tmin), float(tmax))
            cache_missing = self._res_cache is None or self._res_cache_meta != meta
            if cache_missing or (refresh_every > 0 and it % refresh_every == 0):
                self._refresh_residual_cache(tmin, tmax)
            if self._res_cache is not None:
                ct, cx = self._res_cache
                idx = np.random.choice(ct.shape[0], n_adapt, replace=(ct.shape[0] < n_adapt))
                ts.append(ct[idx, :]); xs.append(cx[idx, :])
            else:
                tr_a, xr_a = self._sample_uniform_tx(n_adapt, tmin, tmax)
                ts.append(tr_a); xs.append(xr_a)

        tr = np.vstack(ts).astype(np.float32)
        xr = np.vstack(xs).astype(np.float32)
        perm = np.random.permutation(tr.shape[0])
        return tr[perm, :], xr[perm, :]

    def _update_adaptive_loss_scales(self, names, vals):
        """Python-side adaptive weighting based on current weighted loss groups.
        It balances residual / Faradaic-flux / co-ion diagnostic groups without
        changing the governing equations."""
        if not bool(H_PAR.get("adaptive_loss_weights", True)):
            return
        groups = {"res": 0.0, "flux": 0.0, "coion": 0.0}
        for name, val in zip(names, vals):
            v = float(val)
            if not np.isfinite(v):
                continue
            if name == "res_a":
                groups["res"] += max(v, 0.0)
            elif (("JaF_flux" in name) or ("HF_flux" in name) or ("OHF_flux" in name)
                  or ("_H_flux" in name) or ("_OH_flux" in name)):
                groups["flux"] += max(v, 0.0)
            elif ("HF_zero" in name) or ("OHF_zero" in name):
                groups["coion"] += max(v, 0.0)

        active = {k: v for k, v in groups.items() if v > 1e-16}
        if len(active) < 2:
            return
        logs = [np.log(v + 1e-16) for v in active.values()]
        target = float(np.exp(np.mean(logs)))
        alpha = float(H_PAR.get("adaptive_loss_alpha", 0.35))
        min_s = float(H_PAR.get("adaptive_loss_min_scale", 0.2))
        max_s = float(H_PAR.get("adaptive_loss_max_scale", 5.0))

        def _upd(current, value):
            factor = (target / (value + 1e-16)) ** alpha
            return float(np.clip(current * factor, min_s, max_s))

        if "res" in active:
            self._adapt_scale_res = _upd(self._adapt_scale_res, active["res"])
        if "flux" in active:
            self._adapt_scale_flux = _upd(self._adapt_scale_flux, active["flux"])
        if "coion" in active:
            self._adapt_scale_coion = _upd(self._adapt_scale_coion, active["coion"])

    def _remember_best(self, loss_value, it_label):
        loss_value = float(loss_value)
        if np.isfinite(loss_value) and loss_value < self._best_loss:
            self._best_loss = loss_value
            self._best_weights = self.export_weights()
            self._best_it = it_label
            return True
        return False

    def _restore_best_checkpoint(self):
        if self._best_weights is None:
            return False
        w0, b0, a0 = self._best_weights
        assigns = []
        for var, val in zip(self.mlp.weights, w0):
            assigns.append(tf.compat.v1.assign(var, np.asarray(val, dtype=np.float32)))
        for var, val in zip(self.mlp.biases, b0):
            assigns.append(tf.compat.v1.assign(var, np.asarray(val, dtype=np.float32)))
        for var, val in zip(self.mlp.A, a0):
            assigns.append(tf.compat.v1.assign(var, np.asarray(val, dtype=np.float32)))
        self.sess.run(assigns)
        return True

    def _train_window(self, N_iter, batch_size, face_feed, t_low=None, t_high=None, slab_label="full", run_lbfgs=False):
        base_t_left, base_x_left = face_feed["left"]
        base_t_right, base_x_right = face_feed["right"]
        names = sorted(self.loss_parts.keys())
        feed = None
        update_every = int(H_PAR.get("adaptive_loss_update_every", 100))
        t_low_print = float(DOMAIN["tmin"]) if t_low is None else float(t_low)
        t_high_print = float(DOMAIN["tmax"]) if t_high is None else float(t_high)
        print(f"[A/B ] training window {slab_label}: t in [{t_low_print:.4g}, {t_high_print:.4g}] day, iters={N_iter}")

        for it in range(int(N_iter)):
            tr, xr = self._residual_batch(batch_size, t_low, t_high, it)
            if bool(H_PAR.get("time_slab_training", False)):
                tl, xl = self._sample_face_window(base_t_left, base_x_left, t_low, t_high)
                trgt, xrgt = self._sample_face_window(base_t_right, base_x_right, t_low, t_high)
            else:
                tl, xl = base_t_left, base_x_left
                trgt, xrgt = base_t_right, base_x_right

            wfeed, res_w, flux_w, coion_w = self._weight_feed(it, N_iter)
            feed = {
                self.t_res: tr, self.x_res: xr,
                self.t_ic: t_ic, self.x_ic: x_ic,
                self.t_left: tl, self.x_left: xl,
                self.t_right: trgt, self.x_right: xrgt,
            }
            feed.update(wfeed)
            self.sess.run(self.train_op, feed)

            if (update_every > 0) and (it % update_every == 0):
                vals_for_adapt = self.sess.run([self.loss_parts[n] for n in names], feed)
                self._update_adaptive_loss_scales(names, vals_for_adapt)

            if it % 100 == 0:
                vals = self.sess.run([self.loss, self.grad_norm, self.grad_finite_frac] + [self.loss_parts[n] for n in names], feed)
                loss_now = float(vals[0])
                grad_now = float(vals[1])
                finite_now = float(vals[2])
                best_mark = " *best" if self._remember_best(loss_now, f"{slab_label}:{it}") else ""
                bad_grad = finite_now < float(H_PAR.get("min_grad_finite_frac", 0.5))
                if (grad_now <= 1e-12 or bad_grad) and not self._zero_grad_warned:
                    print(f"[A/B warn] near-zero/non-finite gradient at {slab_label} It {it}; finite_grad={finite_now:.2f}.")
                    self._zero_grad_warned = True
                print(
                    f"[A/B ] {slab_label} It {it:5d} L={loss_now:.3e}{best_mark} | "
                    f"grad={grad_now:.2e}, finite_grad={finite_now:.2f} | "
                    f"w_res={res_w:.2g}, w_flux={flux_w:.2g}, w_coion={coion_w:.2g} | "
                    f"scale=({self._adapt_scale_res:.2g},{self._adapt_scale_flux:.2g},{self._adapt_scale_coion:.2g}) | "
                    + ", ".join([f"{n}={v:.2e}" for n, v in zip(names, vals[3:])])
                )
                self.loss_hist.append(loss_now)
                if bad_grad:
                    print(f"[A/B warn] stopping {slab_label} early and restoring best checkpoint because gradients are non-finite.")
                    self._restore_best_checkpoint()
                    break

        if feed is None:
            tr, xr = self._residual_batch(batch_size, t_low, t_high, 0)
            tl, xl = self._sample_face_window(base_t_left, base_x_left, t_low, t_high)
            trgt, xrgt = self._sample_face_window(base_t_right, base_x_right, t_low, t_high)
            wfeed, _, _, _ = self._weight_feed(0, 1)
            feed = {
                self.t_res: tr, self.x_res: xr,
                self.t_ic: t_ic, self.x_ic: x_ic,
                self.t_left: tl, self.x_left: xl,
                self.t_right: trgt, self.x_right: xrgt,
            }
            feed.update(wfeed)

        if run_lbfgs and bool(TRNC.get("hplus_use_lbfgs", True)):
            self.lbfgs.minimize(self.sess, feed_dict=feed, fetches=[self.loss])
            loss_after_lbfgs = float(self.sess.run(self.loss, feed))
            self._remember_best(loss_after_lbfgs, f"{slab_label}:lbfgs")
            print(f"[A/B ] LBFGS done for {slab_label}.")
        return feed

    def train(self, N_iter, batch=True, batch_size=512, face_feed=None):
        if face_feed is None:
            raise ValueError("AcidBaseNet.train requires face_feed with left/right boundary arrays.")

        # Time-slab continuation: train successively on expanding time windows.
        # This keeps early sharp pH transients learned before exposing the network to the full 0--tmax domain.
        if bool(H_PAR.get("time_slab_training", False)):
            tmin = float(DOMAIN["tmin"])
            tmax = float(DOMAIN["tmax"])
            raw_slabs = list(H_PAR.get("time_slabs", [tmax]))
            slabs = sorted({float(s) for s in raw_slabs if float(s) > tmin})
            if len(slabs) == 0 or slabs[-1] < tmax - 1e-12:
                slabs.append(tmax)
            slabs = [min(s, tmax) for s in slabs]
            n_slabs = len(slabs)
            base_iters = int(N_iter) // n_slabs
            rem = int(N_iter) - base_iters * n_slabs
            last_feed = None
            previous_end = tmin
            for k, slab_end in enumerate(slabs):
                slab_iters = base_iters + (1 if k < rem else 0)
                if slab_iters <= 0:
                    continue
                mode = str(H_PAR.get("time_slab_mode", "expanding")).lower()
                if mode == "disjoint":
                    slab_low = previous_end
                else:
                    slab_low = tmin
                slab_label = f"slab{k+1}/{n_slabs}@{slab_end:g}d"
                self._res_cache = None
                self._res_cache_meta = None
                last_feed = self._train_window(
                    slab_iters, batch_size, face_feed,
                    t_low=slab_low, t_high=slab_end,
                    slab_label=slab_label,
                    run_lbfgs=False,
                )
                previous_end = slab_end
            if bool(TRNC.get("hplus_use_lbfgs", True)) and last_feed is not None:
                # One final LBFGS pass on the full sampled domain, not the last random batch.
                base_t_left, base_x_left = face_feed["left"]
                base_t_right, base_x_right = face_feed["right"]
                wfeed, _, _, _ = self._weight_feed(max(int(N_iter) - 1, 0), max(int(N_iter), 1))
                full_feed = {
                    self.t_res: t_res, self.x_res: x_res,
                    self.t_ic: t_ic, self.x_ic: x_ic,
                    self.t_left: base_t_left, self.x_left: base_x_left,
                    self.t_right: base_t_right, self.x_right: base_x_right,
                }
                full_feed.update(wfeed)
                self.lbfgs.minimize(self.sess, feed_dict=full_feed, fetches=[self.loss])
                print("[A/B ] final full-domain LBFGS done.")
            else:
                print("[A/B ] LBFGS skipped.")
            if self._restore_best_checkpoint():
                print(f"[A/B ] restored best checkpoint: {self._best_it}, L={self._best_loss:.3e}")
            return

        self._train_window(
            N_iter, batch_size, face_feed,
            t_low=float(DOMAIN["tmin"]), t_high=float(DOMAIN["tmax"]),
            slab_label="full",
            run_lbfgs=bool(TRNC.get("hplus_use_lbfgs", True)),
        )
        if self._restore_best_checkpoint():
            print(f"[A/B ] restored best checkpoint: {self._best_it}, L={self._best_loss:.3e}")

    def export_weights(self):
        return self.sess.run(self.mlp.weights), self.sess.run(self.mlp.biases), self.sess.run(self.mlp.A)

HPlusNet = AcidBaseNet

# ---------------------------
# 5) Chemistry equilibrium kernel + invariant reconstruction (stable)
# ---------------------------
class ChemEquilKernel:
    """
    Algebraic reconstruction for Pb component chemistry.
    Inputs via nets: theta(t,x), cH_w(t,x), cOH_w(t,x)
    Unknowns (bulk): SOH, SOH2+, SO-, SOPb+, Pp; water species include free Pb2+
    and mononuclear Pb-OH complexes. The transported component is:
        Psi_Pb = theta*C_Pb,aq,total + SOPb + Pp
    where precipitation saturation is controlled by free Pb2+ activity proxy.
    """
    def __init__(self, cfg_pb_chem, water_w, ab_w, surf_init_bulk):
        self.k = cfg_pb_chem
        self.m = tf.constant(float(cfg_pb_chem.get("m", 2.0)), tf.float32)
        self.Ksp = tf.constant(float(cfg_pb_chem.get("Ksp_PbOH2", 1.43e-11)), tf.float32)
        self.water_w = water_w
        self.ab_w = ab_w
        self.K_pr   = tf.constant(self.k["k_pr_f"]/self.k["k_pr_b"], tf.float32)
        self.K_dprp = tf.constant(self.k["k_dpr_f"]/self.k["k_dpr_b"], tf.float32)
        self.K_ad   = tf.constant(self.k["k_ad_f"]/self.k["k_ad_b"], tf.float32)
        self.SOPb0 = tf.constant(float(surf_init_bulk["SOPb0"] + surf_init_bulk.get("Pp0", 0.0)), tf.float32)
        self.S_tot = tf.constant(float(surf_init_bulk["SOH0"] + surf_init_bulk["SOPb0"] +
                                       surf_init_bulk["SOH2_0"] + surf_init_bulk["SOm0"]), tf.float32)
        hyd = cfg_pb_chem.get("HYDROLYSIS", {})
        log_beta_L = list(hyd.get("log_beta_OH_molL", [6.54, 11.06, 13.97, 15.20]))
        while len(log_beta_L) < 4:
            log_beta_L.append(-300.0)
        if not bool(hyd.get("include", True)):
            log_beta_L = [-300.0, -300.0, -300.0, -300.0]
        self.beta_oh_m3 = [10.0 ** (float(log_beta_L[j]) - 3.0 * float(j + 1)) for j in range(4)]
        self.beta_oh = [tf.constant(v, tf.float32) for v in self.beta_oh_m3]

    def _hydrolysis_factors_tf(self, cOH_w):
        cOH = tf.maximum(cOH_w, 1e-30)
        b1 = self.beta_oh[0] * cOH
        b2 = self.beta_oh[1] * tf.square(cOH)
        b3 = self.beta_oh[2] * tf.pow(cOH, 3.0)
        b4 = self.beta_oh[3] * tf.pow(cOH, 4.0)
        alpha = tf.clip_by_value(1.0 + b1 + b2 + b3 + b4, 1.0, 1e16)
        z_num = 2.0 + b1 - b3 - 2.0 * b4
        z_eff = tf.clip_by_value(z_num / tf.maximum(alpha, 1e-30), -2.0, 2.0)
        return alpha, z_eff, (b1, b2, b3, b4)

    def _mlp_eval(self, weights, t, x):
        w,b,a = weights
        H = nondim_tx(t, x)
        for l in range(len(w)):
            H = tf.add(tf.matmul(H, w[l]), b[l])
            if l < len(w)-1:
                H = tf.tanh(NETC["act_scale"] * a[l] * H)
        return H

    def theta(self, t, x):
        w,b,a = self.water_w
        psi = water_head_from_weights(w, b, a, t, x)
        return theta_function(psi)

    def _a_from_pH_tensor(self, pH):
        ln10 = tf.constant(np.log(10.0), tf.float32)
        cH = 1000.0 * tf.exp(-ln10 * pH)
        cOH = Kw_const / tf.maximum(cH, 1e-30)
        return cH - cOH

    def _ab_net_a(self, t, x):
        raw = tf.clip_by_value(self._mlp_eval(self.ab_w, t, x), -8.0, 8.0)
        a_scale = tf.constant(float(H_PAR.get("a_raw_scale", 20.0)), tf.float32)
        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        tau_gate = tf.constant(float(H_PAR.get("a_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(
            t <= t0,
            tf.zeros_like(t),
            1.0 - tf.exp(-(t - t0) / tf.maximum(tau_gate, 1e-12)),
        )
        a_ic = tf.constant(float(H_INIT.get("a_ic", 0.0)), tf.float32)
        a_free = a_ic + a_scale * raw
        a_val = (1.0 - gate) * a_ic + gate * a_free
        return tf.where(tf.math.is_finite(a_val), a_val, tf.zeros_like(a_val))

    def _acid_base_conc(self, t, x):
        a_val = self._ab_net_a(t, x)
        s = tf.sqrt(tf.square(a_val) + 4.0 * Kw_const)
        cH_pos = 0.5 * (a_val + s)
        cH_neg = (2.0 * Kw_const) / tf.maximum(s - a_val, 1e-30)
        cOH_neg = 0.5 * (-a_val + s)
        cOH_pos = (2.0 * Kw_const) / tf.maximum(s + a_val, 1e-30)
        cH = tf.where(a_val >= 0.0, cH_pos, cH_neg)
        cOH = tf.where(a_val <= 0.0, cOH_neg, cOH_pos)
        cH = tf.where(tf.math.is_finite(cH), cH, tf.zeros_like(cH))
        cOH = tf.where(tf.math.is_finite(cOH), cOH, tf.zeros_like(cOH))
        return cH, cOH

    def _acid_base_a(self, t, x):
        return self._ab_net_a(t, x)

    def cH_w(self, t, x):
        cH, _ = self._acid_base_conc(t, x)
        return cH

    def cOH_w(self, t, x):
        _, cOH = self._acid_base_conc(t, x)
        return cOH

    def reconstruct_pb_equil_from_fields(self, theta_loc, cH_w, cOH_w, Psi_b, t,
                                         precip_gate_override=None, stop_acid=True,
                                         return_gate=False):
        def finite(z):
            return tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))

        Psi_b = tf.clip_by_value(finite(Psi_b), 0.0, self.S_tot + 1.0e4)
        theta_loc = tf.stop_gradient(tf.clip_by_value(finite(theta_loc), 1e-6, 1.0))
        cH_w = tf.clip_by_value(finite(cH_w), 1e-12, 1e3)
        cOH_w = tf.clip_by_value(finite(cOH_w), 1e-10, 1e4)
        if stop_acid:
            cH_w = tf.stop_gradient(cH_w)
            cOH_w = tf.stop_gradient(cOH_w)
        cH_b = tf.maximum(theta_loc * cH_w, 1e-12)

        # Kim local partition: native pool + incoming/reactive pool.
        # cPb_aq_total remains water-basis; theta*cPb_aq_total is the bulk Paq.
        aH = self.K_pr * cH_b
        bH = self.K_dprp / cH_b
        B = tf.clip_by_value(1.0 + aH + bH, 1.0, 1e8)

        ln10 = tf.constant(np.log(10.0), tf.float32)
        pH_loc = -tf.math.log(tf.maximum(cH_w, 1e-30) / 1000.0) / ln10
        alpha_native = tf.clip_by_value(0.27 * pH_loc - 0.23, 0.0, 1.0)
        alpha_native = tf.clip_by_value(finite(alpha_native), 0.0, 1.0)

        c0 = self.SOPb0
        ads_capacity = self.S_tot
        native_pool = tf.minimum(Psi_b, c0)
        native_target = alpha_native * c0
        locked_sopb = tf.minimum(native_pool, native_target)
        released_native = tf.maximum(native_pool - locked_sopb, 0.0)

        incoming_pool = tf.maximum(Psi_b - native_pool, 0.0)
        vacant_sites = tf.maximum(ads_capacity - locked_sopb, 0.0)
        eps_alpha = tf.constant(1.0e-6, tf.float32)
        alpha_reactive = tf.clip_by_value(alpha_native, 0.0, 1.0 - eps_alpha)
        adsorption_odds = alpha_reactive / tf.maximum(1.0 - alpha_reactive, eps_alpha)

        ads_incoming_noP = tf.minimum(vacant_sites, alpha_reactive * incoming_pool)
        aq_noP_bulk = tf.maximum(released_native + incoming_pool - ads_incoming_noP, 0.0)

        cPb_sat_water = tf.clip_by_value(self.Ksp / tf.maximum(tf.pow(cOH_w, self.m), 1e-30), 0.0, 1e9)
        cPb_sat_bulk = tf.clip_by_value(theta_loc * cPb_sat_water, 0.0, 1e9)
        natural_precip_gate = aq_noP_bulk > (1.0 + 1.0e-6) * tf.maximum(cPb_sat_bulk, 1e-30)
        if precip_gate_override is None:
            precip_gate = natural_precip_gate
        else:
            precip_gate = tf.cast(precip_gate_override, tf.bool)

        aq_sat_bulk = tf.minimum(cPb_sat_bulk, released_native + incoming_pool)
        ads_incoming_sat = tf.minimum(vacant_sites, tf.minimum(incoming_pool, adsorption_odds * aq_sat_bulk))

        c_aq_bulk = tf.where(precip_gate, aq_sat_bulk, aq_noP_bulk)
        SOPb_reactive = tf.where(precip_gate, ads_incoming_sat, ads_incoming_noP)
        SOPb = locked_sopb + SOPb_reactive

        Pp = tf.nn.relu(Psi_b - SOPb - c_aq_bulk)
        c_aq_bulk = Psi_b - SOPb - Pp
        cPb_aq_total = tf.clip_by_value(c_aq_bulk / tf.maximum(theta_loc, 1e-12), 0.0, 1e9)

        S_free = tf.maximum(ads_capacity - SOPb, 0.0)
        SOH = S_free / tf.maximum(B, 1e-12)
        SOH2 = aH * SOH
        SOm = bH * SOH

        out = (finite(cPb_aq_total), finite(SOPb), finite(SOH),
               finite(SOH2), finite(SOm), finite(Pp), finite(cOH_w))
        if return_gate:
            return out + (tf.cast(natural_precip_gate, tf.float32),)
        return out

    def reconstruct_pb_equil(self, t, x, Psi_b, precip_gate_override=None,
                             stop_acid=True, return_gate=False):
        theta_loc = self.theta(t, x)
        cH_w = self.cH_w(t, x)
        cOH_w = self.cOH_w(t, x)
        return self.reconstruct_pb_equil_from_fields(
            theta_loc, cH_w, cOH_w, Psi_b, t,
            precip_gate_override=precip_gate_override,
            stop_acid=stop_acid,
            return_gate=return_gate,
        )

# ---------------------------
# 6) Pb invariant NP (day units) with equilibrium reconstruction
# ---------------------------

class PbInvNet:
    def __init__(self, layers, water_weights, elec_weights, chem_kernel,
                 leftBC="flux", rightBC="open_outflow",
                 c_ic_water=0.0,
                 J_left_true_val=0.0,
                 dir_left_c=None, dir_right_c=None,
                 init_from=None, freeze_k=0, active_gate_weights=None, stop_acid_in_pb=True):
        """
        Component-PINN for Pb transport with local-equilibrium reconstruction.

        - the neural network outputs Psi_Pb and an auxiliary conservative flux;
          the flux is hard-constrained to J(0,t)=0;
        - dissolved Pb, sorbed Pb, and precipitated Pb are reconstructed from
          mass-action surface complexation, site conservation, and the
          solubility-product complementarity condition.

        """
        pb_layers = list(layers)
        pb_layers[-1] = 2
        self.mlp = DenseMLP(pb_layers, NETC["act_scale"], NETC["trainable_layer_gain"])
        if init_from is not None:
            w0, b0, a0 = init_from
            for i in range(len(self.mlp.weights)):
                self.mlp.weights[i] = tf.Variable(np.array(w0[i]), dtype=tf.float32, trainable=(i >= freeze_k))
                self.mlp.biases[i] = tf.Variable(np.array(b0[i]), dtype=tf.float32, trainable=(i >= freeze_k))
                self.mlp.A[i] = tf.Variable(a0[i], dtype=tf.float32, trainable=(i >= freeze_k))
        self.water_w, self.water_b, self.water_a = water_weights
        self.elec_w,  self.elec_b,  self.elec_a  = elec_weights
        self.chem = chem_kernel
        self.active_gate_weights = active_gate_weights
        self.stop_acid_in_pb = bool(stop_acid_in_pb)
        self.c_ic_water = float(c_ic_water)
        self.Psi_ic_store = float(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"])
        self.leftBC, self.rightBC = leftBC, rightBC

        (self.t_res, self.x_res, self.t_ic, self.x_ic,
         self.t_left, self.x_left, self.t_right, self.x_right,
         self.t_mass) = [tf.compat.v1.placeholder(tf.float32, [None, 1]) for _ in range(9)]
        self.sess = tf.compat.v1.Session(config=TF_CONFIG)

        self.Psi_res, self.residual_res, self.constitutive_res = self.net_res(self.t_res, self.x_res)
        self.J_left_aux_pred  = self.net_J(self.t_left,  self.x_left)
        self.J_right_aux_pred = self.net_J(self.t_right, self.x_right)
        self.J_left_pred  = self.net_J_phys(self.t_left,  self.x_left)
        self.J_right_pred = self.net_J_phys(self.t_right, self.x_right)
        Jl_aux, Jl_adv, Jl_diff, Jl_em, *_ = self.net_J_components(self.t_left, self.x_left)
        self.left_constitutive_res = self._finite(Jl_aux - (Jl_adv + Jl_diff + Jl_em))
        Jr_aux, Jr_adv, Jr_diff, Jr_em, *_ = self.net_J_components(self.t_right, self.x_right)
        self.right_constitutive_res = self._finite(Jr_aux - (Jr_adv + Jr_diff + Jr_em))
        self.Psi_left_pred  = self.net_Psi(tf.concat([self.t_left,  self.x_left],  1))
        self.Psi_right_pred = self.net_Psi(tf.concat([self.t_right, self.x_right], 1))
        self.Psi_ic_pred    = self.net_Psi(tf.concat([self.t_ic,    self.x_ic],    1))
        self.c_left_pred  = self.dissolved_c(self.t_left, self.x_left)
        self.c_right_pred = self.dissolved_c(self.t_right, self.x_right)

        theta_ic = self.chem.theta(self.t_ic, self.x_ic)
        self.Psi_ic_true = theta_ic * tf.fill(tf.shape(self.t_ic), tf.constant(c_ic_water, tf.float32)) \
                           + tf.fill(tf.shape(self.t_ic), tf.constant(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"], tf.float32))
        self.c_left_true = tf.fill(tf.shape(self.c_left_pred), tf.constant(0.0 if dir_left_c is None else dir_left_c, tf.float32))
        self.c_right_true = tf.fill(tf.shape(self.c_right_pred), tf.constant(0.0 if dir_right_c is None else dir_right_c, tf.float32))
        self.J_left_true = tf.fill(tf.shape(self.J_left_pred), tf.constant(J_left_true_val, tf.float32))
        self.mass_balance_res = self.net_mass_balance_res(self.t_mass)
        self.mass_integral_res = self.net_mass_integral_res()

        psi_scale = tf.constant(max(float(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"] + 1.0), 1.0), tf.float32)
        res_norm = tf.constant(float(PB_PAR.get("res_norm", PB_PAR.get("dae_res_norm", 50.0))), tf.float32)
        flux_norm = tf.constant(float(PB_PAR.get("flux_norm", PB_PAR.get("dae_flux_norm", 1.0))), tf.float32)
        constitutive_norm = tf.constant(float(PB_PAR.get("constitutive_norm", PB_PAR.get("flux_norm", 1.0))), tf.float32)
        Lx = max(float(CFG["DOMAIN"]["xmax"]) - float(CFG["DOMAIN"]["xmin"]), 1e-12)
        Tspan = max(float(CFG["DOMAIN"]["tmax"]) - float(CFG["DOMAIN"]["tmin"]), 1e-12)
        mass_norm = tf.constant(float(PB_PAR.get("mass_balance_norm", max((self.Psi_ic_store + 1.0) * Lx / Tspan, 1e-6))), tf.float32)
        mass_integral_norm = tf.constant(float(PB_PAR.get("mass_integral_norm", max(self.Psi_ic_store * Lx, 1e-6))), tf.float32)
        self.residual_res_nd = self.residual_res / res_norm
        self.constitutive_res_nd = self.constitutive_res / constitutive_norm
        w_res = float(PB_PAR.get("res_weight", PB_PAR.get("dae_res_weight", 1.0)))
        w_ic = float(PB_PAR.get("ic_weight", PB_PAR.get("dae_ic_weight", 20.0)))
        w_flux = float(PB_PAR.get("flux_weight", PB_PAR.get("dae_flux_weight", 100.0)))
        w_const = float(PB_PAR.get("constitutive_weight", 0.0))
        w_left_const = float(PB_PAR.get("left_constitutive_weight", 0.0))
        w_right_const = float(PB_PAR.get("right_constitutive_weight", 0.0))
        w_mass = float(PB_PAR.get("mass_balance_weight", 0.0))
        w_mass_int = float(PB_PAR.get("mass_integral_weight", 0.0))

        loss_terms = [
            w_res * tf.reduce_mean(tf.square(self.residual_res_nd)),
            w_ic  * tf.reduce_mean(tf.square((self.Psi_ic_pred - self.Psi_ic_true) / psi_scale)),
            w_const * tf.reduce_mean(tf.square(self.constitutive_res_nd)),
        ]

        if self.leftBC.lower() != "flux":
            raise ValueError("Pb left boundary must be flux with J=0.")
        if self.rightBC.lower() != "open_outflow":
            raise ValueError("Pb right boundary must be open_outflow.")
        loss_terms.append(w_flux * tf.reduce_mean(tf.square((self.J_left_pred - self.J_left_true) / flux_norm)))
        loss_terms.append(w_flux * tf.reduce_mean(tf.square(tf.nn.relu(-self.J_right_pred) / flux_norm)))
        loss_terms.append(w_left_const * tf.reduce_mean(tf.square(self.left_constitutive_res / constitutive_norm)))
        loss_terms.append(w_right_const * tf.reduce_mean(tf.square(self.right_constitutive_res / constitutive_norm)))
        mass_balance_loss = tf.constant(0.0, tf.float32)
        if w_mass > 0.0:
            mass_balance_loss = w_mass * tf.reduce_mean(tf.square(self.mass_balance_res / mass_norm))
        mass_integral_loss = tf.constant(0.0, tf.float32)
        if w_mass_int > 0.0:
            mass_integral_loss = w_mass_int * tf.reduce_mean(tf.square(self.mass_integral_res / mass_integral_norm))
        loss_terms.append(mass_balance_loss)
        loss_terms.append(mass_integral_loss)

        self.loss = tf.add_n(loss_terms)
        self.loss_parts = {"res": loss_terms[0], "ic": loss_terms[1], "constitutive": loss_terms[2],
                           "left_flux_zero": loss_terms[3], "right_no_inflow": loss_terms[4],
                           "left_constitutive": loss_terms[5], "right_constitutive": loss_terms[6],
                           "mass_balance": loss_terms[7], "mass_integral": loss_terms[8]}

        self.global_step = tf.Variable(0, trainable=False)
        lr = tf.compat.v1.train.exponential_decay(TRNC.get("pb_adam_lr", TRNC["adam_lr"]), self.global_step, 1000, 0.9)
        opt = tf.compat.v1.train.AdamOptimizer(lr)
        vars_all = [v for v in (self.mlp.weights + self.mlp.biases + self.mlp.A) if getattr(v, "trainable", True)]
        vars_ = [v for g, v in opt.compute_gradients(self.loss, var_list=vars_all) if g is not None]
        term_grads = [tf.gradients(term, vars_) for term in loss_terms]
        safe_grads, finite_counts, total_counts = [], [], []
        for i, v in enumerate(vars_):
            acc = tf.zeros_like(v)
            for grads in term_grads:
                g = grads[i]
                if g is None:
                    g = tf.zeros_like(v)
                g = tf.convert_to_tensor(g)
                finite_mask = tf.math.is_finite(g)
                finite_counts.append(tf.reduce_sum(tf.cast(finite_mask, tf.float32)))
                total_counts.append(tf.cast(tf.size(g), tf.float32))
                acc = acc + tf.where(finite_mask, g, tf.zeros_like(g))
            safe_grads.append(acc)
        self.grad_finite_frac = tf.add_n(finite_counts) / tf.maximum(tf.add_n(total_counts), 1.0)
        self.grad_norm = tf.linalg.global_norm(safe_grads)
        safe_grads, _ = tf.clip_by_global_norm(safe_grads, float(TRNC.get("pb_grad_clip", 0.5)))
        self.train_op = opt.apply_gradients(list(zip(safe_grads, vars_)), global_step=self.global_step)

        self.lbfgs = dde.optimizers.tensorflow_compat_v1.scipy_optimizer.ScipyOptimizerInterface(
            self.loss, method='L-BFGS-B', options=TRNC["lbfgs"])
        self.sess.run(tf.compat.v1.global_variables_initializer())
        self.loss_hist = []

    def _finite(self, z):
        return tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))

    def net_raw(self, X):
        return self.mlp.forward(X)

    def net_Psi(self, X):
        raw_scale = tf.constant(float(PB_PAR.get("psi_raw_scale", 20.0)), tf.float32)
        raw = raw_scale * tf.tanh(self.net_raw(X)[:, 0:1] / tf.maximum(raw_scale, 1e-6))
        t = X[:, 0:1]
        x = X[:, 1:2]
        theta0 = self.chem.theta(t, x)
        Psi0 = theta0 * tf.fill(tf.shape(t), tf.constant(self.c_ic_water, tf.float32)) \
               + tf.fill(tf.shape(t), tf.constant(self.Psi_ic_store, tf.float32))
        Psi0 = tf.maximum(Psi0, 1e-6)
        eta0 = tf.math.log(tf.math.expm1(Psi0) + 1e-12)
        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        tau = tf.constant(float(PB_PAR.get("psi_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(t <= t0, tf.zeros_like(t), 1.0 - tf.exp(-(t - t0) / tf.maximum(tau, 1e-6)))
        Psi = tf.nn.softplus(eta0 + gate * raw)
        return tf.clip_by_value(self._finite(Psi), 0.0, self.chem.S_tot + 1.0e4)

    def net_J_aux(self, t, x):
        X = tf.concat([t, x], 1)
        raw_scale = tf.constant(float(PB_PAR.get("flux_raw_scale", 2.0)), tf.float32)
        raw = raw_scale * tf.tanh(self.net_raw(X)[:, 1:2] / tf.maximum(raw_scale, 1e-6))
        xmin = tf.constant(float(CFG["DOMAIN"]["xmin"]), tf.float32)
        xmax = tf.constant(float(CFG["DOMAIN"]["xmax"]), tf.float32)
        xrel = (x - xmin) / tf.maximum(xmax - xmin, 1e-12)
        return self._finite(xrel * raw)

    def _water_fields(self, t, x):
        psi = water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
        theta = theta_function(psi)
        K = K_function(psi)
        psi_x = tf.gradients(psi, x)[0]
        q_hyd = -K * psi_x
        return theta, q_hyd

    def _phi_grad(self, t, x):
        H = nondim_tx(t, x)
        for l in range(len(self.elec_w)):
            H = tf.add(tf.matmul(H, self.elec_w[l]), self.elec_b[l])
            if l < len(self.elec_w) - 1:
                H = tf.tanh(NETC["act_scale"] * self.elec_a[l] * H)
        phi = H
        return phi, tf.gradients(phi, x)[0]

    def _precip_gate_override(self, t, x):
        if self.active_gate_weights is None:
            return None
        Psi_gate = pb_psi_from_weights(self.active_gate_weights, (self.water_w, self.water_b, self.water_a), t, x)
        *_, active_gate = self.chem.reconstruct_pb_equil(t, x, Psi_gate, return_gate=True)
        return active_gate > 0.5

    def species_from_Psi(self, t, x, Psi):
        cPb, SOPb, SOH, SOH2, SOm, Pp, cOH = self.chem.reconstruct_pb_equil(
            t, x, Psi, precip_gate_override=self._precip_gate_override(t, x),
            stop_acid=self.stop_acid_in_pb
        )
        si = tf.math.log((tf.maximum(cPb, 0.0) * tf.pow(tf.maximum(cOH, 1e-30), self.chem.m) + 1e-30) /
                         (self.chem.Ksp + 1e-30))
        return (self._finite(cPb), self._finite(SOPb), self._finite(SOH),
                self._finite(SOH2), self._finite(SOm), self._finite(Pp),
                self._finite(cOH), self._finite(si))

    def dissolved_c(self, t, x):
        X = tf.concat([t, x], 1)
        Psi = self.net_Psi(X)
        cPb, *_ = self.species_from_Psi(t, x, Psi)
        return cPb

    def dissolved_c_from_Psi(self, t, x, Psi):
        cPb, *_ = self.species_from_Psi(t, x, Psi)
        return cPb

    def net_J_components(self, t, x):
        X = tf.concat([t, x], 1)
        Psi = self.net_Psi(X)
        cPb, SOPb, SOH, SOH2, SOm, Pp, cOH_w, si = self.species_from_Psi(t, x, Psi)
        cx = self._finite(tf.gradients(cPb, x)[0])
        theta, q_hyd = self._water_fields(t, x)
        _, phi_x = self._phi_grad(t, x)
        q_adv = self._finite(q_hyd + (keo_of_theta(theta) * phi_x if PB_PAR.get("include_eo", True) else 0.0))
        Dstar = Dstar_of_theta(theta, Dw_Pb)
        Deff = self._finite(Dstar + DL_Pb * tf.abs(q_adv))
        z_transport = tf.constant(float(PB_PAR.get("transport_z", PB_PAR.get("z", 2.0))), tf.float32)
        u_star = self._finite(z_transport * tf.constant(float(CFG["GLOBAL"]["CONSTANTS"]["F"]) / (CFG["GLOBAL"]["CONSTANTS"]["R"] * CFG["GLOBAL"]["CONSTANTS"]["T"]), tf.float32) * Dstar)
        J_adv = self._finite(q_adv * cPb)
        J_diff = self._finite(-Deff * cx)
        J_em = self._finite(-u_star * cPb * phi_x)
        J_aux = self.net_J_aux(t, x)
        return J_aux, J_adv, J_diff, J_em, cPb, cx, q_adv, phi_x, Deff, u_star

    def net_J(self, t, x):
        return self.net_J_aux(t, x)

    def net_J_phys(self, t, x):
        _, J_adv, J_diff, J_em, *_ = self.net_J_components(t, x)
        return self._finite(J_adv + J_diff + J_em)

    def net_mass_balance_res(self, t):
        nx = max(int(PB_PAR.get("mass_balance_nx", 31)), 3)
        xmin = float(CFG["DOMAIN"]["xmin"]); xmax = float(CFG["DOMAIN"]["xmax"])
        x_line = tf.linspace(tf.constant(xmin, tf.float32), tf.constant(xmax, tf.float32), nx)[None, :]
        t_grid = tf.tile(t, [1, nx])
        x_grid = tf.tile(x_line, [tf.shape(t)[0], 1])
        tt = tf.reshape(t_grid, [-1, 1])
        xx = tf.reshape(x_grid, [-1, 1])
        Psi = self.net_Psi(tf.concat([tt, xx], 1))
        Psi_t = self._finite(tf.gradients(Psi, tt)[0])
        Psi_t_2d = tf.reshape(Psi_t, [-1, nx])
        dx = tf.constant((xmax - xmin) / float(nx - 1), tf.float32)
        weights = tf.concat([tf.ones([1], tf.float32) * 0.5, tf.ones([nx - 2], tf.float32), tf.ones([1], tf.float32) * 0.5], 0)[None, :]
        dMdt = dx * tf.reduce_sum(Psi_t_2d * weights, axis=1, keepdims=True)
        J_left = self.net_J_phys(t, tf.ones_like(t) * tf.constant(xmin, tf.float32))
        J_right = self.net_J_phys(t, tf.ones_like(t) * tf.constant(xmax, tf.float32))
        return self._finite(dMdt + J_right - J_left)

    def net_mass_integral_res(self):
        nx = max(int(PB_PAR.get("mass_integral_nx", 21)), 3)
        nt = max(int(PB_PAR.get("mass_integral_nt", 21)), 3)
        xmin = float(CFG["DOMAIN"]["xmin"]); xmax = float(CFG["DOMAIN"]["xmax"])
        tmin = float(CFG["DOMAIN"]["tmin"]); tmax = float(CFG["DOMAIN"]["tmax"])
        early = np.array([0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35], dtype=np.float32)
        uniform = np.linspace(tmin, tmax, nt, dtype=np.float32)
        t_vals = np.unique(np.clip(np.concatenate([uniform, early]), tmin, tmax)).astype(np.float32)
        t_vals.sort()
        nt_eff = int(t_vals.size)
        t_col = tf.constant(t_vals.reshape(-1, 1), tf.float32)
        x_row = tf.constant(np.linspace(xmin, xmax, nx, dtype=np.float32).reshape(1, -1), tf.float32)
        tt = tf.reshape(tf.tile(t_col, [1, nx]), [-1, 1])
        xx = tf.reshape(tf.tile(x_row, [nt_eff, 1]), [-1, 1])
        Psi = tf.reshape(self.net_Psi(tf.concat([tt, xx], 1)), [nt_eff, nx])
        dx = tf.constant((xmax - xmin) / float(nx - 1), tf.float32)
        xw = tf.concat([tf.ones([1], tf.float32) * 0.5, tf.ones([nx - 2], tf.float32), tf.ones([1], tf.float32) * 0.5], 0)[None, :]
        M = dx * tf.reduce_sum(Psi * xw, axis=1, keepdims=True)
        x_left = tf.ones_like(t_col) * tf.constant(xmin, tf.float32)
        x_right = tf.ones_like(t_col) * tf.constant(xmax, tf.float32)
        flux_out = self.net_J_phys(t_col, x_right) - self.net_J_phys(t_col, x_left)
        dt = t_col[1:] - t_col[:-1]
        step = 0.5 * (flux_out[1:] + flux_out[:-1]) * dt
        cum_flux = tf.concat([tf.zeros([1, 1], tf.float32), tf.cumsum(step, axis=0)], axis=0)
        return self._finite(M - M[0:1] + cum_flux)

    def net_res(self, t, x):
        X = tf.concat([t, x], 1)
        Psi = self.net_Psi(X)
        Psi_t = self._finite(tf.gradients(Psi, t)[0])
        J_aux, J_adv, J_diff, J_em, *_ = self.net_J_components(t, x)
        J_x = self._finite(tf.gradients(J_aux, x)[0])
        constitutive = self._finite(J_aux - (J_adv + J_diff + J_em))
        return self._finite(Psi), self._finite(Psi_t + J_x), constitutive

    def train(self, N_iter, batch=True, batch_size=512, face_feed=None):
        t_left, x_left = face_feed["left"]
        t_right, x_right = face_feed["right"]
        t_mass_all = face_feed.get("mass_t", t_right)
        mass_bs = min(int(PB_PAR.get("mass_balance_batch_size", 128)), t_mass_all.shape[0])
        names = sorted(self.loss_parts.keys())
        feed = None
        for it in range(N_iter):
            idx = np.random.choice(t_res.shape[0], batch_size, replace=False)
            tr, xr = t_res[idx, :], x_res[idx, :]
            idxm = np.random.choice(t_mass_all.shape[0], mass_bs, replace=False)
            tm = t_mass_all[idxm, :]
            feed = {self.t_res: tr, self.x_res: xr, self.t_ic: t_ic, self.x_ic: x_ic,
                    self.t_left: t_left, self.x_left: x_left, self.t_right: t_right, self.x_right: x_right,
                    self.t_mass: tm}
            self.sess.run(self.train_op, feed)
            if it % 100 == 0:
                vals = self.sess.run([self.loss, self.grad_norm, self.grad_finite_frac] + [self.loss_parts[n] for n in names], feed)
                print(f"[PbEq] It {it:5d} L={vals[0]:.3e} | grad={vals[1]:.2e} | finite_grad={vals[2]:.2f} | " + ", ".join([f"{n}={v:.2e}" for n, v in zip(names, vals[3:])]))
                self.loss_hist.append(vals[0])
        if bool(PB_PAR.get("use_lbfgs", PB_PAR.get("dae_lbfgs", False))) and feed is not None:
            self.lbfgs.minimize(self.sess, feed_dict=feed, fetches=[self.loss])
            print("[PbEq] LBFGS done.")
        else:
            print("[PbEq] LBFGS skipped.")

    def export_weights(self):
        return self.sess.run(self.mlp.weights), self.sess.run(self.mlp.biases), self.sess.run(self.mlp.A)


# ---------------------------
# 7) Sequential fixed-stress-style coupling helpers
# ---------------------------
def pb_psi_from_weights(weights, water_weights, t, x):
    """Evaluate the Pb invariant ansatz from frozen exported weights."""
    w_pb, b_pb, a_pb = weights
    raw_scale = tf.constant(float(PB_PAR.get("psi_raw_scale", 20.0)), tf.float32)
    raw = raw_scale * tf.tanh(
        dense_eval_from_weights(w_pb, b_pb, a_pb, t, x)[:, 0:1] / tf.maximum(raw_scale, 1e-6)
    )
    w_w, b_w, a_w = water_weights
    theta = theta_function(water_head_from_weights(w_w, b_w, a_w, t, x))
    psi0_store = tf.constant(float(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"]), tf.float32)
    c_ic = tf.constant(float(PB_INIT["c_ic"]), tf.float32)
    psi0 = tf.maximum(theta * c_ic + psi0_store, 1e-6)
    eta0 = tf.math.log(tf.math.expm1(psi0) + 1e-12)
    t0 = tf.constant(float(DOMAIN["tmin"]), tf.float32)
    tau = tf.constant(float(PB_PAR.get("psi_ic_gate_tau", 0.02)), tf.float32)
    gate = tf.where(t <= t0, tf.zeros_like(t), 1.0 - tf.exp(-(t - t0) / tf.maximum(tau, 1e-6)))
    return tf.clip_by_value(tf.nn.softplus(eta0 + gate * raw), 0.0, 1.0e4 + float(SURF_INIT["SOH0"] + SURF_INIT["SOPb0"] + SURF_INIT["SOH2_0"] + SURF_INIT["SOm0"]))


def relative_parameter_change(previous, current):
    """Algorithm-1 parameter metric; warm starts make this comparison meaningful."""
    num = 0.0
    den = 0.0
    for old_group, new_group in zip(previous, current):
        for old, new in zip(old_group, new_group):
            old = np.asarray(old, dtype=np.float64)
            new = np.asarray(new, dtype=np.float64)
            num += float(np.sum(np.square(new - old)))
            den += float(np.sum(np.square(new)))
    return float(np.sqrt(num / max(den, 1e-30)))


def coupling_state(water_weights, ab_weights, pb_weights):
    """Physical field state on one fixed global grid for outer convergence checks."""
    nt = max(int(SEQC.get("diagnostic_nt", 81)), 3)
    nx = max(int(SEQC.get("diagnostic_nx", 121)), 3)
    t_line = np.linspace(DOMAIN["tmin"], DOMAIN["tmax"], nt, dtype=np.float32)
    x_line = np.linspace(DOMAIN["xmin"], DOMAIN["xmax"], nx, dtype=np.float32)
    t_grid, x_grid = np.meshgrid(t_line, x_line, indexing="ij")
    # Reuse the notebook's default graph: its material constants were created
    # there.  The following operators depend only on exported frozen weights.
    t = tf.constant(t_grid.reshape(-1, 1), tf.float32)
    x = tf.constant(x_grid.reshape(-1, 1), tf.float32)
    chem_eval = ChemEquilKernel(CFG["FIELDS"]["PB"]["CHEM"], water_weights, ab_weights, SURF_INIT)
    Psi = pb_psi_from_weights(pb_weights, water_weights, t, x)
    cPb, SOPb, _, _, _, Pp, _ = chem_eval.reconstruct_pb_equil(t, x, Psi)
    cH = chem_eval.cH_w(t, x)
    pH = -tf.math.log(tf.maximum(cH, 1e-30) / 1000.0) / tf.constant(np.log(10.0), tf.float32)
    theta = chem_eval.theta(t, x)
    mass_identity = Psi - (theta * cPb + SOPb + Pp)
    with tf.compat.v1.Session() as sess:
        pH_v, Psi_v, fixed_v, mass_v = sess.run([pH, Psi, SOPb + Pp, mass_identity])
    shape = (nt, nx)
    return {
        "pH": pH_v.reshape(shape),
        "Psi": Psi_v.reshape(shape),
        "fixed": fixed_v.reshape(shape),
        "mass_identity_max": float(np.max(np.abs(mass_v))),
    }


def coupling_field_metrics(previous, current):
    def relative_l2(new, old):
        return float(np.linalg.norm(new - old) / max(np.linalg.norm(new), 1e-30))
    dpH = current["pH"] - previous["pH"]
    return {
        "pH_rms": float(np.sqrt(np.mean(np.square(dpH)))),
        "pH_max": float(np.max(np.abs(dpH))),
        "Psi_rel": relative_l2(current["Psi"], previous["Psi"]),
        "fixed_rel": relative_l2(current["fixed"], previous["fixed"]),
        "mass_identity_max": float(current["mass_identity_max"]),
    }


def train_acidbase_sequential_block(model, n_iter, face_feed):
    """One global-domain acid/base block; no time-window relay or LBFGS restart."""
    keys = ("time_slab_training", "adaptive_residual_sampling", "training_schedule")
    saved = {key: H_PAR.get(key) for key in keys}
    saved_lbfgs = TRNC.get("hplus_use_lbfgs", True)
    H_PAR.update({
        "time_slab_training": False,
        "adaptive_residual_sampling": False,
        "training_schedule": None,
    })
    TRNC["hplus_use_lbfgs"] = False
    try:
        model.train(n_iter, batch=True,
                    batch_size=TRNC.get("hplus_batch_size", TRNC["batch_size"]),
                    face_feed=face_feed)
    finally:
        H_PAR.update(saved)
        TRNC["hplus_use_lbfgs"] = saved_lbfgs


def _coupled_acid_residual(h_model, pb_model, chem_live, t, x, active_gate_weights=None):
    a, cH, cOH, cHx, cOHx, theta, q_adv, phi_x = h_model._species_fields(t, x)
    Deff_H, u_H = diffusion_pieces(theta, q_adv, DL_H, Dw_H, z_H)
    Deff_OH, u_OH = diffusion_pieces(theta, q_adv, DL_OH, Dw_OH, z_OH)
    JH = q_adv * cH - Deff_H * cHx - u_H * cH * phi_x
    JOH = q_adv * cOH - Deff_OH * cOHx - u_OH * cOH * phi_x
    Ja = JH - JOH

    X = tf.concat([t, x], 1)
    Psi_live = pb_model.net_Psi(X)
    gate = None
    if active_gate_weights is not None:
        Psi_gate = pb_psi_from_weights(active_gate_weights, (wA, bA, aA), t, x)
        *_, active_gate = chem_live.reconstruct_pb_equil(t, x, Psi_gate, return_gate=True)
        gate = active_gate > 0.5

    _, _, _, _, _, Pp, _ = chem_live.reconstruct_pb_equil_from_fields(
        theta, cH, cOH, Psi_live, t,
        precip_gate_override=gate,
        stop_acid=False,
    )
    S_pb = 2.0 * grad0(Pp, t)
    RH = tf.constant(float(H_PAR.get("H_retardation", 1.0)), tf.float32)
    theta_t = grad0(theta, t)
    cH_t = grad0(cH, t)
    cOH_t = grad0(cOH, t)
    storage = RH * (theta * cH_t + theta_t * cH) - (theta * cOH_t + theta_t * cOH)
    res = storage + grad0(Ja, x) - S_pb
    return tf.where(tf.math.is_finite(res), res, tf.zeros_like(res))


def train_full_coupled_joint(ab_previous, pb_previous, n_iter):
    Hjoint = AcidBaseNet(layers_scalar, (wA, bA, aA), (w_phi_A, b_phi_A, a_phi_A),
                         init_from=ab_previous, freeze_k=0,
                         include_eo=H_PAR["include_eo"],
                         leftBC=CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["left_BC"],
                         rightBC=CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["right_BC"],
                         JH_left_true_val=JH_left, JOH_left_true_val=JOH_left,
                         JH_right_true_val=JH_right, JOH_right_true_val=JOH_right,
                         a_ic_val=A_ic,
                         chem_kernel=None, pb_weights=None,
                         include_pb_source=False)
    chem_live = ChemEquilKernel(
        PB_CHEM,
        (wA, bA, aA),
        (Hjoint.mlp.weights, Hjoint.mlp.biases, Hjoint.mlp.A),
        SURF_INIT,
    )
    Pjoint = PbInvNet(layers_scalar, (wA, bA, aA), (w_phi_A, b_phi_A, a_phi_A), chem_live,
                      leftBC=PB_leftBC, rightBC=PB_rightBC,
                      c_ic_water=float(PB_INIT["c_ic"]),
                      J_left_true_val=float(PB_BC.get("left", {}).get("J", 0.0)),
                      dir_left_c=_dir_c("left"), dir_right_c=_dir_c("right"),
                      init_from=pb_previous, freeze_k=0,
                      active_gate_weights=pb_previous,
                      stop_acid_in_pb=False)
    Hjoint.sess = Pjoint.sess

    finite = lambda z: tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))
    joint_res = _coupled_acid_residual(Hjoint, Pjoint, chem_live, Hjoint.t_res, Hjoint.x_res,
                                       active_gate_weights=pb_previous)
    joint_res_nd = finite(joint_res / tf.maximum(Hjoint.res_a_ref, 1e-12))
    acid_parts = {k: v for k, v in Hjoint.loss_parts.items() if k != "res_a"}
    acid_parts["res_a_coupled"] = Hjoint.residual_weight_ph * tf.reduce_mean(tf.square(joint_res_nd))
    acid_loss = tf.add_n(list(acid_parts.values()))
    pb_loss = Pjoint.loss
    joint_loss = float(FULLC.get("acid_loss_weight", 1.0)) * acid_loss + float(FULLC.get("pb_loss_weight", 1.0)) * pb_loss

    joint_vars = [v for v in (Hjoint.mlp.weights + Hjoint.mlp.biases + Hjoint.mlp.A +
                              Pjoint.mlp.weights + Pjoint.mlp.biases + Pjoint.mlp.A)
                  if getattr(v, "trainable", True)]
    scope = "full_joint_adam_%d" % len(tf.compat.v1.global_variables())
    with tf.compat.v1.variable_scope(scope):
        opt = tf.compat.v1.train.AdamOptimizer(float(FULLC.get("adam_lr", 2.0e-4)))
        grads_vars = [(g, v) for g, v in opt.compute_gradients(joint_loss, var_list=joint_vars) if g is not None]
        grads, vars_ = zip(*grads_vars)
        safe_grads = [tf.where(tf.math.is_finite(g), g, tf.zeros_like(g)) for g in grads]
        grad_norm = tf.linalg.global_norm(safe_grads)
        safe_grads, _ = tf.clip_by_global_norm(safe_grads, float(FULLC.get("grad_clip", 1.0)))
        train_op = opt.apply_gradients(list(zip(safe_grads, vars_)))
    all_vars = tf.compat.v1.global_variables()
    init_flags = Pjoint.sess.run([tf.compat.v1.is_variable_initialized(v) for v in all_vars])
    uninit_vars = [v for v, ok in zip(all_vars, init_flags) if not ok]
    if uninit_vars:
        Pjoint.sess.run(tf.compat.v1.variables_initializer(uninit_vars))

    n_iter = int(n_iter)
    batch_size = int(FULLC.get("batch_size", TRNC.get("batch_size", 512)))
    print_every = max(int(FULLC.get("print_every", 100)), 1)
    mass_t_all = PB_t_right
    mass_bs = min(int(PB_PAR.get("mass_balance_batch_size", 128)), mass_t_all.shape[0])
    acid_names = sorted(acid_parts.keys())
    pb_names = sorted(Pjoint.loss_parts.keys())
    hist = []
    best_loss = np.inf
    best_it = -1
    best_ab_weights = None
    best_pb_weights = None

    def _ab_snapshot():
        return Pjoint.sess.run([Hjoint.mlp.weights, Hjoint.mlp.biases, Hjoint.mlp.A])

    def _restore_weights(ab_weights, pb_weights):
        assign_ops = []
        for vars_group, vals_group in zip((Hjoint.mlp.weights, Hjoint.mlp.biases, Hjoint.mlp.A), ab_weights):
            assign_ops.extend([tf.compat.v1.assign(v, val) for v, val in zip(vars_group, vals_group)])
        for vars_group, vals_group in zip((Pjoint.mlp.weights, Pjoint.mlp.biases, Pjoint.mlp.A), pb_weights):
            assign_ops.extend([tf.compat.v1.assign(v, val) for v, val in zip(vars_group, vals_group)])
        Pjoint.sess.run(assign_ops)

    for it in range(n_iter):
        bs = min(batch_size, t_res.shape[0])
        idx = np.random.choice(t_res.shape[0], bs, replace=(t_res.shape[0] < bs))
        tr, xr = t_res[idx, :], x_res[idx, :]
        idxm = np.random.choice(mass_t_all.shape[0], mass_bs, replace=(mass_t_all.shape[0] < mass_bs))
        feed = {
            Hjoint.t_res: tr, Hjoint.x_res: xr,
            Hjoint.t_ic: t_ic, Hjoint.x_ic: x_ic,
            Hjoint.t_left: H_t_left, Hjoint.x_left: H_x_left,
            Hjoint.t_right: H_t_right, Hjoint.x_right: H_x_right,
            Pjoint.t_res: tr, Pjoint.x_res: xr,
            Pjoint.t_ic: t_ic, Pjoint.x_ic: x_ic,
            Pjoint.t_left: PB_t_left, Pjoint.x_left: PB_x_left,
            Pjoint.t_right: PB_t_right, Pjoint.x_right: PB_x_right,
            Pjoint.t_mass: mass_t_all[idxm, :],
        }
        wfeed, _, _, _ = Hjoint._weight_feed(it, n_iter)
        feed.update(wfeed)
        if it == 0:
            initial_loss = float(Pjoint.sess.run(joint_loss, feed))
            if np.isfinite(initial_loss):
                best_loss = initial_loss
                best_it = -1
                best_ab_weights = _ab_snapshot()
                best_pb_weights = Pjoint.export_weights()
        Pjoint.sess.run(train_op, feed)
        if it % print_every == 0:
            vals = Pjoint.sess.run(
                [joint_loss, grad_norm, acid_loss, pb_loss] +
                [acid_parts[n] for n in acid_names] + [Pjoint.loss_parts[n] for n in pb_names],
                feed,
            )
            loss_now = float(vals[0])
            hist.append(loss_now)
            if np.isfinite(loss_now) and loss_now < best_loss:
                best_loss = loss_now
                best_it = it
                best_ab_weights = _ab_snapshot()
                best_pb_weights = Pjoint.export_weights()
            acid_vals = vals[4:4 + len(acid_names)]
            pb_vals = vals[4 + len(acid_names):]
            print(f"[FullJoint] It {it:5d} L={vals[0]:.3e} | grad={vals[1]:.2e} | acid={vals[2]:.3e} pb={vals[3]:.3e} | " +
                  ", ".join([f"A:{n}={v:.2e}" for n, v in zip(acid_names, acid_vals)]) + " | " +
                  ", ".join([f"Pb:{n}={v:.2e}" for n, v in zip(pb_names, pb_vals)]))

    if best_ab_weights is not None and best_pb_weights is not None:
        _restore_weights(best_ab_weights, best_pb_weights)
        print(f"[FullJoint] restored best checkpoint: It {best_it}, L={best_loss:.3e}")
    ab_weights = _ab_snapshot()
    pb_weights = Pjoint.export_weights()
    Hjoint.loss_hist = hist
    Hjoint.best_loss = best_loss
    Hjoint.best_iteration = best_it
    Pjoint.loss_hist = hist
    Pjoint.best_loss = best_loss
    Pjoint.best_iteration = best_it
    return Hjoint, Pjoint, ab_weights, pb_weights, hist


# ---------------------------


## Three-parameter inverse model

The inverse graph and staged optimizer are defined below.


In [ ]:
import argparse
import json
import math
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

HERE = REPOSITORY_ROOT / "model_and_data"

CASES = {
    "theta020": {"psi_ic_m": -166.9363086307076, "fixed_saturated": False},
    "theta047": {"psi_ic_m": -16.33394573035923, "fixed_saturated": False},
}
CASE_NAMES = tuple(CASES)


def save_parameter_trajectory(history, outdir, final_values):
    """Save a phase-aware parameter-versus-training-step plot for every run."""
    if not history:
        return
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    frame = pd.DataFrame(history).copy()
    stage_order = [
        stage for stage in ("electric", "water", "acid", "pb", "joint")
        if stage in set(frame["stage"])
    ]
    offsets = {}
    offset = 0
    for stage in stage_order:
        offsets[stage] = offset
        offset += int(frame.loc[frame["stage"] == stage, "iteration"].max())
    frame["cumulative_iteration"] = [
        offsets[stage] + iteration
        for stage, iteration in zip(frame["stage"], frame["iteration"])
    ]
    boundaries = [offsets[stage] for stage in stage_order] + [offset]
    colors = {"Keos": "#2864b4", "RH": "#d47c16", "pH50": "#24855b"}
    final_map = dict(zip(("Keos", "RH", "pH50"), map(float, final_values)))

    fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.7), constrained_layout=True)
    for axis, parameter in zip(axes, ("Keos", "RH", "pH50")):
        axis.plot(
            frame["cumulative_iteration"], frame[parameter],
            color=colors[parameter], marker="o", markersize=2.8, linewidth=1.6,
        )
        axis.axhline(final_map[parameter], color="#333333", linestyle=":", linewidth=1.2,
                     label="selected final")
        for boundary in boundaries[1:-1]:
            axis.axvline(boundary, color="#aaaaaa", linewidth=0.8)
        axis.set_xlabel("cumulative training step")
        axis.set_ylabel(parameter)
        axis.set_title(parameter)
        axis.grid(alpha=0.22)
    for left, right, stage in zip(boundaries[:-1], boundaries[1:], stage_order):
        axes[0].text(
            (left + right) / 2, 1.02, stage,
            transform=axes[0].get_xaxis_transform(), ha="center", va="bottom",
            fontsize=7, rotation=22,
        )
    axes[0].legend(frameon=False, fontsize=7)
    fig.suptitle("Inverse parameters versus cumulative training step", fontsize=11)
    fig.savefig(outdir / "parameter_trajectory.png", dpi=220)
    fig.savefig(outdir / "parameter_trajectory.svg")
    plt.close(fig)


def finite(z):
    """Fail fast on invalid physics instead of turning NaN/Inf into zero residual."""
    return tf.debugging.check_numerics(z, "non-finite value in inverse model")


def mse(z):
    return tf.reduce_mean(tf.square(finite(z)))


def bounded_parameter(name, initial, low, high):
    if not low < initial < high:
        raise ValueError(f"{name}: require {low} < {initial} < {high}")
    q = (initial - low) / (high - low)
    raw0 = math.log(q / (1.0 - q))
    raw = tf.Variable(raw0, dtype=tf.float32, trainable=True, name=f"raw_{name}")
    value = tf.constant(low, tf.float32) + tf.constant(high - low, tf.float32) * tf.sigmoid(raw)
    return raw, tf.identity(value, name=name)


def inversion_parameter(name, initial, low, high, fixed=None):
    """Return one physical parameter and its optional trainable raw variable."""
    if fixed is None:
        raw, value = bounded_parameter(name, initial, low, high)
        return value, [raw]
    if not low <= fixed <= high:
        raise ValueError(f"{name}: fixed value must be within [{low}, {high}]")
    return tf.constant(float(fixed), tf.float32, name=name), []


def make_optimizer(loss, var_list, learning_rate, clip_norm, name, clip_each=False,
                   decay_steps=0, decay_rate=1.0):
    with tf.compat.v1.variable_scope(name):
        if not var_list:
            return tf.no_op(), tf.constant(0.0, tf.float32), []
        optimizer_step = tf.compat.v1.get_variable(
            "optimizer_step", shape=(), dtype=tf.int64,
            initializer=tf.zeros_initializer(), trainable=False,
        )
        if int(decay_steps) > 0 and float(decay_rate) < 1.0:
            effective_learning_rate = tf.compat.v1.train.exponential_decay(
                float(learning_rate), optimizer_step, int(decay_steps),
                float(decay_rate), staircase=True,
            )
        else:
            effective_learning_rate = tf.constant(float(learning_rate), tf.float32)
        optimizer = tf.compat.v1.train.AdamOptimizer(effective_learning_rate)
        pairs = [(g, v) for g, v in optimizer.compute_gradients(loss, var_list=var_list) if g is not None]
        if not pairs:
            return tf.no_op(), tf.constant(0.0, tf.float32), []
        grads, variables = zip(*pairs)
        checked = [tf.debugging.check_numerics(g, f"non-finite gradient in {name}") for g in grads]
        norm = tf.linalg.global_norm(checked)
        # During the fully coupled step, one large state-network gradient must
        # not suppress every shared physical-parameter gradient.  Clip tensors
        # separately there; retain global clipping for the warm-up stages.
        if clip_each:
            safe = [tf.clip_by_norm(g, float(clip_norm)) for g in checked]
        else:
            safe, _ = tf.clip_by_global_norm(checked, float(clip_norm))
        return optimizer.apply_gradients(
            list(zip(safe, variables)), global_step=optimizer_step
        ), norm, list(variables)


class CaseGraph:
    def __init__(self, name, meta, layers, params, data, args, evaluation_data=None):
        self.name = name
        self.meta = meta
        self.args = args
        self.Keos, self.RH, self.pH50 = params
        self.psi_ic = float(meta["psi_ic_m"])
        self.fixed_saturated = bool(meta["fixed_saturated"])
        self.xmin, self.xmax = 0.0, 0.2
        self.tmin, self.tmax = 0.0, 5.0
        self.L = self.xmax - self.xmin

        with tf.compat.v1.variable_scope(name):
            self.t_res = tf.compat.v1.placeholder(tf.float32, [None, 1], name="t_res")
            self.x_res = tf.compat.v1.placeholder(tf.float32, [None, 1], name="x_res")
            self.t_ic = tf.compat.v1.placeholder(tf.float32, [None, 1], name="t_ic")
            self.x_ic = tf.compat.v1.placeholder(tf.float32, [None, 1], name="x_ic")
            self.t_left = tf.compat.v1.placeholder(tf.float32, [None, 1], name="t_left")
            self.x_left = tf.compat.v1.placeholder(tf.float32, [None, 1], name="x_left")
            self.t_right = tf.compat.v1.placeholder(tf.float32, [None, 1], name="t_right")
            self.x_right = tf.compat.v1.placeholder(tf.float32, [None, 1], name="x_right")

            with tf.compat.v1.variable_scope("electric"):
                self.electric_mlp = DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
            if not self.fixed_saturated:
                with tf.compat.v1.variable_scope("water"):
                    self.water_mlp = DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
            else:
                self.water_mlp = None
            with tf.compat.v1.variable_scope("acid"):
                self.acid_mlp = DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
            pb_layers = list(layers)
            pb_layers[-1] = 2
            with tf.compat.v1.variable_scope("pb"):
                self.pb_mlp = DenseMLP(pb_layers, NETC["act_scale"], NETC["trainable_layer_gain"])

        self.phi_vars = tf.compat.v1.get_collection(tf.compat.v1.GraphKeys.TRAINABLE_VARIABLES, scope=f"{name}/electric")
        self.water_vars = tf.compat.v1.get_collection(tf.compat.v1.GraphKeys.TRAINABLE_VARIABLES, scope=f"{name}/water")
        self.acid_vars = tf.compat.v1.get_collection(tf.compat.v1.GraphKeys.TRAINABLE_VARIABLES, scope=f"{name}/acid")
        self.pb_vars = tf.compat.v1.get_collection(tf.compat.v1.GraphKeys.TRAINABLE_VARIABLES, scope=f"{name}/pb")

        self._build_data_tensors(data, evaluation_data)
        self._build_losses()
        self._build_profile_tensors()

    def phi(self, t, x):
        return self.electric_mlp.forward(tf.concat([t, x], 1))

    def head(self, t, x):
        if self.fixed_saturated:
            return tf.zeros_like(t)
        X = tf.concat([t, x], 1)
        return water_head_from_raw(self.water_mlp.forward(X), t, x)

    def theta(self, t, x):
        return theta_function(self.head(t, x))

    def water_flux(self, t, x):
        h = self.head(t, x)
        theta = theta_function(h)
        q_hyd = -K_function(h) * grad0(h, x)
        phi_x = grad0(self.phi(t, x), x)
        q_eo = self.Keos * keo_of_theta(theta) * phi_x
        return finite(q_hyd), finite(q_eo), finite(q_hyd + q_eo)

    def acid_a(self, t, x):
        raw = tf.clip_by_value(self.acid_mlp.forward(tf.concat([t, x], 1)), -8.0, 8.0)
        gate_tau = tf.constant(float(H_PAR.get("a_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(t <= self.tmin, tf.zeros_like(t), 1.0 - tf.exp(-(t - self.tmin) / tf.maximum(gate_tau, 1e-12)))
        a_ic = tf.constant(float(self.a_ic), tf.float32)
        return finite(a_ic + gate * tf.constant(float(H_PAR.get("a_raw_scale", 20.0)), tf.float32) * raw)

    @staticmethod
    def acid_species_from_a(a):
        s = tf.sqrt(tf.square(a) + 4.0 * Kw_const)
        cH = tf.where(a >= 0.0, 0.5 * (a + s), (2.0 * Kw_const) / tf.maximum(s - a, 1e-30))
        cOH = tf.where(a <= 0.0, 0.5 * (-a + s), (2.0 * Kw_const) / tf.maximum(s + a, 1e-30))
        return finite(cH), finite(cOH)

    def acid_species(self, t, x):
        return self.acid_species_from_a(self.acid_a(t, x))

    def pH(self, t, x):
        cH, _ = self.acid_species(t, x)
        value = -tf.math.log(tf.maximum(cH, 1e-30) / 1000.0) / tf.constant(np.log(10.0), tf.float32)
        return tf.clip_by_value(finite(value), float(H_PAR.get("pH_min", 0.0)), float(H_PAR.get("pH_max", 14.0)))

    def acid_fluxes(self, t, x):
        cH, cOH = self.acid_species(t, x)
        theta = self.theta(t, x)
        _, _, q = self.water_flux(t, x)
        phi_x = grad0(self.phi(t, x), x)
        Deff_H, u_H = diffusion_pieces(theta, q, DL_H, Dw_H, z_H)
        Deff_OH, u_OH = diffusion_pieces(theta, q, DL_OH, Dw_OH, z_OH)
        cHx, cOHx = grad0(cH, x), grad0(cOH, x)
        JH = q * cH - Deff_H * cHx - u_H * cH * phi_x
        JOH = q * cOH - Deff_OH * cOHx - u_OH * cOH * phi_x
        JH_nonadv = -Deff_H * cHx - u_H * cH * phi_x
        JOH_nonadv = -Deff_OH * cOHx - u_OH * cOH * phi_x
        return finite(JH), finite(JOH), finite(JH_nonadv), finite(JOH_nonadv)

    def pb_total(self, t, x):
        raw = self.pb_mlp.forward(tf.concat([t, x], 1))[:, 0:1]
        scale = tf.constant(float(PB_PAR.get("psi_raw_scale", 20.0)), tf.float32)
        raw = scale * tf.tanh(raw / tf.maximum(scale, 1e-6))
        initial = tf.ones_like(t) * tf.constant(float(SURF_INIT["SOPb0"] + SURF_INIT.get("Pp0", 0.0)), tf.float32)
        eta0 = tf.math.log(tf.math.expm1(tf.maximum(initial, 1e-6)) + 1e-12)
        tau = tf.constant(float(PB_PAR.get("psi_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(t <= self.tmin, tf.zeros_like(t), 1.0 - tf.exp(-(t - self.tmin) / tf.maximum(tau, 1e-6)))
        total = tf.nn.softplus(eta0 + gate * raw)
        site_total = float(SURF_INIT["SOH0"] + SURF_INIT["SOPb0"] + SURF_INIT["SOH2_0"] + SURF_INIT["SOm0"])
        return tf.clip_by_value(finite(total), 0.0, site_total + 1.0e4)

    def pb_aux_flux(self, t, x):
        raw = self.pb_mlp.forward(tf.concat([t, x], 1))[:, 1:2]
        scale = tf.constant(float(PB_PAR.get("flux_raw_scale", 2.0)), tf.float32)
        raw = scale * tf.tanh(raw / tf.maximum(scale, 1e-6))
        return finite((x - self.xmin) / self.L * raw)

    def pb_species(self, t, x, total=None):
        total = self.pb_total(t, x) if total is None else total
        theta = tf.clip_by_value(finite(self.theta(t, x)), 1e-6, 1.0)
        cH, cOH = self.acid_species(t, x)
        cH = tf.clip_by_value(finite(cH), 1e-12, 1e3)
        cOH = tf.clip_by_value(finite(cOH), 1e-10, 1e4)
        pH = self.pH(t, x)

        n_ads = tf.constant(float(self.args.adsorption_slope_n), tf.float32)
        exponent = tf.clip_by_value(tf.constant(np.log(10.0), tf.float32) * n_ads * (self.pH50 - pH), -60.0, 60.0)
        f_ads = tf.clip_by_value(1.0 / (1.0 + tf.exp(exponent)), 0.0, 1.0)

        c0 = tf.constant(float(SURF_INIT["SOPb0"] + SURF_INIT.get("Pp0", 0.0)), tf.float32)
        site_total = tf.constant(float(SURF_INIT["SOH0"] + SURF_INIT["SOPb0"] + SURF_INIT["SOH2_0"] + SURF_INIT["SOm0"]), tf.float32)
        native_pool = tf.minimum(total, c0)
        # Absolute target based on initial inventory: this is the non-zero-Pb history rule.
        locked = tf.minimum(native_pool, f_ads * c0)
        released = tf.maximum(native_pool - locked, 0.0)
        incoming = tf.maximum(total - native_pool, 0.0)
        vacant = tf.maximum(site_total - locked, 0.0)
        f_reactive = tf.clip_by_value(f_ads, 0.0, 1.0 - 1e-6)
        odds = f_reactive / tf.maximum(1.0 - f_reactive, 1e-6)

        ads_no_precip = tf.minimum(vacant, f_reactive * incoming)
        aq_no_precip = tf.maximum(released + incoming - ads_no_precip, 0.0)
        ksp = tf.constant(float(CFG["FIELDS"]["PB"]["CHEM"].get("Ksp_PbOH2", 1.43e-11)), tf.float32)
        m_precip = tf.constant(float(CFG["FIELDS"]["PB"]["CHEM"].get("m", 2.0)), tf.float32)
        aq_sat = tf.minimum(theta * ksp / tf.maximum(tf.pow(cOH, m_precip), 1e-30), released + incoming)
        precip_gate = aq_no_precip > (1.0 + 1e-6) * tf.maximum(aq_sat, 1e-30)
        ads_precip = tf.minimum(vacant, tf.minimum(incoming, odds * aq_sat))
        aq_bulk = tf.where(precip_gate, aq_sat, aq_no_precip)
        ads = locked + tf.where(precip_gate, ads_precip, ads_no_precip)
        precip = tf.nn.relu(total - ads - aq_bulk)
        aq_bulk = total - ads - precip
        cPb = tf.clip_by_value(aq_bulk / tf.maximum(theta, 1e-12), 0.0, 1e9)

        k_pr = tf.constant(float(CFG["FIELDS"]["PB"]["CHEM"]["k_pr_f"] / CFG["FIELDS"]["PB"]["CHEM"]["k_pr_b"]), tf.float32)
        k_dpr = tf.constant(float(CFG["FIELDS"]["PB"]["CHEM"]["k_dpr_f"] / CFG["FIELDS"]["PB"]["CHEM"]["k_dpr_b"]), tf.float32)
        cH_bulk = tf.maximum(theta * cH, 1e-12)
        aH, bH = k_pr * cH_bulk, k_dpr / cH_bulk
        free_sites = tf.maximum(site_total - ads, 0.0)
        SOH = free_sites / tf.maximum(1.0 + aH + bH, 1e-12)
        SOH2, SOm = aH * SOH, bH * SOH
        return tuple(finite(v) for v in (cPb, ads, SOH, SOH2, SOm, precip, cOH))

    def pb_fluxes(self, t, x):
        total = self.pb_total(t, x)
        cPb, *_ = self.pb_species(t, x, total)
        cPb_x = grad0(cPb, x)
        theta = self.theta(t, x)
        _, _, q = self.water_flux(t, x)
        phi_x = grad0(self.phi(t, x), x)
        Dstar = Dstar_of_theta(theta, Dw_Pb)
        Deff = Dstar + DL_Pb * tf.abs(q)
        z = tf.constant(float(PB_PAR.get("transport_z", PB_PAR.get("z", 2.0))), tf.float32)
        u = z * F_c / (R_c * T_c) * Dstar
        J_adv, J_diff, J_em = q * cPb, -Deff * cPb_x, -u * cPb * phi_x
        J_aux = self.pb_aux_flux(t, x)
        return tuple(finite(v) for v in (J_aux, J_adv, J_diff, J_em))

    def pb_physical_flux(self, t, x):
        _, adv, diff, em = self.pb_fluxes(t, x)
        return finite(adv + diff + em)

    def acid_residual_terms(self, t, x, coupled):
        cH, cOH = self.acid_species(t, x)
        JH, JOH, _, _ = self.acid_fluxes(t, x)
        theta = self.theta(t, x)
        storage = self.RH * (theta * grad0(cH, t) + grad0(theta, t) * cH) \
                  - (theta * grad0(cOH, t) + grad0(theta, t) * cOH)
        source = tf.zeros_like(t)
        if coupled:
            _, ads, _, SOH2, SOm, precip, _ = self.pb_species(t, x)
            source = grad0(-SOH2 + SOm + ads + 2.0 * precip, t)
            # Picard-style coupling: use the current Pb source value in the
            # acid residual, but do not backpropagate through the nonsmooth
            # min/where species partition.  The source is recomputed on every
            # update, so the forward coupling is retained without undefined
            # gradients at phase-switch surfaces.
            source = tf.stop_gradient(source)
        transport_source = grad0(JH - JOH, x) - source
        residual = storage + transport_source
        return finite(storage), finite(transport_source), finite(residual)

    def acid_residual(self, t, x, coupled):
        return self.acid_residual_terms(t, x, coupled)[2]

    def _build_data_tensors(self, data, evaluation_data=None):
        if data is None:
            self.ph_cal_df = self.pb_cal_df = self.ph_val_df = self.pb_val_df = pd.DataFrame()
            self.ph_cal = self.pb_cal = self.ph_val = self.pb_val = None
            self.ph_cal_pred = self.pb_cal_pred = self.ph_val_pred = self.pb_val_pred = None
            return

        def tensors(df, value_col, sigma_col, prefix):
            t = tf.constant(df["time_day"].to_numpy(np.float32).reshape(-1, 1), name=f"{prefix}_t")
            x = tf.constant((df["distance_cm"].to_numpy(np.float32) / 100.0).reshape(-1, 1), name=f"{prefix}_x")
            y = tf.constant(df[value_col].to_numpy(np.float32).reshape(-1, 1), name=f"{prefix}_y")
            sigma = tf.constant(df[sigma_col].to_numpy(np.float32).reshape(-1, 1), name=f"{prefix}_sigma")
            return t, x, y, sigma

        self.ph_cal_df = data["ph_cal"].reset_index(drop=True)
        self.pb_cal_df = data["pb_cal"].reset_index(drop=True)
        self.ph_val_df = data["ph_val"].reset_index(drop=True)
        self.pb_val_df = data["pb_val"].reset_index(drop=True)
        self.ph_cal = tensors(self.ph_cal_df, "pH_obs", "pH_measurement_sd", "ph_cal")
        self.pb_cal = tensors(self.pb_cal_df, "TotalPb_obs_mol_m3_bulk", "TotalPb_measurement_sd", "pb_cal")
        self.ph_val = tensors(self.ph_val_df, "pH_obs", "pH_measurement_sd", "ph_val")
        self.pb_val = tensors(self.pb_val_df, "TotalPb_obs_mol_m3_bulk", "TotalPb_measurement_sd", "pb_val")
        self.ph_cal_pred = self.pH(self.ph_cal[0], self.ph_cal[1])
        self.pb_cal_pred = self.pb_total(self.pb_cal[0], self.pb_cal[1])
        self.ph_val_pred = self.pH(self.ph_val[0], self.ph_val[1])
        self.pb_val_pred = self.pb_total(self.pb_val[0], self.pb_val[1])

        evaluation_data = evaluation_data or data
        self.ph_cal_full_df = evaluation_data["ph_cal"].reset_index(drop=True)
        self.pb_cal_full_df = evaluation_data["pb_cal"].reset_index(drop=True)
        self.ph_val_full_df = evaluation_data["ph_val"].reset_index(drop=True)
        self.pb_val_full_df = evaluation_data["pb_val"].reset_index(drop=True)
        self.ph_cal_full = tensors(self.ph_cal_full_df, "pH_obs", "pH_measurement_sd", "full_ph_cal")
        self.pb_cal_full = tensors(
            self.pb_cal_full_df, "TotalPb_obs_mol_m3_bulk",
            "TotalPb_measurement_sd", "full_pb_cal"
        )
        self.ph_val_full = tensors(self.ph_val_full_df, "pH_obs", "pH_measurement_sd", "full_ph_val")
        self.pb_val_full = tensors(
            self.pb_val_full_df, "TotalPb_obs_mol_m3_bulk",
            "TotalPb_measurement_sd", "full_pb_val"
        )
        self.ph_cal_full_pred = self.pH(self.ph_cal_full[0], self.ph_cal_full[1])
        self.pb_cal_full_pred = self.pb_total(self.pb_cal_full[0], self.pb_cal_full[1])
        self.ph_val_full_pred = self.pH(self.ph_val_full[0], self.ph_val_full[1])
        self.pb_val_full_pred = self.pb_total(self.pb_val_full[0], self.pb_val_full[1])

    def _mass_integral_loss(self):
        nx, nt = self.args.mass_nx, self.args.mass_nt
        t_values = np.linspace(self.tmin, self.tmax, nt, dtype=np.float32)
        x_values = np.linspace(self.xmin, self.xmax, nx, dtype=np.float32)
        tt, xx = np.meshgrid(t_values, x_values, indexing="ij")
        t = tf.constant(tt.reshape(-1, 1), tf.float32)
        x = tf.constant(xx.reshape(-1, 1), tf.float32)
        total = tf.reshape(self.pb_total(t, x), [nt, nx])
        dx = tf.constant(self.L / (nx - 1), tf.float32)
        xw = tf.constant(np.r_[0.5, np.ones(nx - 2), 0.5].reshape(1, -1), tf.float32)
        mass = dx * tf.reduce_sum(total * xw, axis=1, keepdims=True)
        tc = tf.constant(t_values.reshape(-1, 1), tf.float32)
        left = self.pb_physical_flux(tc, tf.ones_like(tc) * self.xmin)
        right = self.pb_physical_flux(tc, tf.ones_like(tc) * self.xmax)
        dt = tc[1:] - tc[:-1]
        cumulative = tf.concat([tf.zeros([1, 1]), tf.cumsum(0.5 * (right[1:] - left[1:] + right[:-1] - left[:-1]) * dt, axis=0)], 0)
        return mse((mass - mass[0:1] + cumulative) / tf.constant(5.0 * self.L, tf.float32))

    def _build_profile_tensors(self):
        self.profile_x_cm = np.linspace(0.0, 20.0, 201, dtype=np.float32)
        t = tf.ones([self.profile_x_cm.size, 1], tf.float32) * 5.0
        x = tf.constant((self.profile_x_cm / 100.0).reshape(-1, 1), tf.float32)
        total = self.pb_total(t, x)
        cPb, ads, _, _, _, precip, _ = self.pb_species(t, x, total)
        theta = self.theta(t, x)
        aq_bulk = theta * cPb
        self.profile_tensors = (self.pH(t, x), total, cPb, aq_bulk, ads, precip,
                                total - aq_bulk - ads - precip)

    def _build_losses(self):
        phi_res = grad0(sigma_eff_of_theta(self.theta(self.t_res, self.x_res)) * grad0(self.phi(self.t_res, self.x_res), self.x_res), self.x_res)
        phi_res_pre = grad0(grad0(self.phi(self.t_res, self.x_res), self.x_res), self.x_res)
        phi_bc = mse((self.phi(self.t_left, self.x_left) - float(ELEC_BD["phi_anode"])) / 5.0) \
                 + mse((self.phi(self.t_right, self.x_right) - float(ELEC_BD["phi_cathode"])) / 5.0)
        self.loss_electric_pre = mse(phi_res_pre) + 100.0 * phi_bc
        self.loss_electric = mse(phi_res / tf.constant(10.0, tf.float32)) + 100.0 * phi_bc

        theta = self.theta(self.t_res, self.x_res)
        _, _, q = self.water_flux(self.t_res, self.x_res)
        water_res = grad0(theta, self.t_res) + grad0(q, self.x_res)
        theta_ic = theta_function(tf.ones_like(self.t_ic) * self.psi_ic)
        water_ic = mse((self.theta(self.t_ic, self.x_ic) - theta_ic) / 0.5)
        tau = float(CFG["FIELDS"]["WATER"]["PARAM"].get("head_transition_tau", 0.02))
        gate_l = 1.0 - tf.exp(-self.t_left / max(tau, 1e-6))
        gate_r = 1.0 - tf.exp(-self.t_right / max(tau, 1e-6))
        h_left_target = self.psi_ic + gate_l * (0.0 - self.psi_ic)
        h_right_target = self.psi_ic + gate_r * (0.0 - self.psi_ic)
        head_scale = max(abs(self.psi_ic), 1.0)
        water_bc = mse((self.head(self.t_left, self.x_left) - h_left_target) / head_scale) \
                   + mse((self.head(self.t_right, self.x_right) - h_right_target) / head_scale)
        self.loss_water = mse(water_res / 0.5) + 500.0 * water_ic + 50.0 * water_bc

        ja_ref = max(float(H_BC.get("current_efficiency", 0.19)) * float(ELEC_PAR.get("sigma_sat", 0.5)) * 5.0 / self.L / float(CFG["GLOBAL"]["CONSTANTS"]["F"]) * SEC_PER_DAY, 1e-8)
        res_scale = tf.constant(ja_ref / self.L, tf.float32)
        ramp_l = 1.0 - tf.exp(-self.t_left / max(float(H_PAR.get("faraday_ramp_tau", 0.01)), 1e-12))
        ramp_r = 1.0 - tf.exp(-self.t_right / max(float(H_PAR.get("faraday_ramp_tau", 0.01)), 1e-12))
        sigma_l = sigma_eff_of_theta(self.theta(self.t_left, self.x_left))
        sigma_r = sigma_eff_of_theta(self.theta(self.t_right, self.x_right))
        j_l = float(H_BC.get("current_efficiency", 0.19)) * tf.abs(-sigma_l * grad0(self.phi(self.t_left, self.x_left), self.x_left)) / F_c * SEC_PER_DAY
        j_r = float(H_BC.get("current_efficiency", 0.19)) * tf.abs(-sigma_r * grad0(self.phi(self.t_right, self.x_right), self.x_right)) / F_c * SEC_PER_DAY
        _, _, jh_l, joh_l = self.acid_fluxes(self.t_left, self.x_left)
        _, _, jh_r, joh_r = self.acid_fluxes(self.t_right, self.x_right)
        flux_loss = mse((jh_l - j_l * ramp_l) / ja_ref) + mse(joh_l / ja_ref) \
                    + mse(jh_r / ja_ref) + mse((joh_r + j_r * ramp_r) / ja_ref)
        acid_bc_loss = flux_loss
        acid_bc_multiplier = float(self.args.faraday_weight)
        if str(H_BC.get("left_BC", "")).lower() == "dirichlet" and str(H_BC.get("right_BC", "")).lower() == "dirichlet":
            a_left_target, a_right_target = self._dirichlet_a_targets()
            a_ref = max(abs(float(self._a_from_pH_scalar(float(H_BC.get("pH_left", 0.0))))),
                        abs(float(self._a_from_pH_scalar(float(H_BC.get("pH_right", 14.0))))), 1e-8)
            acid_bc_loss = mse((self.acid_a(self.t_left, self.x_left) - a_left_target) / a_ref) \
                           + mse((self.acid_a(self.t_right, self.x_right) - a_right_target) / a_ref)
            acid_bc_multiplier = float(self.args.acid_bc_weight)
        self.loss_pH_data = (mse((self.ph_cal_pred - self.ph_cal[2]) / self.ph_cal[3])
                             if self.ph_cal is not None else tf.constant(0.0, tf.float32))
        # Keep the acid PDE, boundary, and observation terms separate.
        warm_storage, warm_transport_source, warm_acid_residual = self.acid_residual_terms(
            self.t_res, self.x_res, False
        )
        coupled_storage, coupled_transport_source, coupled_acid_residual = self.acid_residual_terms(
            self.t_res, self.x_res, True
        )
        relative_floor = tf.constant(float(self.args.acid_relative_floor), tf.float32) * res_scale
        warm_relative_residual = warm_acid_residual / tf.stop_gradient(
            tf.abs(warm_storage) + tf.abs(warm_transport_source) + relative_floor
        )
        coupled_relative_residual = coupled_acid_residual / tf.stop_gradient(
            tf.abs(coupled_storage) + tf.abs(coupled_transport_source) + relative_floor
        )
        self.loss_acid_pde_warm = mse(warm_acid_residual / res_scale)
        self.loss_acid_pde = mse(coupled_acid_residual / res_scale)
        self.loss_acid_relative_pde_warm = (
            float(self.args.acid_relative_weight) * mse(warm_relative_residual)
        )
        self.loss_acid_relative_pde = (
            float(self.args.acid_relative_weight) * mse(coupled_relative_residual)
        )
        self.loss_acid_bc = acid_bc_multiplier * acid_bc_loss
        self.loss_pH_weighted = float(self.args.pH_weight) * self.loss_pH_data
        self.loss_acid_warm = (
            self.loss_acid_pde_warm + self.loss_acid_relative_pde_warm
            + self.loss_acid_bc + self.loss_pH_weighted
        )
        self.loss_acid = (
            self.loss_acid_pde + self.loss_acid_relative_pde
            + self.loss_acid_bc + self.loss_pH_weighted
        )

        total = self.pb_total(self.t_res, self.x_res)
        j_aux, j_adv, j_diff, j_em = self.pb_fluxes(self.t_res, self.x_res)
        pb_res = grad0(total, self.t_res) + grad0(j_aux, self.x_res)
        constitutive = j_aux - (j_adv + j_diff + j_em)
        left_phys = self.pb_physical_flux(self.t_left, self.x_left)
        right_phys = self.pb_physical_flux(self.t_right, self.x_right)
        left_aux, la, ld, le = self.pb_fluxes(self.t_left, self.x_left)
        right_aux, ra, rd, re = self.pb_fluxes(self.t_right, self.x_right)
        pb_ic = mse((self.pb_total(self.t_ic, self.x_ic) - 5.0) / 5.0)
        self.loss_Pb_data = (mse((self.pb_cal_pred - self.pb_cal[2]) / self.pb_cal[3])
                             if self.pb_cal is not None else tf.constant(0.0, tf.float32))
        self.loss_pb_transport = 5.0 * mse(pb_res / 10.0) + 5.0 * mse(constitutive)
        self.loss_pb_ic = 20.0 * pb_ic
        self.loss_pb_boundary = (
            100.0 * mse(left_phys) + 100.0 * mse(tf.nn.relu(-right_phys))
            + 100.0 * mse(left_aux - (la + ld + le)) + 30.0 * mse(right_aux - (ra + rd + re))
        )
        self.loss_pb_mass = float(self.args.mass_weight) * self._mass_integral_loss()
        self.loss_Pb_weighted = float(self.args.Pb_weight) * self.loss_Pb_data
        self.loss_pb_physics = (
            self.loss_pb_transport + self.loss_pb_ic + self.loss_pb_boundary + self.loss_pb_mass
        )
        self.loss_pb = self.loss_pb_physics + self.loss_Pb_weighted

        self.loss_total = self.loss_electric + self.loss_water + self.loss_acid + self.loss_pb
        self.diagnostics = {
            "electric": self.loss_electric, "water": self.loss_water,
            "acid": self.loss_acid,
            "acid_pde": self.loss_acid_pde,
            "acid_relative_pde": self.loss_acid_relative_pde,
            "acid_boundary": self.loss_acid_bc,
            "pH_data": self.loss_pH_data,
            "pH_data_weighted": self.loss_pH_weighted,
            "pb": self.loss_pb,
            "pb_transport": self.loss_pb_transport,
            "pb_initial": self.loss_pb_ic,
            "pb_boundary": self.loss_pb_boundary,
            "pb_mass": self.loss_pb_mass,
            "Pb_data": self.loss_Pb_data,
            "Pb_data_weighted": self.loss_Pb_weighted,
        }

    @property
    def a_ic(self):
        cH = 1000.0 * 10.0 ** -7.0
        return cH - float(CFG["GLOBAL"]["CHEM"]["Kw"]) / cH

    @staticmethod
    def _a_from_pH_scalar(pH):
        cH = 1000.0 * 10.0 ** (-float(pH))
        return cH - float(CFG["GLOBAL"]["CHEM"]["Kw"]) / max(cH, 1e-30)

    @staticmethod
    def _a_from_pH_tensor(pH):
        cH = 1000.0 * tf.exp(-tf.constant(np.log(10.0), tf.float32) * pH)
        return cH - Kw_const / tf.maximum(cH, 1e-30)

    def _dirichlet_a_targets(self):
        pH0 = float(H_INIT.get("pH_ic", 7.0))
        pH_left = float(H_BC.get("pH_left", pH0))
        pH_right = float(H_BC.get("pH_right", pH0))
        tau = max(float(H_BC.get("reservoir_pH_ramp_tau", H_PAR.get("faraday_ramp_tau", 0.05))), 1e-12)
        ramp_l = 1.0 - tf.exp(-tf.maximum(self.t_left - self.tmin, 0.0) / tau)
        ramp_r = 1.0 - tf.exp(-tf.maximum(self.t_right - self.tmin, 0.0) / tau)
        return (self._a_from_pH_tensor(pH0 + ramp_l * (pH_left - pH0)),
                self._a_from_pH_tensor(pH0 + ramp_r * (pH_right - pH0)))


def read_case_data(data_dir, case, stride=1):
    def select(name):
        df = pd.read_csv(data_dir / name)
        df = df[df["case"] == case].copy()
        if df.empty:
            raise ValueError(f"No rows for case={case!r} in {data_dir / name}")
        if name.startswith("pH_"):
            local_stride = max(int(stride), 1)
            if local_stride > 1:
                group_columns = ["distance_cm"]
                selected_indices = []
                for _, block in df.groupby(group_columns, sort=False):
                    selected_indices.extend(block.index[::local_stride].tolist())
                df = df.loc[selected_indices]
        return df.reset_index(drop=True)
    return {
        "ph_cal": select("pH_calibration.csv"),
        "pb_cal": select("TotalPb_calibration.csv"),
        "ph_val": select("pH_validation.csv"),
        "pb_val": select("TotalPb_validation.csv"),
    }


def gaussian_observation_likelihood(ph_df, ph_prediction, pb_df, pb_prediction):
    """Heteroscedastic Gaussian likelihood using supplied observation SDs."""
    channels = (
        (
            "pH", ph_df, ph_prediction,
            "pH_obs", "pH_measurement_sd",
        ),
        (
            "TotalPb", pb_df, pb_prediction,
            "TotalPb_obs_mol_m3_bulk", "TotalPb_measurement_sd",
        ),
    )
    components = {}
    chi_square_components = {}
    n_observations = 0
    for name, frame, prediction, observation_column, sigma_column in channels:
        observation = frame[observation_column].to_numpy(float)
        prediction = np.asarray(prediction, dtype=float).reshape(-1)
        sigma = frame[sigma_column].to_numpy(float)
        if len(observation) != len(prediction):
            raise ValueError(f"{name} observation/prediction length mismatch")
        if not np.isfinite(sigma).all() or np.any(sigma <= 0.0):
            raise ValueError(f"{name} standard deviations must be finite and positive")
        standardized_residual = (observation - prediction) / sigma
        chi_square = float(np.sum(standardized_residual ** 2))
        components[name] = float(
            chi_square + np.sum(np.log(2.0 * np.pi * sigma ** 2))
        )
        chi_square_components[name] = chi_square
        n_observations += len(observation)
    total = float(sum(components.values()))
    total_chi_square = float(sum(chi_square_components.values()))
    return {
        "neg2loglik": total,
        "mean_neg2loglik": total / max(n_observations, 1),
        "chi_square": total_chi_square,
        "reduced_chi_square": total_chi_square / max(n_observations, 1),
        "n_observations": int(n_observations),
        "components": components,
        "chi_square_components": chi_square_components,
        "error_model": "independent Gaussian errors with observation SDs supplied in the input tables",
    }


def iterations(args):
    if args.smoke:
        return {"electric": 3, "water": 3, "acid": 5, "pb": 5, "joint": 5}
    return {"electric": args.electric_iters, "water": args.water_iters,
            "acid": args.acid_iters, "pb": args.pb_iters, "joint": args.joint_iters}


def sample_feed(cases, rng, n_res, n_face, n_ic):
    feed = {}
    for case in cases:
        feed[case.t_res] = rng.uniform(0.0, 5.0, (n_res, 1)).astype(np.float32)
        feed[case.x_res] = rng.uniform(0.0, 0.2, (n_res, 1)).astype(np.float32)
        feed[case.t_ic] = np.zeros((n_ic, 1), np.float32)
        feed[case.x_ic] = rng.uniform(0.0, 0.2, (n_ic, 1)).astype(np.float32)
        tb = rng.uniform(0.0, 5.0, (n_face, 1)).astype(np.float32)
        feed[case.t_left], feed[case.x_left] = tb, np.zeros_like(tb)
        feed[case.t_right], feed[case.x_right] = tb, np.full_like(tb, 0.2)
    return feed


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", type=Path, default=HERE / "data")
    ap.add_argument("--output-root", type=Path, default=HERE / "outputs")
    ap.add_argument("--seed", type=int, default=20260729)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--width", type=int, default=20)
    ap.add_argument("--depth", type=int, default=5)
    ap.add_argument("--n-res", type=int, default=3000)
    ap.add_argument("--n-face", type=int, default=500)
    ap.add_argument("--n-ic", type=int, default=500)
    ap.add_argument("--data-stride", type=int, default=1)
    ap.add_argument("--electric-iters", type=int, default=500)
    ap.add_argument("--water-iters", type=int, default=800)
    ap.add_argument("--acid-iters", type=int, default=2500)
    ap.add_argument("--pb-iters", type=int, default=3000)
    ap.add_argument("--joint-iters", type=int, default=300)
    ap.add_argument("--learning-rate", type=float, default=2e-4)
    ap.add_argument("--joint-learning-rate", type=float, default=2e-5,
                    help="Learning rate for state-network updates in the joint stage.")
    ap.add_argument("--pb-acid-learning-rate", type=float, default=5e-5,
                    help="Learning rate for acid-state adaptation during the coupled chemistry stage.")
    ap.add_argument("--joint-parameter-learning-rate", type=float, default=1e-4,
                    help="Learning rate for the three physical parameters during alternating joint updates.")
    ap.add_argument("--warm-lr-decay-steps", type=int, default=2000,
                    help="Optimizer updates between staircase learning-rate decays in warm-up stages; 0 disables decay.")
    ap.add_argument("--coupled-lr-decay-steps", type=int, default=2000,
                    help="Optimizer updates between staircase learning-rate decays in Pb/joint stages; 0 disables decay.")
    ap.add_argument("--lr-decay-rate", type=float, default=0.5,
                    help="Multiplicative staircase learning-rate decay factor.")
    ap.add_argument("--block-coordinate-curriculum", action="store_true",
                    help="Alternate state-only and physics-parameter-only blocks to limit compensation.")
    ap.add_argument("--warm-parameter-learning-rate", type=float, default=1e-4,
                    help="Learning rate for isolated Keos/RH updates in the water/acid curriculum.")
    ap.add_argument("--keos-parameter-learning-rate", type=float,
                    help="Override the isolated Keos learning rate.")
    ap.add_argument("--rh-parameter-learning-rate", type=float,
                    help="Override the isolated RH learning rate.")
    ap.add_argument("--ph50-parameter-learning-rate", type=float,
                    help="Override the isolated pH50 learning rate.")
    ap.add_argument("--water-state-pretrain-iters", type=int, default=500)
    ap.add_argument("--acid-state-pretrain-iters", type=int, default=1000)
    ap.add_argument("--pb-state-pretrain-iters", type=int, default=500)
    ap.add_argument("--joint-state-pretrain-iters", type=int, default=200)
    ap.add_argument("--alternating-block-size", type=int, default=100)
    ap.add_argument("--max-numerical-recoveries", type=int, default=5,
                    help="Bad collocation batches skipped after safe-state rollback before aborting a stage.")
    ap.add_argument("--joint-state-steps", type=int, default=1)
    ap.add_argument("--joint-parameter-steps", type=int, default=1)
    ap.add_argument("--joint-loss-mode", choices=("scaled", "balanced", "raw"), default="scaled",
                    help="Use frozen-scale component balancing, legacy log1p balancing, or the raw sum.")
    ap.add_argument("--balance-scale-floor", type=float, default=1e-3,
                    help="Minimum frozen warm-up scale for a joint physics component.")
    ap.add_argument("--balance-scales-json", type=Path,
                    help="Reuse frozen physics scales from a baseline run for comparable diagnostic scores.")
    ap.add_argument("--pH-weight", type=float, default=10.0)
    ap.add_argument("--Pb-weight", type=float, default=5.0)
    ap.add_argument("--faraday-weight", type=float, default=100.0)
    ap.add_argument("--acid-bc-weight", type=float, default=float(CFG["FIELDS"]["HPLUS"]["BOUNDARY"].get("pH_dirichlet_weight", 30.0)))
    ap.add_argument("--acid-relative-weight", type=float, default=2.0,
                    help="Weight on the scale-free acid balance residual used to strengthen RH sensitivity.")
    ap.add_argument("--acid-relative-floor", type=float, default=0.05,
                    help="Denominator floor as a fraction of the acid residual reference scale.")
    ap.add_argument("--mass-weight", type=float, default=50.0)
    ap.add_argument("--mass-nx", type=int, default=21)
    ap.add_argument("--mass-nt", type=int, default=21)
    ap.add_argument("--guard-every", type=int, default=150)
    ap.add_argument("--pb-patience", type=int, default=5)
    ap.add_argument("--pb-min-delta", type=float, default=1e-4)
    ap.add_argument("--joint-patience", type=int, default=5)
    ap.add_argument("--joint-min-delta", type=float, default=1e-4)
    ap.add_argument("--adaptive-stop", action="store_true",
                    help="Stop a stage only after its fixed-monitor objective and active parameters are stable.")
    ap.add_argument("--monitor-every", type=int, default=100)
    ap.add_argument("--stability-window", type=int, default=5)
    ap.add_argument("--stability-score-rtol", type=float, default=2e-3)
    ap.add_argument("--stability-parameter-rtol", type=float, default=2e-3)
    ap.add_argument("--min-electric-iters", type=int, default=500)
    ap.add_argument("--min-water-iters", type=int, default=800)
    ap.add_argument("--min-acid-iters", type=int, default=2500)
    ap.add_argument("--min-pb-iters", type=int, default=1000)
    ap.add_argument("--min-joint-iters", type=int, default=100)
    ap.add_argument("--selection-physics-weight", type=float, default=0.5,
                    help="Weight on the fixed-collocation, physics-only scaled score during checkpoint selection.")
    ap.add_argument(
        "--selection-burn-in-iters", type=int, default=0,
        help=(
            "Do not admit a resumed stage-entry state or early updates into the best-checkpoint "
            "guard until this many stage iterations have completed."
        ),
    )
    ap.add_argument(
        "--selection-data-mode",
        choices=("standardized-rmse", "gaussian-likelihood"),
        default="standardized-rmse",
        help=(
            "Checkpoint data score. Gaussian likelihood evaluates every calibration "
            "row using the observation SD supplied in the input table."
        ),
    )
    ap.add_argument("--adsorption-slope-n", type=float, default=4.0 * 0.27 / np.log(10.0))
    ap.add_argument("--init-pH50", type=float, default=7.0)
    ap.add_argument("--init-Keos", type=float, default=0.5)
    ap.add_argument("--init-RH", type=float, default=35.0)
    ap.add_argument("--ph50-lower", type=float, default=1.5)
    ap.add_argument("--ph50-upper", type=float, default=10.0)
    ap.add_argument("--fixed-keos", type=float)
    ap.add_argument("--fixed-rh", type=float)
    ap.add_argument("--fixed-ph50", type=float)
    ap.add_argument("--run-role", choices=("inversion", "diagnostic"), default="inversion")
    ap.add_argument("--run-label", default="")
    ap.add_argument("--resume-checkpoint", type=Path,
                    help="Restore model variables from a TensorFlow checkpoint prefix before training.")
    ap.add_argument(
        "--resume-network-only", action="store_true",
        help=(
            "Restore only state-network variables from --resume-checkpoint and retain "
            "the requested physical-parameter initial values. Required for genuine "
            "multi-start inverse diagnostics."
        ),
    )
    ap.add_argument("--start-stage", choices=("electric", "water", "acid", "pb", "joint"),
                    default="electric")
    ap.add_argument("--profile-rh-values", default="",
                    help="Comma-separated fixed-state RH objective slice evaluated after training.")
    ap.add_argument("--summary-only", action="store_true",
                    help="Write metrics/history only; skip checkpoints and prediction tables.")
    args = ap.parse_args(argv)

    if args.selection_physics_weight < 0.0:
        ap.error("--selection-physics-weight must be non-negative")
    if args.selection_burn_in_iters < 0:
        ap.error("--selection-burn-in-iters must be non-negative")
    if args.balance_scale_floor <= 0.0:
        ap.error("--balance-scale-floor must be positive")
    if args.acid_relative_weight < 0.0 or args.acid_relative_floor <= 0.0:
        ap.error("acid relative weight must be non-negative and its floor must be positive")
    if args.ph50_lower >= args.ph50_upper:
        ap.error("--ph50-lower must be smaller than --ph50-upper")
    if args.monitor_every < 1 or args.stability_window < 2:
        ap.error("monitor interval must be >= 1 and stability window must be >= 2")
    if args.stability_score_rtol < 0.0 or args.stability_parameter_rtol < 0.0:
        ap.error("stability tolerances must be non-negative")
    if args.joint_state_steps < 1 or args.joint_parameter_steps < 0:
        ap.error("joint state steps must be >= 1 and parameter steps must be >= 0")
    if args.warm_lr_decay_steps < 0 or args.coupled_lr_decay_steps < 0:
        ap.error("learning-rate decay steps must be non-negative")
    if not 0.0 < args.lr_decay_rate <= 1.0:
        ap.error("--lr-decay-rate must be in (0, 1]")
    curriculum_pretrains = (
        args.water_state_pretrain_iters, args.acid_state_pretrain_iters,
        args.pb_state_pretrain_iters, args.joint_state_pretrain_iters,
    )
    if min(curriculum_pretrains) < 0 or args.alternating_block_size < 1:
        ap.error("curriculum pretraining iterations must be non-negative and block size >= 1")
    if args.max_numerical_recoveries < 0:
        ap.error("--max-numerical-recoveries must be non-negative")
    parameter_learning_rates = (
        args.warm_parameter_learning_rate,
        args.keos_parameter_learning_rate,
        args.rh_parameter_learning_rate,
        args.ph50_parameter_learning_rate,
    )
    if args.warm_parameter_learning_rate <= 0.0 or any(
            value is not None and value <= 0.0 for value in parameter_learning_rates[1:]):
        ap.error("curriculum parameter learning rates must be positive")

    keos_parameter_learning_rate = (
        args.keos_parameter_learning_rate or args.warm_parameter_learning_rate
    )
    rh_parameter_learning_rate = (
        args.rh_parameter_learning_rate or args.warm_parameter_learning_rate
    )
    ph50_parameter_learning_rate = (
        args.ph50_parameter_learning_rate or args.warm_parameter_learning_rate
    )

    if args.smoke:
        args.width, args.depth = 8, 2
        args.n_res, args.n_face, args.n_ic = 64, 32, 32
        args.mass_nx, args.mass_nt = 7, 7
        args.data_stride = max(args.data_stride, 25)

    for name in ("pH_calibration.csv", "TotalPb_calibration.csv", "pH_validation.csv", "TotalPb_validation.csv"):
        if not (args.data_dir / name).exists():
            raise FileNotFoundError(f"Observation file not found: {args.data_dir / name}")

    CFG["DOMAIN"]["tmax"] = 5.0
    DOMAIN["tmax"] = 5.0
    NETC["width"], NETC["hidden_layers"] = args.width, args.depth
    tf.compat.v1.set_random_seed(args.seed)
    np.random.seed(args.seed)

    with tf.compat.v1.variable_scope("inverse_parameters"):
        Keos, keos_vars = inversion_parameter("Keos", args.init_Keos, 0.05, 3.0, args.fixed_keos)
        RH, rh_vars = inversion_parameter("RH", args.init_RH, 1.0, 70.0, args.fixed_rh)
        pH50, ph50_vars = inversion_parameter(
            "pH50", args.init_pH50, args.ph50_lower, args.ph50_upper, args.fixed_ph50
        )
    parameter_vars = keos_vars + rh_vars + ph50_vars
    parameter_tensors = [Keos, RH, pH50]

    layers = [2] + [args.width] * args.depth + [1]
    cases = [
        CaseGraph(
            name, meta, layers, (Keos, RH, pH50),
            read_case_data(args.data_dir, name, args.data_stride), args,
            evaluation_data=read_case_data(args.data_dir, name, 1),
        )
        for name, meta in CASES.items()
    ]

    electric_loss = tf.add_n([c.loss_electric_pre for c in cases])
    water_loss = tf.add_n([c.loss_water for c in cases])
    acid_loss = tf.add_n([c.loss_acid_warm for c in cases])
    # Coupled chemistry alternates two safe updates.  The acid state follows the
    # coupled acid residual, while the Pb state plus RH/pH50 use the stable Pb +
    # uncoupled-acid objective.  This avoids differentiating the nonsmooth Pb
    # mass integral through the acid network (a source of NaN gradients), while
    # still allowing the acid state to adapt between parameter updates.
    pb_loss = tf.add_n([c.loss_pb + c.loss_acid_warm for c in cases])
    pb_coupled_acid_loss = tf.add_n([c.loss_acid for c in cases])
    joint_loss = tf.add_n([c.loss_total for c in cases])
    joint_loss_balanced = tf.add_n([
        tf.math.log1p(tf.maximum(component, 0.0))
        for case in cases
        for component in (case.loss_electric, case.loss_water, case.loss_acid, case.loss_pb)
    ])

    physics_components = {}
    data_components = {}
    for case in cases:
        physics_components.update({
            f"{case.name}.electric": case.loss_electric,
            f"{case.name}.water": case.loss_water,
            f"{case.name}.acid_pde": case.loss_acid_pde,
            f"{case.name}.acid_relative_pde": case.loss_acid_relative_pde,
            f"{case.name}.acid_boundary": case.loss_acid_bc,
            f"{case.name}.pb_transport": case.loss_pb_transport,
            f"{case.name}.pb_initial": case.loss_pb_ic,
            f"{case.name}.pb_boundary": case.loss_pb_boundary,
            f"{case.name}.pb_mass": case.loss_pb_mass,
        })
        # These are already standardized mean-square errors, so a target scale
        # of one has a direct statistical interpretation.
        data_components.update({
            f"{case.name}.pH_data": case.loss_pH_data,
            f"{case.name}.Pb_data": case.loss_Pb_data,
        })

    balance_scale_vars = {}
    balance_scale_inputs = {}
    balance_scale_assignments = []
    with tf.compat.v1.variable_scope("joint_balance_scales"):
        for key in physics_components:
            safe_key = key.replace(".", "_")
            scale = tf.compat.v1.get_variable(
                safe_key, shape=(), dtype=tf.float32,
                initializer=tf.compat.v1.constant_initializer(1.0), trainable=False,
            )
            scale_input = tf.compat.v1.placeholder(tf.float32, shape=(), name=f"{safe_key}_input")
            balance_scale_vars[key] = scale
            balance_scale_inputs[key] = scale_input
            balance_scale_assignments.append(scale.assign(scale_input))

    scaled_physics_terms = [
        tf.math.log1p(tf.maximum(component, 0.0) / balance_scale_vars[key])
        for key, component in physics_components.items()
    ]
    scaled_data_terms = [
        tf.math.log1p(tf.maximum(component, 0.0))
        for component in data_components.values()
    ]
    joint_loss_scaled = tf.add_n(scaled_physics_terms + scaled_data_terms) / float(
        len(scaled_physics_terms) + len(scaled_data_terms)
    )
    joint_physics_scaled = tf.add_n(scaled_physics_terms) / float(len(scaled_physics_terms))
    pb_acid_state_terms = []
    for case in cases:
        pb_acid_state_terms.extend([
            tf.math.log1p(
                tf.maximum(case.loss_acid_pde, 0.0)
                / balance_scale_vars[f"{case.name}.acid_pde"]
            ),
            tf.math.log1p(
                tf.maximum(case.loss_acid_relative_pde, 0.0)
                / balance_scale_vars[f"{case.name}.acid_relative_pde"]
            ),
            tf.math.log1p(
                tf.maximum(case.loss_acid_bc, 0.0)
                / balance_scale_vars[f"{case.name}.acid_boundary"]
            ),
            tf.math.log1p(tf.maximum(case.loss_pH_data, 0.0)),
        ])
    pb_acid_state_objective = tf.add_n(pb_acid_state_terms) / float(len(pb_acid_state_terms))
    joint_objective = {
        "scaled": joint_loss_scaled,
        "balanced": joint_loss_balanced,
        "raw": joint_loss,
    }[args.joint_loss_mode]
    electric_vars = sum((c.phi_vars for c in cases), []) + parameter_vars
    water_state_vars = sum((c.water_vars for c in cases), [])
    acid_state_vars = sum((c.acid_vars for c in cases), [])
    water_vars = water_state_vars + parameter_vars
    acid_vars = acid_state_vars + parameter_vars
    pb_acid_state_vars = sum((c.acid_vars for c in cases), [])
    pb_state_vars = sum((c.pb_vars for c in cases), [])
    pb_transport_vars = pb_state_vars + rh_vars + ph50_vars
    state_vars = sum((c.phi_vars + c.water_vars + c.acid_vars + c.pb_vars for c in cases), [])
    all_model_vars = state_vars + parameter_vars

    ops = {}
    ops["electric"] = make_optimizer(
        electric_loss, electric_vars, args.learning_rate, 5.0, "opt_electric",
        decay_steps=args.warm_lr_decay_steps, decay_rate=args.lr_decay_rate,
    )
    curriculum_state_ops = {}
    curriculum_parameter_ops = {}
    if args.block_coordinate_curriculum:
        curriculum_state_ops["water"] = make_optimizer(
            water_loss, water_state_vars, args.learning_rate, 5.0, "opt_water_state",
            decay_steps=args.warm_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
        curriculum_parameter_ops["water"] = make_optimizer(
            water_loss, keos_vars, keos_parameter_learning_rate, 1.0,
            "opt_water_parameter", clip_each=True,
            decay_steps=args.warm_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
        curriculum_state_ops["acid"] = make_optimizer(
            acid_loss, acid_state_vars, args.learning_rate, 2.0, "opt_acid_state",
            decay_steps=args.warm_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
        # Keos has already been isolated by the water equations.  Only RH is
        # released here; pH50 remains reserved for the Pb-observation stage.
        curriculum_parameter_ops["acid"] = make_optimizer(
            acid_loss, rh_vars, rh_parameter_learning_rate, 1.0,
            "opt_acid_parameter", clip_each=True,
            decay_steps=args.warm_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
        ops["water"] = curriculum_state_ops["water"]
        ops["acid"] = curriculum_state_ops["acid"]
    else:
        ops["water"] = make_optimizer(
            water_loss, water_vars, args.learning_rate, 5.0, "opt_water",
            decay_steps=args.warm_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
        ops["acid"] = make_optimizer(
            acid_loss, acid_vars, args.learning_rate, 2.0, "opt_acid",
            decay_steps=args.warm_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
    pb_acid_state_op = make_optimizer(
        pb_acid_state_objective, pb_acid_state_vars, args.pb_acid_learning_rate, 0.5,
        "opt_pb_acid_state", clip_each=True,
        decay_steps=args.coupled_lr_decay_steps, decay_rate=args.lr_decay_rate,
    )
    if args.block_coordinate_curriculum:
        pb_state_op = make_optimizer(
            pb_loss, pb_state_vars, args.learning_rate, 1.0,
            "opt_pb_state", clip_each=True,
            decay_steps=args.coupled_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
        # RH is learned from the dense pH history.  The seven spatial locations
        # contain only four Pb calibration locations per case, all at day 5, so
        # the sparse Pb stage releases pH50 alone to prevent RH-pH50 compensation.
        pb_parameter_op = make_optimizer(
            pb_loss, ph50_vars, ph50_parameter_learning_rate, 1.0,
            "opt_pb_parameter", clip_each=True,
            decay_steps=args.coupled_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
        pb_transport_op = None
    else:
        pb_state_op = None
        pb_parameter_op = None
        pb_transport_op = make_optimizer(
            pb_loss, pb_transport_vars, args.learning_rate, 1.0,
            "opt_pb_transport", clip_each=True,
            decay_steps=args.coupled_lr_decay_steps, decay_rate=args.lr_decay_rate,
        )
    joint_state_op = make_optimizer(
        joint_objective, state_vars, args.joint_learning_rate, 1.0,
        "opt_joint_state", clip_each=True,
        decay_steps=args.coupled_lr_decay_steps, decay_rate=args.lr_decay_rate,
    )
    joint_parameter_op = make_optimizer(
        joint_objective, parameter_vars, args.joint_parameter_learning_rate, 1.0,
        "opt_joint_parameters", clip_each=True,
        decay_steps=args.coupled_lr_decay_steps, decay_rate=args.lr_decay_rate,
    )

    parameter_names = {v.name for v in parameter_vars}
    stage_parameter_gradients = {
        stage: [v.name for v in active_vars if v.name in parameter_names]
        for stage, (_, _, active_vars) in ops.items()
    }
    if args.block_coordinate_curriculum:
        stage_parameter_gradients["water"] = [
            v.name for v in curriculum_parameter_ops["water"][2]
            if v.name in parameter_names
        ]
        stage_parameter_gradients["acid"] = [
            v.name for v in curriculum_parameter_ops["acid"][2]
            if v.name in parameter_names
        ]
        stage_parameter_gradients["pb"] = [
            v.name for v in pb_parameter_op[2] if v.name in parameter_names
        ]
    else:
        stage_parameter_gradients["pb"] = [
            v.name for v in pb_transport_op[2] if v.name in parameter_names
        ]
    stage_parameter_gradients["joint"] = [
        v.name for v in joint_parameter_op[2] if v.name in parameter_names
    ]

    args.output_root.mkdir(parents=True, exist_ok=True)
    outdir = args.output_root / datetime.now().strftime("%Y%m%d_%H%M%S")
    outdir.mkdir(parents=True, exist_ok=False)
    run_config = dict(vars(args))
    run_config["evaluation_data_stride"] = 1
    run_config["inversion_version"] = "three_parameter_inversion"
    run_config["effective_iterations"] = iterations(args)
    run_config["shared_trainable_parameters"] = [v.name for v in parameter_vars]
    run_config["stage_parameter_gradients"] = stage_parameter_gradients
    run_config["pb_stage"] = (
        "block-coordinate acid/Pb state updates alternating with isolated pH50 updates; "
        "Keos and RH frozen" if args.block_coordinate_curriculum else
        "alternating coupled-acid state and Pb+acid-warm transport/parameter updates; "
        "RH and pH50 trainable; Keos fixed"
    )
    run_config["acid_Pb_source_coupling"] = "Picard value coupling with stopped source gradient"
    run_config["pb_acid_state_objective"] = (
        "scaled acid PDE + scaled relative acid balance + scaled acid boundary + standardized pH data"
    )
    run_config["rh_sensitivity_strengthening"] = (
        "relative full acid balance residual with stopped adaptive denominator; original PDE zero set retained"
    )
    run_config["stopping_rule"] = (
        "adaptive fixed-monitor objective plus active-parameter stability with safety caps"
        if args.adaptive_stop else "iteration caps plus legacy PB/joint patience"
    )
    run_config["parameter_compensation_control"] = (
        "process curriculum plus state/parameter block-coordinate optimization; "
        "water->Keos, acid/pH->RH, sparse day-5 Pb->pH50, then low-rate joint refinement"
        if args.block_coordinate_curriculum else "joint state/parameter optimization"
    )
    run_config["effective_isolated_parameter_learning_rates"] = {
        "Keos": keos_parameter_learning_rate,
        "RH": rh_parameter_learning_rate,
        "pH50": ph50_parameter_learning_rate,
    }
    run_config["checkpoint_selection"] = "calibration_plus_fixed_collocation_scaled_physics_only"
    run_config["validation_usage"] = "final reporting only"
    (outdir / "run_config.json").write_text(json.dumps(run_config, default=str, indent=2), encoding="utf-8")
    model_saver = tf.compat.v1.train.Saver(var_list=all_model_vars, max_to_keep=6)
    network_saver = tf.compat.v1.train.Saver(var_list=state_vars)
    saver = None if args.summary_only else model_saver
    rng = np.random.default_rng(args.seed)
    history = []
    rh_objective_slice = []
    stage_iters = iterations(args)
    n_res = args.n_res
    min_stage_iters = {
        "electric": args.min_electric_iters,
        "water": args.min_water_iters,
        "acid": args.min_acid_iters,
        "pb": args.min_pb_iters,
        "joint": args.min_joint_iters,
    }
    if args.block_coordinate_curriculum:
        # A stage may not be declared stable while it is still in state-only
        # pretraining.  Require a full stability window after parameters have
        # first been released.
        curriculum_pretrain_limits = {
            "water": args.water_state_pretrain_iters,
            "acid": args.acid_state_pretrain_iters,
            "pb": args.pb_state_pretrain_iters,
            "joint": args.joint_state_pretrain_iters,
        }
        for stage, pretrain_iterations in curriculum_pretrain_limits.items():
            min_stage_iters[stage] = max(
                int(min_stage_iters[stage]),
                int(pretrain_iterations)
                + int(args.stability_window) * int(args.monitor_every),
            )
    stage_parameter_indices = {
        "electric": (),
        "water": (0,) if keos_vars else (),
        "acid": tuple(index for index, variables in enumerate((keos_vars, rh_vars, ph50_vars))
                      if (index == 1 if args.block_coordinate_curriculum else index < 2) and variables),
        "pb": tuple(index for index, variables in enumerate((keos_vars, rh_vars, ph50_vars))
                    if (index == 2 if args.block_coordinate_curriculum else index in (1, 2)) and variables),
        "joint": tuple(index for index, variables in enumerate((keos_vars, rh_vars, ph50_vars))
                       if variables),
    }
    stability_records = {stage: [] for stage in ("electric", "water", "acid", "pb", "joint")}
    stage_stop_status = {}
    warm_best = {
        stage: {"score": np.inf, "iteration": None, "values": None, "parameters": None}
        for stage in ("electric", "water", "acid")
    }

    def update_stability(stage, iteration, score, parameter_values):
        records = stability_records[stage]
        records.append((float(score), np.asarray(parameter_values, dtype=float)))
        records[:] = records[-int(args.stability_window):]
        stats = {"stable": False, "score_relative_span": None, "parameter_relative_span": None}
        if not args.adaptive_stop or iteration < int(min_stage_iters[stage]):
            return False, stats
        if len(records) < int(args.stability_window):
            return False, stats
        scores = np.asarray([record[0] for record in records], dtype=float)
        score_scale = max(abs(float(np.mean(scores))), 1e-12)
        score_span = float(np.ptp(scores) / score_scale)
        indices = stage_parameter_indices[stage]
        parameter_span = 0.0
        if indices:
            values = np.asarray([record[1][list(indices)] for record in records], dtype=float)
            scales = np.maximum(np.abs(np.mean(values, axis=0)), 1.0)
            parameter_span = float(np.max(np.ptp(values, axis=0) / scales))
        stable = (
            score_span <= float(args.stability_score_rtol)
            and parameter_span <= float(args.stability_parameter_rtol)
        )
        stats.update({
            "stable": bool(stable),
            "score_relative_span": score_span,
            "parameter_relative_span": parameter_span,
        })
        return bool(stable), stats

    with tf.compat.v1.Session(config=TF_CONFIG) as sess:
        sess.run(tf.compat.v1.global_variables_initializer())
        if args.resume_checkpoint is not None:
            checkpoint_prefix = str(args.resume_checkpoint)
            if not Path(checkpoint_prefix + ".index").exists():
                raise FileNotFoundError(f"Checkpoint prefix not found: {checkpoint_prefix}")
            (network_saver if args.resume_network_only else model_saver).restore(
                sess, checkpoint_prefix
            )
            print(json.dumps({"restored_training_checkpoint": checkpoint_prefix,
                              "network_only": bool(args.resume_network_only),
                              "start_stage": args.start_stage}))
        guard_feed = sample_feed(
            cases, np.random.default_rng(args.seed + 2), n_res, args.n_face, args.n_ic
        )
        best = {"score": np.inf, "stage": None, "iteration": None, "values": None, "parameters": None, "metrics": None}
        balance_scales = {}

        def initialize_balance_scales():
            """Freeze warm-up physics scales before any Pb checkpoint is ranked."""
            if args.balance_scales_json is not None:
                loaded = json.loads(args.balance_scales_json.read_text(encoding="utf-8"))
                loaded = loaded.get("balance_scales", loaded)
                missing = sorted(set(physics_components) - set(loaded))
                unsupported_missing = [key for key in missing if not key.endswith(".acid_relative_pde")]
                if unsupported_missing:
                    raise ValueError(
                        f"Missing balance scales in {args.balance_scales_json}: {unsupported_missing}"
                    )
                balance_scales.update({
                    key: max(float(loaded[key]), float(args.balance_scale_floor))
                    for key in physics_components if key in loaded
                })
                if missing:
                    missing_values = sess.run(
                        [physics_components[key] for key in missing], guard_feed
                    )
                    balance_scales.update({
                        key: max(float(value), float(args.balance_scale_floor))
                        for key, value in zip(missing, missing_values)
                    })
                scale_source = str(args.balance_scales_json)
            else:
                component_values = sess.run(list(physics_components.values()), guard_feed)
                balance_scales.update({
                    key: max(float(value), float(args.balance_scale_floor))
                    for key, value in zip(physics_components, component_values)
                })
                scale_source = "current_run_after_acid_warmup"
            assign_feed = {
                balance_scale_inputs[key]: value for key, value in balance_scales.items()
            }
            sess.run(balance_scale_assignments, assign_feed)
            run_config["balance_scales"] = balance_scales
            run_config["balance_scale_source"] = scale_source
            (outdir / "run_config.json").write_text(
                json.dumps(run_config, default=str, indent=2), encoding="utf-8"
            )
            print(json.dumps({"joint_balance_scales_frozen": True,
                              "source": scale_source, **balance_scales}))

        def calibration_metrics_and_score():
            """Calibration plus fixed physics score; validation remains untouched."""
            metrics_now = {}
            ph_frames, ph_predictions = [], []
            pb_frames, pb_predictions = [], []
            for case in cases:
                ph_pred, pb_pred = sess.run(
                    [case.ph_cal_full_pred, case.pb_cal_full_pred]
                )
                ph_err = np.asarray(ph_pred).reshape(-1) - case.ph_cal_full_df["pH_obs"].to_numpy(float)
                pb_err = np.asarray(pb_pred).reshape(-1) - case.pb_cal_full_df["TotalPb_obs_mol_m3_bulk"].to_numpy(float)
                ph_measurement_sd = case.ph_cal_full_df["pH_measurement_sd"].to_numpy(float)
                pb_measurement_sd = case.pb_cal_full_df["TotalPb_measurement_sd"].to_numpy(float)
                metrics_now[f"{case.name}_pH_calibration_RMSE"] = float(np.sqrt(np.mean(ph_err ** 2)))
                metrics_now[f"{case.name}_TotalPb_calibration_RMSE"] = float(np.sqrt(np.mean(pb_err ** 2)))
                metrics_now[f"{case.name}_pH_calibration_standardized_RMSE"] = float(
                    np.sqrt(np.mean((ph_err / ph_measurement_sd) ** 2))
                )
                metrics_now[f"{case.name}_TotalPb_calibration_standardized_RMSE"] = float(
                    np.sqrt(np.mean((pb_err / pb_measurement_sd) ** 2))
                )
                ph_frames.append(case.ph_cal_full_df)
                ph_predictions.append(np.asarray(ph_pred).reshape(-1))
                pb_frames.append(case.pb_cal_full_df)
                pb_predictions.append(np.asarray(pb_pred).reshape(-1))
            likelihood = gaussian_observation_likelihood(
                pd.concat(ph_frames, ignore_index=True), np.concatenate(ph_predictions),
                pd.concat(pb_frames, ignore_index=True), np.concatenate(pb_predictions),
            )
            metrics_now["calibration_neg2loglik"] = likelihood["neg2loglik"]
            metrics_now["calibration_mean_neg2loglik"] = likelihood["mean_neg2loglik"]
            ph_score = np.mean([v for k, v in metrics_now.items() if k.endswith("pH_calibration_standardized_RMSE")])
            pb_score = np.mean([v for k, v in metrics_now.items() if k.endswith("TotalPb_calibration_standardized_RMSE")])
            data_score = (
                float(likelihood["mean_neg2loglik"])
                if args.selection_data_mode == "gaussian-likelihood"
                else float(0.5 * (ph_score + pb_score))
            )
            physics_score = float(sess.run(joint_physics_scaled, guard_feed))
            metrics_now["selection_data_score"] = data_score
            metrics_now["selection_physics_objective"] = physics_score
            total_score = data_score + float(args.selection_physics_weight) * physics_score
            return metrics_now, total_score

        def remember_best(stage, iteration, min_delta=0.0):
            metrics_now, score = calibration_metrics_and_score()
            improved = False
            if score < best["score"] - float(min_delta):
                best.update({
                    "score": score,
                    "stage": stage,
                    "iteration": int(iteration),
                    "values": sess.run(all_model_vars),
                    "parameters": [float(v) for v in sess.run(parameter_tensors)],
                    "metrics": metrics_now,
                })
                print(json.dumps({"best_guard": True, "stage": stage, "iteration": int(iteration),
                                  "selection_basis": "calibration_plus_scaled_physics_only",
                                  "score": score, **metrics_now}))
                improved = True
            return improved, metrics_now, score

        def restore_best():
            if best["values"] is None:
                return
            sess.run([v.assign(value) for v, value in zip(all_model_vars, best["values"])])
            print(json.dumps({"restored_best_guard": True, "stage": best["stage"],
                              "iteration": best["iteration"], "score": best["score"],
                              "parameters": best["parameters"], "metrics": best["metrics"]}))

        def remember_warm_best(stage, iteration, score):
            """Keep the lowest fixed-monitor warm-up state, independent of random batches."""
            record = warm_best[stage]
            if not np.isfinite(score) or score >= record["score"]:
                return False
            record.update({
                "score": float(score),
                "iteration": int(iteration),
                "values": sess.run(all_model_vars),
                "parameters": [float(v) for v in sess.run(parameter_tensors)],
            })
            if saver is not None:
                saver.save(
                    sess, str(outdir / f"checkpoint_{stage}_best.ckpt"),
                    write_meta_graph=False,
                )
            print(json.dumps({
                "warm_best": True, "stage": stage, "iteration": int(iteration),
                "fixed_monitor_objective": float(score),
                "parameters": record["parameters"],
            }))
            return True

        def restore_warm_best(stage, reason):
            record = warm_best[stage]
            if record["values"] is None:
                return False
            sess.run([v.assign(value) for v, value in zip(all_model_vars, record["values"])])
            print(json.dumps({
                "restored_warm_best": True, "stage": stage, "reason": reason,
                "iteration": record["iteration"], "score": record["score"],
                "parameters": record["parameters"],
            }))
            return True

        stage_sequence = ("electric", "water", "acid", "pb", "joint")
        stage_sequence = stage_sequence[stage_sequence.index(args.start_stage):]
        curriculum_pretrain = {
            "water": args.water_state_pretrain_iters,
            "acid": args.acid_state_pretrain_iters,
            "pb": args.pb_state_pretrain_iters,
            "joint": args.joint_state_pretrain_iters,
        }
        for stage in stage_sequence:
            if stage in ("pb", "joint") and not balance_scales:
                initialize_balance_scales()
            # A continuous run already carries the best Pb guard into the joint
            # stage. A resumed run starts with an empty in-memory guard, so seed
            # it from the restored checkpoint before applying even one update.
            # This keeps resumed Pb/joint refinement monotone under the fixed
            # calibration-plus-physics selection score.
            if (stage in ("pb", "joint") and best["values"] is None
                    and args.selection_burn_in_iters == 0):
                remember_best(f"{stage}_entry", 0)
            if args.block_coordinate_curriculum and stage in ("water", "acid"):
                active_vars = (
                    list(curriculum_state_ops[stage][2])
                    + list(curriculum_parameter_ops[stage][2])
                )
            elif args.block_coordinate_curriculum and stage == "pb":
                active_vars = (
                    list(pb_acid_state_op[2]) + list(pb_state_op[2])
                    + list(pb_parameter_op[2])
                )
            elif stage == "joint":
                active_vars = list(joint_state_op[2]) + list(joint_parameter_op[2])
            elif stage == "pb":
                active_vars = list(pb_acid_state_op[2]) + list(pb_transport_op[2])
            else:
                _, _, active_vars = ops[stage]
            count = stage_iters[stage]
            print(f"\n== {stage.upper()} | iters={count} | active={len(active_vars)} | "
                  f"trainable shared parameters={len(stage_parameter_gradients[stage])} ==")
            raw_loss_tensor = {"electric": electric_loss, "water": water_loss, "acid": acid_loss,
                               "pb": pb_loss + pb_coupled_acid_loss, "joint": joint_loss}[stage]
            objective_tensor = joint_objective if stage == "joint" else raw_loss_tensor
            no_improve = 0
            stage_stopped_stable = False
            stage_numerical_abort = False
            numerical_recovery_count = 0
            last_iteration = 0
            stage_entry_values = sess.run(all_model_vars)
            monitor_interval = (
                max(1, int(args.monitor_every)) if args.adaptive_stop
                else max(1, count // 10)
            )
            for it in range(count):
                last_iteration = it + 1
                feed = sample_feed(cases, rng, n_res, args.n_face, args.n_ic)
                curriculum_phase = "joint_update"
                if args.block_coordinate_curriculum and stage in curriculum_pretrain:
                    if it < int(curriculum_pretrain[stage]):
                        curriculum_phase = "state_pretrain"
                    else:
                        block_index = (
                            (it - int(curriculum_pretrain[stage]))
                            // int(args.alternating_block_size)
                        )
                        curriculum_phase = "parameter" if block_index % 2 == 0 else "state"
                    # In profile fits the parameter assigned to this isolated
                    # stage may be fixed.  Use that block for state training
                    # instead of spending half the staged refit on a no-op.
                    if curriculum_phase == "parameter":
                        if (
                            stage in ("water", "acid")
                            and not curriculum_parameter_ops[stage][2]
                        ):
                            curriculum_phase = "state"
                        elif stage == "pb" and not pb_parameter_op[2]:
                            curriculum_phase = "state"
                        elif stage == "joint" and not joint_parameter_op[2]:
                            curriculum_phase = "state"
                try:
                    if args.block_coordinate_curriculum and stage in ("water", "acid"):
                        selected_op = (
                            curriculum_parameter_ops[stage]
                            if curriculum_phase == "parameter"
                            else curriculum_state_ops[stage]
                        )
                        sess.run(selected_op[0], feed)
                    elif args.block_coordinate_curriculum and stage == "pb":
                        if curriculum_phase == "parameter":
                            sess.run(pb_parameter_op[0], feed)
                        else:
                            sess.run(pb_acid_state_op[0], feed)
                            sess.run(pb_state_op[0], feed)
                    elif args.block_coordinate_curriculum and stage == "joint":
                        if curriculum_phase == "parameter":
                            for _ in range(args.joint_parameter_steps):
                                sess.run(joint_parameter_op[0], feed)
                        else:
                            for _ in range(args.joint_state_steps):
                                sess.run(joint_state_op[0], feed)
                    elif stage == "joint":
                        for _ in range(args.joint_state_steps):
                            sess.run(joint_state_op[0], feed)
                        for _ in range(args.joint_parameter_steps):
                            sess.run(joint_parameter_op[0], feed)
                    elif stage == "pb":
                        sess.run(pb_acid_state_op[0], feed)
                        sess.run(pb_transport_op[0], feed)
                    else:
                        sess.run(ops[stage][0], feed)
                except tf.errors.InvalidArgumentError as exc:
                    if not args.adaptive_stop:
                        raise
                    recovered_from = None
                    if stage in warm_best and restore_warm_best(stage, "non_finite_gradient"):
                        recovered_from = "warm_stage_best"
                    elif stage in ("pb", "joint") and best["values"] is not None:
                        restore_best()
                        recovered_from = "coupled_guard_best"
                    else:
                        sess.run([v.assign(value) for v, value in zip(all_model_vars, stage_entry_values)])
                        recovered_from = "stage_entry"
                    print(json.dumps({
                        "numerical_recovery": True, "stage": stage,
                        "iteration": it + 1, "recovered_from": recovered_from,
                        "recovery_count": numerical_recovery_count + 1,
                        "error": str(exc).splitlines()[0],
                    }))
                    numerical_recovery_count += 1
                    if numerical_recovery_count <= int(args.max_numerical_recoveries):
                        # A check-numerics failure occurs before Adam applies the
                        # update.  Roll back the model, discard this random batch,
                        # and keep training so one pathological collocation draw
                        # cannot truncate parameter exploration.
                        continue
                    print(json.dumps({
                        "numerical_recovery_exhausted": True, "stage": stage,
                        "iteration": it + 1,
                        "max_numerical_recoveries": int(args.max_numerical_recoveries),
                    }))
                    stage_numerical_abort = True
                    break
                log_now = it == 0 or (it + 1) % monitor_interval == 0 or it + 1 == count
                if log_now:
                    if stage == "joint":
                        if args.block_coordinate_curriculum:
                            selected_op = (
                                joint_parameter_op if curriculum_phase == "parameter"
                                else joint_state_op
                            )
                            objective_value, loss_value, active_grad_value, values = sess.run(
                                [objective_tensor, raw_loss_tensor, selected_op[1], parameter_tensors], feed
                            )
                            state_grad_value = (
                                0.0 if curriculum_phase == "parameter" else active_grad_value
                            )
                            parameter_grad_value = (
                                active_grad_value if curriculum_phase == "parameter" else 0.0
                            )
                        else:
                            objective_value, loss_value, state_grad_value, parameter_grad_value, values = sess.run(
                                [objective_tensor, raw_loss_tensor, joint_state_op[1],
                                 joint_parameter_op[1], parameter_tensors], feed
                            )
                        grad_value = float(math.hypot(state_grad_value, parameter_grad_value))
                    elif stage == "pb":
                        if args.block_coordinate_curriculum:
                            if curriculum_phase == "parameter":
                                objective_value, loss_value, parameter_grad_value, values = sess.run(
                                    [objective_tensor, raw_loss_tensor, pb_parameter_op[1], parameter_tensors], feed
                                )
                                state_grad_value = 0.0
                            else:
                                objective_value, loss_value, acid_grad_value, pb_grad_value, values = sess.run(
                                    [objective_tensor, raw_loss_tensor, pb_acid_state_op[1],
                                     pb_state_op[1], parameter_tensors], feed
                                )
                                state_grad_value = float(math.hypot(acid_grad_value, pb_grad_value))
                                parameter_grad_value = 0.0
                        else:
                            objective_value, loss_value, acid_grad_value, transport_grad_value, values = sess.run(
                                [objective_tensor, raw_loss_tensor, pb_acid_state_op[1],
                                 pb_transport_op[1], parameter_tensors], feed
                            )
                            state_grad_value, parameter_grad_value = acid_grad_value, transport_grad_value
                        grad_value = float(math.hypot(state_grad_value, parameter_grad_value))
                    else:
                        if args.block_coordinate_curriculum and stage in ("water", "acid"):
                            selected_op = (
                                curriculum_parameter_ops[stage]
                                if curriculum_phase == "parameter"
                                else curriculum_state_ops[stage]
                            )
                            objective_value, loss_value, grad_value, values = sess.run(
                                [objective_tensor, raw_loss_tensor, selected_op[1], parameter_tensors], feed
                            )
                            state_grad_value = (
                                0.0 if curriculum_phase == "parameter" else float(grad_value)
                            )
                            parameter_grad_value = (
                                float(grad_value) if curriculum_phase == "parameter" else 0.0
                            )
                        else:
                            objective_value, loss_value, grad_value, values = sess.run(
                                [objective_tensor, raw_loss_tensor, ops[stage][1], parameter_tensors], feed
                            )
                            state_grad_value, parameter_grad_value = float(grad_value), 0.0
                    if (not np.isfinite(objective_value) or not np.isfinite(loss_value)
                            or not np.isfinite(values).all()):
                        raise FloatingPointError(
                            f"Non-finite state at {stage} iteration {it + 1}: "
                            f"objective={objective_value}, loss={loss_value}, parameters={values}"
                        )
                    row = {"stage": stage, "iteration": it + 1, "loss": float(loss_value),
                           "optimization_loss": float(objective_value),
                           "grad_norm": float(grad_value), "Keos": float(values[0]),
                           "RH": float(values[1]), "pH50": float(values[2])}
                    if stage == "joint":
                        row["state_grad_norm"] = float(state_grad_value)
                        row["parameter_grad_norm"] = float(parameter_grad_value)
                    elif stage == "pb":
                        row["acid_state_grad_norm"] = float(state_grad_value)
                        row["pb_transport_grad_norm"] = float(parameter_grad_value)
                    if args.block_coordinate_curriculum and stage != "electric":
                        row["curriculum_phase"] = curriculum_phase
                        row["state_grad_norm"] = float(state_grad_value)
                        row["parameter_grad_norm"] = float(parameter_grad_value)
                    history.append(row)
                    print(json.dumps(row))
                    pd.DataFrame(history).to_csv(outdir / "training_history.csv", index=False)
                    if stage == "joint":
                        selection_eligible = (
                            it + 1 >= int(args.selection_burn_in_iters)
                            and (
                                not args.block_coordinate_curriculum
                                or curriculum_phase == "state"
                            )
                        )
                        if selection_eligible:
                            improved, _, selection_score = remember_best(
                                stage, it + 1, args.joint_min_delta
                            )
                        else:
                            _, selection_score = calibration_metrics_and_score()
                            improved = False
                            print(json.dumps({
                                "selection_deferred": True, "stage": stage,
                                "iteration": it + 1,
                                "eligible_after_iteration": int(args.selection_burn_in_iters),
                                "curriculum_phase": curriculum_phase,
                                "selection_score": float(selection_score),
                            }))
                        if improved:
                            no_improve = 0
                        else:
                            no_improve += 1
                        if selection_eligible:
                            stability_score = float(best["score"])
                            stability_parameters = np.asarray(best["parameters"], dtype=float)
                            stable, stability_stats = update_stability(
                                stage, it + 1, stability_score, stability_parameters
                            )
                        else:
                            stable = False
                            stability_stats = {"stability_ready": False}
                        if args.adaptive_stop:
                            print(json.dumps({
                                "adaptive_monitor": True, "stage": stage,
                                "iteration": it + 1, "selection_score": selection_score,
                                "guarded_best_score": (
                                    None if best["values"] is None else float(best["score"])
                                ),
                                **stability_stats,
                            }))
                        if args.adaptive_stop and stable:
                            print(json.dumps({
                                "stable_stop": True, "stage": stage, "iteration": it + 1,
                                "selection_basis": "calibration_plus_scaled_physics_only",
                                **stability_stats,
                            }))
                            stage_stopped_stable = True
                            break
                        if (not args.adaptive_stop and args.joint_patience > 0
                                and no_improve >= args.joint_patience):
                            print(json.dumps({"early_stop": True, "stage": stage, "iteration": it + 1,
                                              "selection_basis": "calibration_plus_scaled_physics_only",
                                              "best_stage": best["stage"], "best_iteration": best["iteration"],
                                              "best_score": best["score"]}))
                            break
                    elif stage in ("electric", "water", "acid") and args.adaptive_stop:
                        fixed_monitor_objective = float(sess.run(objective_tensor, guard_feed))
                        remember_warm_best(stage, it + 1, fixed_monitor_objective)
                        stable, stability_stats = update_stability(
                            stage, it + 1, fixed_monitor_objective, values
                        )
                        print(json.dumps({
                            "adaptive_monitor": True, "stage": stage,
                            "iteration": it + 1,
                            "fixed_monitor_objective": fixed_monitor_objective,
                            **stability_stats,
                        }))
                        if stable:
                            print(json.dumps({
                                "stable_stop": True, "stage": stage, "iteration": it + 1,
                                "selection_basis": "fixed_monitor_objective_plus_parameter_stability",
                                **stability_stats,
                            }))
                            stage_stopped_stable = True
                            break
                pb_guard_interval = (
                    max(1, int(args.monitor_every)) if args.adaptive_stop
                    else max(1, int(args.guard_every))
                )
                if stage == "pb" and ((it + 1) % pb_guard_interval == 0 or it + 1 == count):
                    improved, _, selection_score = remember_best(stage, it + 1, args.pb_min_delta)
                    if improved:
                        no_improve = 0
                    else:
                        no_improve += 1
                    values = sess.run(parameter_tensors)
                    stable, stability_stats = update_stability(
                        stage, it + 1, selection_score, values
                    )
                    if args.adaptive_stop:
                        print(json.dumps({
                            "adaptive_monitor": True, "stage": stage,
                            "iteration": it + 1, "selection_score": selection_score,
                            **stability_stats,
                        }))
                    if args.adaptive_stop and stable:
                        print(json.dumps({
                            "stable_stop": True, "stage": stage, "iteration": it + 1,
                            "selection_basis": "calibration_plus_scaled_physics_only",
                            **stability_stats,
                        }))
                        stage_stopped_stable = True
                        break
                    if (not args.adaptive_stop and args.pb_patience > 0
                            and no_improve >= args.pb_patience):
                        print(json.dumps({"early_stop": True, "stage": stage, "iteration": it + 1,
                                          "selection_basis": "calibration_plus_scaled_physics_only",
                                          "best_stage": best["stage"], "best_iteration": best["iteration"],
                                          "best_score": best["score"]}))
                        break
            if args.adaptive_stop and stage in warm_best:
                restore_warm_best(
                    stage,
                    "stable_stop" if stage_stopped_stable
                    else "numerical_abort" if stage_numerical_abort
                    else "safety_cap",
                )
            if (args.adaptive_stop and count > 0 and not stage_stopped_stable
                    and not stage_numerical_abort):
                print(json.dumps({
                    "safety_cap_reached": True, "stage": stage, "iteration": count,
                    "message": "stage reached its safety cap before satisfying stability criteria",
                }))
            stage_stop_status[stage] = {
                "iteration": int(last_iteration),
                "reason": (
                    "stable" if stage_stopped_stable
                    else "numerical_abort" if stage_numerical_abort
                    else "safety_cap" if args.adaptive_stop and count > 0
                    else "zero_iterations" if count == 0
                    else "completed_or_legacy_patience"
                ),
                "numerical_recoveries": int(numerical_recovery_count),
            }
            pd.DataFrame(history).to_csv(outdir / "training_history.csv", index=False)
            if stage == "pb":
                restore_best()
            if saver is not None:
                saver.save(sess, str(outdir / f"checkpoint_{stage}.ckpt"), write_meta_graph=(stage == "electric"))

        if best["values"] is None:
            remember_best("restored", 0)
        restore_best()
        final_values = sess.run(parameter_tensors)
        final_feed = sample_feed(cases, np.random.default_rng(args.seed + 1), n_res, args.n_face, args.n_ic)
        final_joint_loss, final_joint_objective, final_physics_objective = [float(v) for v in sess.run(
            [joint_loss, joint_objective, joint_physics_scaled], final_feed
        )]
        final_loss_breakdown = {}
        for case in cases:
            names = tuple(case.diagnostics)
            values = sess.run([case.diagnostics[name] for name in names], final_feed)
            final_loss_breakdown[case.name] = {name: float(value) for name, value in zip(names, values)}
        if saver is not None:
            saver.save(sess, str(outdir / "model.ckpt"))
        metrics = {}
        likelihood_inputs = {
            "calibration": {"ph_frames": [], "ph_predictions": [], "pb_frames": [], "pb_predictions": []},
            "validation": {"ph_frames": [], "ph_predictions": [], "pb_frames": [], "pb_predictions": []},
        }
        for case in cases:
            ph_cal_pred, pb_cal_pred, ph_val_pred, pb_val_pred = sess.run(
                [case.ph_cal_full_pred, case.pb_cal_full_pred,
                 case.ph_val_full_pred, case.pb_val_full_pred]
            )
            for df, pred, name, obs_col, sigma_col in (
                (case.ph_cal_full_df, ph_cal_pred, "pH_calibration", "pH_obs", "pH_measurement_sd"),
                (case.pb_cal_full_df, pb_cal_pred, "TotalPb_calibration", "TotalPb_obs_mol_m3_bulk", "TotalPb_measurement_sd"),
                (case.ph_val_full_df, ph_val_pred, "pH_validation", "pH_obs", "pH_measurement_sd"),
                (case.pb_val_full_df, pb_val_pred, "TotalPb_validation", "TotalPb_obs_mol_m3_bulk", "TotalPb_measurement_sd"),
            ):
                result = df.copy()
                result["prediction"] = np.asarray(pred).reshape(-1)
                if not args.summary_only:
                    result.to_csv(outdir / f"{case.name}_{name}.csv", index=False)
                error = result["prediction"] - result[obs_col]
                metrics[f"{case.name}_{name}_RMSE"] = float(np.sqrt(np.mean(error ** 2)))
                metrics[f"{case.name}_{name}_standardized_RMSE"] = float(
                    np.sqrt(np.mean((error / result[sigma_col]) ** 2))
                )
                split = "calibration" if name.endswith("calibration") else "validation"
                channel = "ph" if name.startswith("pH_") else "pb"
                likelihood_inputs[split][f"{channel}_frames"].append(df)
                likelihood_inputs[split][f"{channel}_predictions"].append(
                    np.asarray(pred).reshape(-1)
                )

            profile = sess.run(case.profile_tensors)
            profile_df = pd.DataFrame({
                "time_day": 5.0,
                "distance_cm": case.profile_x_cm,
                "pH": profile[0].reshape(-1),
                "TotalPb_mol_m3_bulk": profile[1].reshape(-1),
                "Pb_aqueous_mol_m3_water": profile[2].reshape(-1),
                "Pb_aqueous_mol_m3_bulk": profile[3].reshape(-1),
                "Pb_adsorbed_mol_m3_bulk": profile[4].reshape(-1),
                "Pb_precipitated_mol_m3_bulk": profile[5].reshape(-1),
                "mass_gap_mol_m3_bulk": profile[6].reshape(-1),
            })
            if not args.summary_only:
                profile_df.to_csv(outdir / f"{case.name}_Pb_species_day5.csv", index=False)
            metrics[f"{case.name}_adsorbed_Pb_day5_min"] = float(profile_df["Pb_adsorbed_mol_m3_bulk"].min())
            metrics[f"{case.name}_mass_gap_day5_abs_max"] = float(profile_df["mass_gap_mol_m3_bulk"].abs().max())

        observation_likelihood = {}
        for split, values in likelihood_inputs.items():
            observation_likelihood[split] = gaussian_observation_likelihood(
                pd.concat(values["ph_frames"], ignore_index=True),
                np.concatenate(values["ph_predictions"]),
                pd.concat(values["pb_frames"], ignore_index=True),
                np.concatenate(values["pb_predictions"]),
            )

        if args.profile_rh_values:
            if len(rh_vars) != 1:
                raise ValueError("--profile-rh-values requires trainable RH")
            original_raw_rh = float(sess.run(rh_vars[0]))
            for requested_rh in (float(value) for value in args.profile_rh_values.split(",")):
                if not 1.0 < requested_rh < 70.0:
                    raise ValueError("RH profile values must satisfy 1 < RH < 70")
                q = (requested_rh - 1.0) / 69.0
                sess.run(rh_vars[0].assign(math.log(q / (1.0 - q))))
                profile_metrics, profile_score = calibration_metrics_and_score()
                rh_objective_slice.append({
                    "RH": requested_rh,
                    "selection_score": float(profile_score),
                    "selection_data_score": float(profile_metrics["selection_data_score"]),
                    "selection_physics_objective": float(profile_metrics["selection_physics_objective"]),
                })
            sess.run(rh_vars[0].assign(original_raw_rh))

    pd.DataFrame(history).to_csv(outdir / "training_history.csv", index=False)
    if rh_objective_slice:
        pd.DataFrame(rh_objective_slice).to_csv(outdir / "rh_objective_slice.csv", index=False)
    summary = {
        "inversion_version": "three_parameter_inversion",
        "run_role": args.run_role,
        "run_label": args.run_label,
        "fixed_parameters": {
            "Keos": args.fixed_keos,
            "RH": args.fixed_rh,
            "pH50": args.fixed_ph50,
        },
        "parameters": {"Keos": float(final_values[0]), "RH": float(final_values[1]), "pH50": float(final_values[2])},
        "calibration_points_cm": sorted(pd.read_csv(args.data_dir / "pH_calibration.csv")["distance_cm"].unique().tolist()),
        "validation_points_cm": sorted(pd.read_csv(args.data_dir / "pH_validation.csv")["distance_cm"].unique().tolist()),
        "metrics": metrics,
        "observation_likelihood": observation_likelihood,
        "metrics_evaluation_data_stride": 1,
        "final_joint_loss": final_joint_loss,
        "final_joint_objective": final_joint_objective,
        "final_physics_objective": final_physics_objective,
        "joint_loss_mode": args.joint_loss_mode,
        "balance_scales": balance_scales,
        "rh_objective_slice": rh_objective_slice,
        "final_loss_breakdown": final_loss_breakdown,
        "best_guard": {
            "selection_basis": (
                "full_calibration_gaussian_likelihood_plus_fixed_collocation_scaled_physics"
                if args.selection_data_mode == "gaussian-likelihood" else
                "full_calibration_standardized_RMSE_plus_fixed_collocation_scaled_physics"
            ),
            "stage": best["stage"],
            "iteration": best["iteration"],
            "score": best["score"],
            "parameters": best["parameters"],
            "metrics": best["metrics"],
        },
        "stage_parameter_gradients": stage_parameter_gradients,
        "stage_stop_status": stage_stop_status,
        "smoke": bool(args.smoke),
        "output": str(outdir),
    }
    (outdir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    try:
        save_parameter_trajectory(history, outdir, final_values)
    except Exception as exc:
        print(json.dumps({"parameter_trajectory_warning": str(exc)}))
    print("\n" + json.dumps(summary, indent=2))


## Training configuration

The block-coordinate curriculum alternates state-network and physical-parameter updates to reduce compensation.


In [ ]:
RUN_ARGUMENTS = [
    "--data-dir", str(DATA_DIR),
    "--output-root", str(OUTPUT_ROOT),
    "--seed", "20260729",
    "--init-Keos", "0.5", "--init-RH", "35", "--init-pH50", "7",
    "--start-stage", "electric",
    "--width", "20", "--depth", "5", "--data-stride", "1",
    "--n-res", "3000", "--n-face", "500", "--n-ic", "500",
    "--mass-nx", "21", "--mass-nt", "21",
    "--electric-iters", "5000", "--water-iters", "6000",
    "--acid-iters", "12000", "--pb-iters", "6000", "--joint-iters", "2000",
    "--adaptive-stop", "--monitor-every", "100", "--stability-window", "5",
    "--stability-score-rtol", "0.01", "--stability-parameter-rtol", "0.002",
    "--min-electric-iters", "500", "--min-water-iters", "800",
    "--min-acid-iters", "2500", "--min-pb-iters", "1000", "--min-joint-iters", "300",
    "--block-coordinate-curriculum", "--alternating-block-size", "100",
    "--water-state-pretrain-iters", "500", "--acid-state-pretrain-iters", "1000",
    "--pb-state-pretrain-iters", "500", "--joint-state-pretrain-iters", "200",
    "--joint-state-steps", "1", "--joint-parameter-steps", "1",
    "--warm-parameter-learning-rate", "0.0001",
    "--keos-parameter-learning-rate", "0.0003",
    "--rh-parameter-learning-rate", "0.0003",
    "--ph50-parameter-learning-rate", "0.0015",
    "--joint-learning-rate", "0.00002", "--joint-parameter-learning-rate", "0.0001",
    "--warm-lr-decay-steps", "2000", "--coupled-lr-decay-steps", "2000",
    "--lr-decay-rate", "0.5",
    "--acid-relative-weight", "2.0", "--acid-relative-floor", "0.05",
    "--pH-weight", "10.0", "--Pb-weight", "5.0",
    "--faraday-weight", "100.0", "--acid-bc-weight", "30.0", "--mass-weight", "50.0",
    "--adsorption-slope-n", "0.46903804045551195",
    "--selection-physics-weight", "0.5", "--selection-burn-in-iters", "500",
    "--selection-data-mode", "gaussian-likelihood",
    "--pb-patience", "0", "--joint-patience", "0",
    "--run-role", "inversion", "--run-label", "baseline_three_parameter",
]

print("Starting point: Keos=0.5, RH=35, pH50=7")
print("Resume checkpoint: none")
print("Stages: electric -> water -> acid -> Pb -> joint")


## Run

The following cell runs electric, water, acid, Pb, and joint stages in the current notebook kernel. Progress and adaptive-stop decisions appear directly below the cell.


In [ ]:
main(RUN_ARGUMENTS)
